# Exploração dos dados das cheias do Guaíba — maio de 2024

## Objetivo e escopo

Este notebook registra a exploração inicial das **43 camadas** do GeoPackage `cheias_rhguaiba_2024_db_v5.gpkg`, disponibilizado no [Zenodo](https://zenodo.org/records/11185049), para avaliar possíveis contribuições às análises do INCT SIMAI.

O conteúdo reúne verificação do arquivo, inventário, diagnóstico por camada, investigação espacial das escolas, agrupamento temático e revisão do glossário. A preparação está registrada em notebook próprio.

Os códigos e resultados registram as verificações realizadas; usos futuros são possibilidades de análise. Nenhuma limpeza ou alteração das fontes originais foi realizada neste notebook.


## 1. Preparação do ambiente

As bibliotecas abaixo permitem manipular caminhos (`pathlib`), consultar o banco SQLite (`sqlite3`) e trabalhar com dados geoespaciais (`geopandas` e `pyogrio`). O uso de `closing` garante o fechamento da conexão após a verificação de integridade. O ambiente precisa conter essas dependências para executar as células seguintes.


In [1]:
from pathlib import Path
import sqlite3
from contextlib import closing
import geopandas as gpd
import pyogrio
import pandas as pd


### 1.1 Localização do GeoPackage

O caminho do banco é construído a partir do diretório de execução. **O código pressupõe execução a partir de `notebooks/cheias_guaiba/`**. `Path.cwd().resolve().parents[1]` sobe dois níveis e define a raiz do projeto. Se o notebook for executado a partir de outro local, será necessário conferir o caminho antes de prosseguir.

O GeoPackage está em `dados/brutos/`, destinado aos arquivos originais coletados.


In [2]:
RAIZ = Path.cwd().resolve().parents[1]

ARQUIVO_BANCO = RAIZ / "dados" / "brutos" / "cheias_rhguaiba_2024_db_v5.gpkg"


## 2. Verificações iniciais do arquivo

### 2.1 Tamanho em disco

A consulta ao tamanho confirma a presença do arquivo e registra uma característica útil para acompanhar a coleta. A saída salva informa **471.748.608 bytes (449,89 MiB)**. O tamanho, isoladamente, não garante que o download esteja completo ou corresponda à versão oficial.


In [3]:
tamanho_bytes = ARQUIVO_BANCO.stat().st_size
tamanho_mib = tamanho_bytes / (1024 ** 2)

print(f"Tamanho do arquivo: {tamanho_bytes} bytes ({tamanho_mib:.2f} MiB)")

Tamanho do arquivo: 471748608 bytes (449.89 MiB)


### 2.2 Integridade estrutural do SQLite

Como o GeoPackage utiliza SQLite, a instrução `PRAGMA integrity_check` verifica a consistência estrutural do banco. A conexão é aberta em modo somente leitura (`mode=ro`) e fechada ao término da consulta.

O resultado salvo foi **`ok`**. Essa verificação não substitui a comparação com um checksum oficial nem valida a qualidade dos atributos, das geometrias ou o significado dos dados.


In [4]:
with closing(
    sqlite3.connect(
        ARQUIVO_BANCO.as_uri() + "?mode=ro", uri=True
    )
) as conexao:
    resultado = conexao.execute("PRAGMA integrity_check").fetchone()[0]

print(f"Resultado da verificação de integridade: {resultado}")

Resultado da verificação de integridade: ok


## 3. Inventário das camadas

### 3.1 Nomes e tipos de geometria

A listagem identifica **43 camadas** e seus tipos de geometria. Os nomes sugerem conteúdos de divisões territoriais, educação, saúde, abastecimento, drenagem, abrigos e inundação, com diferentes recortes espaciais e temporais.

Algumas geometrias incluem a dimensão Z. Sua presença não significa, por si só, que os valores representem altitude medida ou profundidade de inundação. A interpretação depende da documentação da fonte.


In [5]:
camadas = gpd.list_layers(ARQUIVO_BANCO) # Listar as camadas do banco de dados

display(camadas)

,name,geometry_type
0,poa_admin_bairros,MultiPolygon Z
1,poa_edu_escolas,Point Z
2,poa_inundacao_1941_475cm,MultiPolygon
3,poa_inundacao_1967_313cm,MultiPolygon
4,poa_inundacao_2015_294cm,MultiPolygon
5,poa_inundacao_2023_nov_346cm,Polygon
6,poa_inundacao_2023_set_318cm,MultiPolygon
7,poa_inundacao_2024_530cm,MultiPolygon
8,poa_inundacao_simulada_500cm,MultiPolygon
9,poa_inundacao_simulada_550cm,MultiPolygon


### 3.2 Estrutura e sistema de referência de cada camada

A consulta com `pyogrio.read_info()` reúne o nome, o tipo de geometria, o número de registros, os atributos e o sistema de referência de coordenadas (CRS), sem carregar todas as camadas completas em memória.

Esse inventário orienta a exploração individual. Há camadas em **EPSG:4326** e **EPSG:31982**; cruzamentos espaciais exigirão compatibilizar os sistemas de referência. Esta tabela descreve a estrutura do banco, mas ainda não fornece uma avaliação de qualidade ou relevância de todas as camadas.


In [6]:
registros = []

for nome_camada in camadas['name']:
    info = pyogrio.read_info(ARQUIVO_BANCO, layer=nome_camada)

    registros.append({
        "camada": nome_camada,
        "tipo_geometria": info['geometry_type'],
        "registros": info['features'],
        "colunas": info['fields'],
        "crs": info['crs']
    })

df_registros = gpd.GeoDataFrame(registros)
display(df_registros)

,camada,tipo_geometria,registros,colunas,crs
0,poa_admin_bairros,MultiPolygon Z,83,"[nmbop, cdbop, nmrop, cdrop, nmrp, cdrp, cota_...",EPSG:31982
1,poa_edu_escolas,Point Z,1062,"[Name, descriptio, descri____, MAPA, MEC_INEP_...",EPSG:4326
2,poa_inundacao_1941_475cm,MultiPolygon,248,"[dn, area_ha]",EPSG:31982
3,poa_inundacao_1967_313cm,MultiPolygon,507,[dn],EPSG:31982
4,poa_inundacao_2015_294cm,MultiPolygon,450,[dn],EPSG:31982
5,poa_inundacao_2023_nov_346cm,Polygon,424,[dn],EPSG:31982
6,poa_inundacao_2023_set_318cm,MultiPolygon,526,"[dn, area_ha]",EPSG:31982
7,poa_inundacao_2024_530cm,MultiPolygon,478,[DN],EPSG:31982
8,poa_inundacao_simulada_500cm,MultiPolygon,254,[DN],EPSG:31982
9,poa_inundacao_simulada_550cm,MultiPolygon,227,[DN],EPSG:31982


## 4. Procedimento e cuidados gerais

`exploracao_inicial()` carrega uma camada completa, exibe **até 100 linhas** e apresenta tipos, ausências, linhas inteiramente duplicadas, CRS e estatísticas descritivas. Todos os cálculos utilizam a camada completa.

Cuidados comuns à interpretação:

- `isna()` não detecta textos vazios, “s/ infos” ou zeros usados como marcadores. Geometrias vazias também precisam de verificação própria.
- Ausência de linhas duplicadas não garante identificadores únicos; coordenadas compartilhadas podem corresponder a serviços no mesmo endereço.
- A função não valida geometrias, sobreposições, cobertura ou precisão espacial. A investigação adicional das escolas aparece na seção 7.
- Eventuais cruzamentos exigem CRS compatíveis. Medidas planas de área e comprimento exigem projeção adequada em metros: **1 ha = 10.000 m²** e **1 km² = 1.000.000 m²**. A unidade dos atributos existentes depende da documentação.
- A dimensão Z não comprova altitude real ou profundidade de inundação.
- Relação espacial com uma mancha não demonstra danos, interrupção de serviços ou causalidade. Simulações, estimativas históricas e produtos observacionais têm metodologias distintas.

Os diagnósticos seguintes destacam resultados e limitações específicos. A seleção de usos e eventuais tratamentos depende das perguntas de análise e dos esclarecimentos documentais.


In [7]:
def exploracao_inicial(nome_camada):
    gdf = gpd.read_file(ARQUIVO_BANCO, layer=nome_camada)

    print(f"\nExploração inicial da camada: {nome_camada}\n")

    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        if len(gdf) > 100:
            print(f"Exibindo as primeiras 100 de {len(gdf)} linhas. As verificações usam a camada completa.")
            display(gdf.head(100))
        else:
            display(gdf)

    print(f"\nTipos de dados:")
    display(gdf.dtypes)

    print(f'\nValores ausentes:')
    display(gdf.isna().sum())

    print("\nValores duplicados:")
    display(gdf.duplicated().sum())

    print("\nSistema de cordenadas (CRS):")
    display(gdf.crs)

    print("\nEstatísticas descritivas:")
    display(gdf.describe(include='all'))

## 5. Exploração das divisões territoriais

### Camada `poa_admin_bairros`

A primeira camada da listagem possui **83 registros**, com geometrias `MultiPolygon Z`, em **EPSG:31982**. Os atributos incluem nomes e códigos territoriais, além de `cota_mean`. Os significados oficiais das abreviações e a metodologia de cálculo da cota precisam ser confirmados.

Esta camada é candidata a apoiar mapas, agrupamentos territoriais e cruzamentos com áreas de inundação. A correspondência com os 94 nomes de bairros do JSON é aprofundada no [notebook de preparação](preparacao_cheias_guaiba_2024.ipynb); não se deve presumir compatibilidade direta entre as divisões territoriais.


In [8]:
exploracao_inicial(camadas['name'][0])  # Explorando a primeira camada como exemplo


Exploração inicial da camada: poa_admin_bairros



,nmbop,cdbop,nmrop,cdrop,nmrp,cdrp,cota_mean,geometry
0,AGRONOMIA,BOP71,LOMBA DO PINHEIRO,OP04,LOMBA DO PINHEIRO - PARTENON,RP07,125.157306,"MULTIPOLYGON Z (((489305.255 6676873.088 0, 48..."
1,ANCHIETA,BOP19,HUMAITA-NAVEGANTES,OP01,HUMAITA - NAVEGANTES - ILHAS E NOROESTE,RP02,1.954820,"MULTIPOLYGON Z (((484961.018 6685092.233 0, 48..."
2,ARQUIPELAGO,BOP36,ILHAS,OP17,HUMAITA - NAVEGANTES - ILHAS E NOROESTE,RP02,1.162602,"MULTIPOLYGON Z (((471983.597 6674237.441 0, 47..."
3,AUXILIADORA,BOP01,CENTRO,OP16,CENTRO,RP01,28.703022,"MULTIPOLYGON Z (((482207 6678761.422 0, 482243..."
4,AZENHA,BOP02,CENTRO,OP16,CENTRO,RP01,12.897710,"MULTIPOLYGON Z (((479842.718 6675588.541 0, 47..."
5,BELA VISTA,BOP03,CENTRO,OP16,CENTRO,RP01,57.941924,"MULTIPOLYGON Z (((482557.279 6677932.464 0, 48..."
6,BELEM NOVO,BOP79,EXTREMO SUL,OP13,RESTINGA - EXTREMO SUL,RP08,7.026549,"MULTIPOLYGON Z (((484231.641 6652138.177 0, 48..."
7,BELEM VELHO,BOP49,GLORIA,OP09,GLORIA - CRUZEIRO E CRISTAL,RP05,92.959028,"MULTIPOLYGON Z (((482960.672 6670341.853 0, 48..."
8,BOA VISTA,BOP24,NOROESTE,OP02,HUMAITA - NAVEGANTES - ILHAS E NOROESTE,RP02,42.304067,"MULTIPOLYGON Z (((483409.753 6678170.355 0, 48..."
9,BOM FIM,BOP04,CENTRO,OP16,CENTRO,RP01,12.993476,"MULTIPOLYGON Z (((479499.494 6677686.641 0, 47..."



Tipos de dados:


nmbop             str
cdbop             str
nmrop             str
cdrop             str
nmrp              str
cdrp              str
cota_mean     float64
geometry     geometry
dtype: object


Valores ausentes:


nmbop        0
cdbop        0
nmrop        0
cdrop        0
nmrp         0
cdrp         0
cota_mean    0
geometry     0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,nmbop,cdbop,nmrop,cdrop,nmrp,cdrp,cota_mean,geometry
count,83,83,83,83,83,83,83.000000,83
unique,83,83,17,17,8,8,NaN,83
top,AGRONOMIA,BOP71,CENTRO,OP16,HUMAITA - NAVEGANTES - ILHAS E NOROESTE,RP02,NaN,MULTIPOLYGON Z (((489305.25516188843 6676873.0...
freq,1,1,18,18,18,18,NaN,1
mean,NaN,NaN,NaN,NaN,NaN,NaN,37.706239,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,34.848165,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,1.162602,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,9.979408,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,30.327546,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,52.861722,NaN


### Resultados iniciais e limitações — divisões territoriais

Nas saídas salvas, não foram identificados valores ausentes nem linhas inteiramente duplicadas. Os campos `nmbop` e `cdbop` apresentam 83 valores distintos cada. Esses resultados favorecem sua investigação como identificadores territoriais, mas não comprovam a validade das geometrias ou a cobertura completa do município.

O atributo `cota_mean` varia aproximadamente de **1,16 a 176,06**. Sua unidade, referência vertical e forma de cálculo ainda precisam ser documentadas. Portanto, esses valores não devem ser interpretados como profundidade de inundação.

Permanecem pendentes a validação geométrica, a conferência das divisões territoriais e a comparação com os bairros utilizados no JSON do INCT.


## 6. Exploração dos estabelecimentos escolares

### Camada `poa_edu_escolas`

A segunda camada da listagem contém **1.062 registros**, com geometrias `Point Z`, em **EPSG:4326**. Inclui identificação, rede de ensino, endereço, bairro e campos relacionados a matrículas e indicadores educacionais.

O interesse inicial é avaliar a distribuição territorial dos estabelecimentos e a possibilidade de cruzá-los com manchas de inundação. Antes de realizar esses cruzamentos ou utilizar os indicadores, precisamos examinar os identificadores, a localização e o período de referência dos atributos.


In [9]:
exploracao_inicial(camadas['name'][1])


Exploração inicial da camada: poa_edu_escolas

Exibindo as primeiras 100 de 1062 linhas. As verificações usam a camada completa.


,Name,descriptio,descri____,MAPA,MEC_INEP_C,C__digo,CRE,Redes_de_E,Munic__pio,INEP,Escolas_20,Local,Endere__o,N__mero_In,CEP,DDD,Telefone,E_mail,Matr__cula,Matr__cu_1,Matr__cu_2,Matr__cu_3,Matr__cu_4,Matr__cu_5,Matr__cu_6,Total_de_M,Creche,Pr___Escol,Ensino_Fun,Ensino_M__,Educa____o,Educa____1,Educa____2,IDEB_2013,IDEB_2013_,IDEB__2014,IDEB__2013,BAIRRO_GEO,ROP,Regi__o_do,CDL_GEO,NUM_GEO,geometry
0,5066 EEI MEU NENE (C PE ),descrição: MAPA: <br>MEC/INEP/CENSO ESCOLAR: <...,MAPA: \nMEC/INEP/CENSO ESCOLAR: \nCódigo: 258P...,None,NaN,258P,1,PARTICULAR,PORTO ALEGRE,43187137,5066 EEI MEU NENE,Urbana,R ERNESTO LISCANO,195,91540310,51,33193144,cristiane-fogassafonseca@hotmail.com,30,17,0,0,0,0,0,47,C,PE,NaN,NaN,NaN,NaN,NaN,0,0,0,0,AGRONOMIA,4,LOMBA DO PINHEIRO,8771073,195,POINT Z (-51.12672 -30.08765 0)
1,EEEM AGRONOMO PEDRO PEREIRA ( EF EM EJA),descrição: MAPA: <br>MEC/INEP/CENSO ESCOLAR: <...,MAPA: \nMEC/INEP/CENSO ESCOLAR: \nCódigo: 21E\...,None,NaN,21E,1,ESTADUAL,PORTO ALEGRE,43105769,EEEM AGRONOMO PEDRO PEREIRA,Urbana,BENTO GONCALVES,8426,91540000,51,33195606,empedropereira-01cre@seduc.rs.gov.br,0,0,355,330,0,0,435,1120,NaN,NaN,EF,EM,NaN,NaN,EJA,"4,9","4,9","3,5","3,5",AGRONOMIA,4,LOMBA DO PINHEIRO,7975113,8426,POINT Z (-51.13402 -30.07458 0)
2,EEEF IRMAO MIGUEL DARIO ( EJA),descrição: MAPA: <br>MEC/INEP/CENSO ESCOLAR: <...,MAPA: \nMEC/INEP/CENSO ESCOLAR: \nCódigo: 220E...,None,NaN,220E,1,ESTADUAL,PORTO ALEGRE,43108776,EEEF IRMAO MIGUEL DARIO,Urbana,AV BENTO GONCALVES,7500,90540000,51,33395090,migueldario01cre@educacao.rs.gov.br,0,0,0,0,0,0,107,107,NaN,NaN,NaN,NaN,NaN,NaN,EJA,0,0,0,0,AGRONOMIA,4,LOMBA DO PINHEIRO,7975113,7500,POINT Z (-51.14153 -30.06961 0)
3,COLEGIO DE APLICACAO UFRGS ( EF EM EJA),descrição: MAPA: <br>MEC/INEP/CENSO ESCOLAR: <...,MAPA: \nMEC/INEP/CENSO ESCOLAR: \nCódigo: 2F\n...,None,NaN,2F,1,FEDERAL,PORTO ALEGRE,43104932,COLEGIO DE APLICACAO UFRGS,Urbana,AV. BENTO GONCALVES,9500,91501970,51,33086977,gerencia_aplicacao@ufrgs.br,0,0,318,221,0,0,108,647,NaN,NaN,EF,EM,NaN,NaN,EJA,6,6,"4,9","4,9",AGRONOMIA,4,LOMBA DO PINHEIRO,7975113,9500,POINT Z (-51.1257 -30.07602 0)
4,EEEF DESIDERIO TORQUATO FINAMOR ( EF ),descrição: MAPA: <br>MEC/INEP/CENSO ESCOLAR: <...,MAPA: \nMEC/INEP/CENSO ESCOLAR: \nCódigo: 57E\...,None,NaN,57E,1,ESTADUAL,PORTO ALEGRE,43106153,EEEF DESIDERIO TORQUATO FINAMOR,Urbana,AV BENTO GONCALVES,7500,91540000,51,33851023,desideriotfinamor01cre@educacao.rs.gov.br,0,0,529,0,0,0,0,529,NaN,NaN,EF,NaN,NaN,NaN,NaN,"4,2","4,2","2,5","2,5",AGRONOMIA,4,LOMBA DO PINHEIRO,7975113,7500,POINT Z (-51.14153 -30.06961 0)
5,5198 EEI MEUS PRIMEIROS PASSOS (C PE ),descrição: MAPA: <br>MEC/INEP/CENSO ESCOLAR: <...,MAPA: \nMEC/INEP/CENSO ESCOLAR: \nCódigo: 676P...,None,NaN,676P,1,PARTICULAR,PORTO ALEGRE,43367380,5198 EEI MEUS PRIMEIROS PASSOS,Urbana,JOAO DE OLIVEIRA REMIAO,176,91550000,51,33150064,jessica.p-r@hotmail.com,61,44,0,0,0,0,0,105,C,PE,NaN,NaN,NaN,NaN,NaN,0,0,0,0,AGRONOMIA,4,LOMBA DO PINHEIRO,8772089,176,POINT Z (-51.13132 -30.07624 0)
6,EEEF ONOFRE PIRES ( EF ),descrição: MAPA: <br>MEC/INEP/CENSO ESCOLAR: h...,MAPA: \nMEC/INEP/CENSO ESCOLAR: http://idebesc...,None,http://idebescola.inep.gov.br/ideb/escola/dado...,215E,1,ESTADUAL,PORTO ALEGRE,43108725,EEEF ONOFRE PIRES,Urbana,RUA BECO DO DAVID,269,91550160,51,33195333,onofrepires01cre@educacao.rs.gov.br,0,0,425,0,0,0,0,425,NaN,NaN,EF,NaN,NaN,NaN,NaN,5,5,4,4,AGRONOMIA,4,LOMBA DO PINHEIRO,8771016,269,POINT Z (-51.13929 -30.08423 0)
7,ESCOLA FUNDAMENTAL LA SALLE ESMERALDA ( EF ),descrição: MAPA: <br>MEC/INEP/CENSO ESCOLAR: <...,MAPA: \nMEC/INEP/CENSO ESCOLAR: \nCódigo: 164P...,None,NaN,164P,1,PARTICULAR,PORTO ALEGRE,43108849,ESCOLA FUNDAMENTAL LA SALLE ESMERALDA,Urbana,RUA MARIO TAVARES DE SOUZA,400,91540150,51,33196344,monica.matos@lasalle.org.br,0,0,565,0,0,0,0,565,NaN,NaN,EF,NaN,NaN,NaN,NaN,0,0,0,0,AGRONOMIA,4,LOMBA DO PINHEIRO,8871030,400,POINT Z (-51.12532 -30.08874 0)
8,EEEF PROFESSOR SYLVIO TORRES ( EF EJA),descrição: MAPA: <br>MEC/INEP/CENSO ESCOLAR: <...,M


Tipos de dados:


Name               str
descriptio         str
descri____         str
MAPA            object
MEC_INEP_C         str
C__digo            str
CRE                str
Redes_de_E         str
Munic__pio         str
INEP               str
Escolas_20         str
Local              str
Endere__o          str
N__mero_In         str
CEP                str
DDD                str
Telefone           str
E_mail             str
Matr__cula         str
Matr__cu_1         str
Matr__cu_2         str
Matr__cu_3         str
Matr__cu_4         str
Matr__cu_5         str
Matr__cu_6         str
Total_de_M         str
Creche             str
Pr___Escol         str
Ensino_Fun         str
Ensino_M__         str
Educa____o         str
Educa____1         str
Educa____2         str
IDEB_2013          str
IDEB_2013_         str
IDEB__2014         str
IDEB__2013         str
BAIRRO_GEO         str
ROP                str
Regi__o_do         str
CDL_GEO            str
NUM_GEO            str
geometry      geometry
dtype: obje


Valores ausentes:


Name             0
descriptio       0
descri____       0
MAPA          1062
MEC_INEP_C     925
C__digo          0
CRE              0
Redes_de_E       0
Munic__pio       0
INEP             0
Escolas_20       0
Local            0
Endere__o        0
N__mero_In       0
CEP              0
DDD              0
Telefone         0
E_mail          46
Matr__cula       0
Matr__cu_1       0
Matr__cu_2       0
Matr__cu_3       0
Matr__cu_4       0
Matr__cu_5       0
Matr__cu_6       0
Total_de_M       0
Creche         432
Pr___Escol     295
Ensino_Fun     696
Ensino_M__     917
Educa____o    1005
Educa____1    1041
Educa____2     966
IDEB_2013        0
IDEB_2013_       0
IDEB__2014       0
IDEB__2013       0
BAIRRO_GEO       0
ROP              0
Regi__o_do       0
CDL_GEO          0
NUM_GEO          0
geometry         0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Name,descriptio,descri____,MAPA,MEC_INEP_C,C__digo,CRE,Redes_de_E,Munic__pio,INEP,...,IDEB_2013,IDEB_2013_,IDEB__2014,IDEB__2013,BAIRRO_GEO,ROP,Regi__o_do,CDL_GEO,NUM_GEO,geometry
count,1062,1062,1062,0,137,1062,1062,1062,1062,1062,...,1062,1062,1062,1062,1062,1062,1062,1062,1062,1062
unique,1059,1062,1062,0,135,1062,1,5,1,1062,...,41,41,36,36,83,17,17,768,627,1048
top,EEI PRIMEIROS PASSOS (C PE ),descrição: MAPA: <br>MEC/INEP/CENSO ESCOLAR: <...,MAPA: \nMEC/INEP/CENSO ESCOLAR: \nCódigo: 258P...,NaN,http://idebescola.inep.gov.br/ideb/escola/dado...,258P,1,PARTICULAR,PORTO ALEGRE,43187137,...,0,0,0,0,SARANDI,16,CENTRO,7977218,10,POINT Z (-51.15317 -30.010438000000356 0)
freq,2,1,1,NaN,2,1,1062,697,1062,1,...,828,828,866,866,68,229,229,10,38,4


### Resultados iniciais e limitações — escolas

As saídas salvas mostram os seguintes pontos:

- Não há linhas inteiramente duplicadas; `INEP` e `C__digo` apresentam 1.062 valores distintos e nenhuma ausência reconhecida. O formato e a validade desses códigos ainda precisam ser verificados.
- `MAPA` está ausente em todos os registros. `MEC_INEP_C` possui 925 ausências, e os exemplos preenchidos contêm links.
- Há nomes de colunas truncados ou com caracteres substituídos. Campos aparentemente numéricos estão armazenados como texto, incluindo exemplos de IDEB com vírgula decimal.
- Os campos de IDEB contêm muitos zeros. É necessário esclarecer se representam valores reais, ausência de avaliação ou outra convenção antes de calcular estatísticas.
- Há 1.048 geometrias distintas para 1.062 registros. Coordenadas compartilhadas podem representar escolas no mesmo endereço e não justificam, isoladamente, remover registros.
- `BAIRRO_GEO` apresenta 83 valores distintos. A correspondência com os polígonos territoriais e com o JSON ainda não foi validada.

A ausência de geometria nula não confirma que os pontos estejam corretamente localizados. Também precisamos confirmar a data dos dados escolares: os nomes de alguns indicadores remetem a anos anteriores a 2024.

A seção 7 complementa este diagnóstico com verificações geométricas e territoriais. Formatos de códigos e exposição às manchas ainda não foram avaliados.


### Atributos candidatos — escolas

O foco considerado nesta etapa é **localização e identificação**: `geometry`, `INEP`, `Name`, `C__digo` e `BAIRRO_GEO`. Rede de ensino e endereço poderiam complementar agrupamentos ou conferências.

`MAPA` está totalmente ausente; descrições e links podem servir à interpretação. Matrículas e IDEB ficam fora do foco atual: possuem limitações de formato, referência temporal e significado dos zeros. Nenhuma coluna foi excluída.


## 7. Verificação espacial das escolas

Verificamos geometrias nulas, vazias e inválidas, coincidência de coordenadas e relação dos pontos com a união de `poa_admin_bairros`. `INEP` e `Name` identificam os resultados.

A comparação testa a **cobertura dos polígonos disponíveis**, sem confirmar a completude do limite municipal. `intersects` inclui pontos no interior e na borda.


In [10]:
escolas = gpd.read_file(ARQUIVO_BANCO, layer="poa_edu_escolas")
bairros = gpd.read_file(ARQUIVO_BANCO, layer="poa_admin_bairros")

if escolas.crs is None or bairros.crs is None:
    raise ValueError("As duas camadas precisam ter um CRS definido.")

geometria_nula = escolas.geometry.isna()
geometria_vazia = escolas.geometry.is_empty
geometria_invalida = ~geometria_nula & ~geometria_vazia & ~escolas.geometry.is_valid
geometria_utilizavel = ~geometria_nula & ~geometria_vazia & ~geometria_invalida

print(f"Total de registros: {len(escolas)}")
print(f"Geometrias nulas: {geometria_nula.sum()}")
print(f"Geometrias vazias: {geometria_vazia.sum()}")
print(f"Geometrias inválidas não vazias: {geometria_invalida.sum()}")
print(f"Geometrias utilizáveis: {geometria_utilizavel.sum()}")

# Não corrigir automaticamente polígonos que possam comprometer a referência.
if (bairros.geometry.isna().any() or bairros.geometry.is_empty.any()
        or not bairros.geometry.is_valid.all()):
    raise ValueError("A camada territorial contém geometrias problemáticas; investigar antes da união.")
if not escolas.loc[geometria_utilizavel].geom_type.eq("Point").all():
    raise ValueError("Há geometrias utilizáveis que não são pontos; investigar antes de continuar.")

Total de registros: 1062
Geometrias nulas: 0
Geometrias vazias: 0
Geometrias inválidas não vazias: 0
Geometrias utilizáveis: 1062


### 7.1 Coordenadas compartilhadas

A comparação abaixo usa as coordenadas horizontais X e Y na camada original, com igualdade exata, desconsiderando Z. Pequenas diferenças entre coordenadas não serão identificadas como coincidências. Estabelecimentos distintos podem ocupar o mesmo endereço; nesta etapa, nenhuma escola foi removida.


In [11]:
escolas_localizaveis = escolas.loc[geometria_utilizavel].copy()
escolas_localizaveis["x_original"] = escolas_localizaveis.geometry.x
escolas_localizaveis["y_original"] = escolas_localizaveis.geometry.y

coordenadas_repetidas = escolas_localizaveis.duplicated(
    subset=["x_original", "y_original"], keep=False
)
escolas_coincidentes = escolas_localizaveis.loc[coordenadas_repetidas].sort_values(
    ["x_original", "y_original"]
)
quantidade_locais = escolas_coincidentes[["x_original", "y_original"]].drop_duplicates().shape[0]

print(f"Registros com coordenadas compartilhadas: {len(escolas_coincidentes)}")
print(f"Locais com mais de um registro: {quantidade_locais}")
display(escolas_coincidentes[["INEP", "Name", "BAIRRO_GEO", "x_original", "y_original"]])

Registros com coordenadas compartilhadas: 26
Locais com mais de um registro: 12


,INEP,Name,BAIRRO_GEO,x_original,y_original
138,43108130,COLEGIO ACM CENTRO ( PE EF EM ),CENTRO HISTÓRICO,-51.229754,-30.036659
156,43201776,CENTRO TECNOLOGICO DA ACM ( EP ),CENTRO HISTÓRICO,-51.229754,-30.036659
182,43107443,ESC DE ENS FUND LA SALLE PAO DOS POBRES ( E...,CIDADE BAIXA,-51.226964,-30.040226
197,43009174,ESC EDUC PROF PAO DOS POBRES ( EP ),CIDADE BAIXA,-51.226964,-30.040226
141,43175511,COLEGIO UNIFICADO EDUCACAO DE JOVENS E ADULTOS...,CENTRO HISTÓRICO,-51.221921,-30.027338
150,43186700,COLEGIO UNIFICADO ( EM ),CENTRO HISTÓRICO,-51.221921,-30.027338
166,43250009,FACULDADE TECNOLOGIA TECBRASIL - UNIDADE PORTO...,CENTRO HISTÓRICO,-51.219885,-30.024834
170,43009735,TECBRASIL - ESC EDUC PROFISSIONAL ( EP ...,CENTRO HISTÓRICO,-51.219885,-30.024834
463,43105246,EMEM EMILIO MEYER ( EM EP ),MEDIANEIRA,-51.207972,-30.068406
465,43108903,ESC EST DE ENS MEDIO PRESIDENTE COSTA E SILVA ...,MEDIANEIRA,-51.207972,-30.068406


### 7.2 Relação com a cobertura territorial

Os pontos utilizáveis são reprojetados para o CRS dos bairros. A união dos polígonos mantém eventuais lacunas da fonte. O código calcula `dentro_limite` apenas para os pontos utilizáveis e deixa os demais com classificação ausente.

O código expressa a distância dos pontos externos até a cobertura em metros, usando a referência EPSG:31982. Distância pequena pode sugerir um caso de borda, mas não comprova erro ou acerto de localização.


In [12]:
if bairros.crs.to_epsg() != 31982:
    raise ValueError("Confirmar uma projeção em metros antes de calcular distâncias.")

limite_referencia = bairros.geometry.union_all()
escolas_verificadas = escolas.to_crs(bairros.crs).copy()
escolas_verificadas["dentro_limite"] = None
escolas_verificadas.loc[geometria_utilizavel, "dentro_limite"] = (
    escolas_verificadas.loc[geometria_utilizavel].geometry.intersects(limite_referencia)
)

fora_cobertura = geometria_utilizavel & escolas_verificadas["dentro_limite"].eq(False)
escolas_fora = escolas_verificadas.loc[fora_cobertura].copy()
escolas_fora["distancia_cobertura_m"] = escolas_fora.geometry.distance(limite_referencia)
escolas_fora = escolas_fora.sort_values("distancia_cobertura_m")

print(f"Dentro da cobertura ou sobre a borda: {escolas_verificadas['dentro_limite'].eq(True).sum()}")
print(f"Fora da cobertura: {len(escolas_fora)}")
print(f"Sem classificação espacial: {escolas_verificadas['dentro_limite'].isna().sum()}")
display(escolas_fora[["INEP", "Name", "BAIRRO_GEO", "distancia_cobertura_m", "geometry"]])

Dentro da cobertura ou sobre a borda: 1057
Fora da cobertura: 5
Sem classificação espacial: 0


,INEP,Name,BAIRRO_GEO,distancia_cobertura_m,geometry
404,43002005,EEI GOLFINHO SAPECA (C PE ),LOMBA DO PINHEIRO,2.781385,POINT Z (489360.538 6669544.31 0)
8,43107966,EEEF PROFESSOR SYLVIO TORRES ( EF EJA),AGRONOMIA,8.261509,POINT Z (488807.251 6672266.582 0)
992,43188311,5114 EEI CAPELA NAVEGANTES (C PE ),VILA ASSUNÇÃO,18.718006,POINT Z (474538.855 6670272.961 0)
415,43208428,EEIEF FAG NHIN ( PE EF EJA),LOMBA DO PINHEIRO,28.605080,POINT Z (490976.531 6666042.597 0)
424,43167535,EMEF SAO PEDRO ( PE EF EJA),LOMBA DO PINHEIRO,85.356541,POINT Z (489712.999 6669514.505 0)


### Resultados espaciais — escolas

Os **1.062 registros** não apresentam geometrias nulas, vazias ou inválidas. Há **26 registros em 12 locais com coordenadas horizontais compartilhadas**. A comparação territorial encontrou **1.057 pontos dentro da cobertura ou na borda e cinco fora**, sem pontos não classificados.


### Critério adotado — pertencimento municipal

Foi decidido considerar os **1.062 registros pertencentes a Porto Alegre** no recorte desta etapa, inclusive os cinco pontos externos, com base nos bairros informados:

| Escola | INEP | Bairro informado | Distância até a cobertura dos polígonos |
| --- | --- | --- | ---: |
| EEI Golfinho Sapeca | 43002005 | Lomba do Pinheiro | 2,78 m |
| EEEF Professor Sylvio Torres | 43107966 | Agronomia | 8,26 m |
| EEI Capela Navegantes | 43188311 | Vila Assunção | 18,72 m |
| EEIEF Fag Nhin | 43208428 | Lomba do Pinheiro | 28,61 m |
| EMEF São Pedro | 43167535 | Lomba do Pinheiro | 85,36 m |

Esse critério não corrige coordenadas nem altera `dentro_limite`, que continua representando a relação geométrica com os polígonos. Nenhum dos cinco registros foi excluído.


## 8. Exploração da camada de inundação de 1941

### Camada `poa_inundacao_1941_475cm`

O nome da camada faz referência ao evento de 1941 e a uma cota de 475 cm. A origem, o método de delimitação e a referência dessa cota ainda precisam ser confirmados na documentação; ela não deve ser interpretada automaticamente como profundidade de água em cada polígono.

O objetivo desta etapa é reconhecer a estrutura da camada e avaliar seu potencial para mapas e comparações históricas. Ainda não realizamos cruzamentos com as escolas nem alterações nos dados.


In [13]:
exploracao_inicial(camadas['name'][2])  # Camada de inundação de 1941


Exploração inicial da camada: poa_inundacao_1941_475cm

Exibindo as primeiras 100 de 248 linhas. As verificações usam a camada completa.


,dn,area_ha,geometry
0,1,8,"MULTIPOLYGON (((473660.024 6688037.737, 473696..."
1,1,0,"MULTIPOLYGON (((477739.91 6685465.636, 477743...."
2,1,537,"MULTIPOLYGON (((473988.015 6687869.731, 474000..."
3,1,62,"MULTIPOLYGON (((475875.962 6686145.663, 475895..."
4,1,0,"MULTIPOLYGON (((488627.606 6684837.611, 488631..."
5,1,0,"MULTIPOLYGON (((485127.704 6684749.608, 485179..."
6,1,0,"MULTIPOLYGON (((483575.747 6684601.602, 483587..."
7,1,0,"MULTIPOLYGON (((483603.746 6684605.602, 483643..."
8,1,0,"MULTIPOLYGON (((483483.75 6684585.601, 483511...."
9,1,0,"MULTIPOLYGON (((483475.75 6684581.601, 483483...."



Tipos de dados:


dn             int32
area_ha        int64
geometry    geometry
dtype: object


Valores ausentes:


dn          0
area_ha     0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,dn,area_ha,geometry
count,248.0,248.000000,248
unique,NaN,NaN,248
top,NaN,NaN,MULTIPOLYGON (((473660.02367629454 6688037.737...
freq,NaN,NaN,1
mean,1.0,53.774194,NaN
std,0.0,415.408415,NaN
min,1.0,0.000000,NaN
25%,1.0,0.000000,NaN
50%,1.0,0.000000,NaN
75%,1.0,0.000000,NaN


**Diagnóstico:** 248 registros, geometria `MultiPolygon`, EPSG:31982. Atributos: `dn` e `area_ha`; o código de classe vale 1 em todos os registros.

Sem ausências ou linhas inteiramente duplicadas. `area_ha` é inteiro e contém zeros; sua precisão e correspondência com a geometria precisam de conferência.


### Possíveis usos das camadas associadas a eventos

As camadas desta sequência poderiam apoiar mapas e comparações entre eventos e cruzamentos com bairros para avaliar áreas e proporções por unidade territorial. Esses usos dependem de esclarecer a metodologia, a referência das cotas e os códigos de classe.

O glossário caracteriza 1941, 1967 e setembro e novembro de 2023 como **manchas estimadas**; para 2015 e 2024 há divergências de descrição, registradas nas respectivas seções. Não se presume observação direta ou compatibilidade entre todos os produtos.


## 9. Exploração da camada de inundação de 1967

### Camada `poa_inundacao_1967_313cm`

O nome faz referência ao evento de 1967 e a uma cota de 313 cm. A metodologia de delimitação e a referência dessa cota ainda precisam ser confirmadas na documentação. A cota não deve ser interpretada automaticamente como profundidade de água em cada polígono.


In [14]:
exploracao_inicial(camadas['name'][3])  # Camada de inundação de 1967


Exploração inicial da camada: poa_inundacao_1967_313cm

Exibindo as primeiras 100 de 507 linhas. As verificações usam a camada completa.


,dn,geometry
0,1,"MULTIPOLYGON (((473660.024 6688037.737, 473696..."
1,1,"MULTIPOLYGON (((477739.91 6685465.636, 477743...."
2,1,"MULTIPOLYGON (((473988.015 6687869.731, 474000..."
3,1,"MULTIPOLYGON (((475875.962 6686145.663, 475895..."
4,1,"MULTIPOLYGON (((488843.6 6685053.62, 488855.6 ..."
5,1,"MULTIPOLYGON (((488619.606 6684817.611, 488627..."
6,1,"MULTIPOLYGON (((488775.602 6684945.616, 488803..."
7,1,"MULTIPOLYGON (((484147.731 6684709.606, 484155..."
8,1,"MULTIPOLYGON (((484111.732 6684705.606, 484131..."
9,1,"MULTIPOLYGON (((489211.59 6684649.604, 489219...."



Tipos de dados:


dn             int32
geometry    geometry
dtype: object


Valores ausentes:


dn          0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,dn,geometry
count,507.0,507
unique,NaN,507
top,NaN,MULTIPOLYGON (((473660.02367629454 6688037.737...
freq,NaN,1
mean,1.0,NaN
std,0.0,NaN
min,1.0,NaN
25%,1.0,NaN
50%,1.0,NaN
75%,1.0,NaN


**Diagnóstico:** 507 registros, geometria `MultiPolygon`, EPSG:31982. Atributos: `dn`; o código de classe vale 1 em todos os registros.

Sem ausências ou linhas inteiramente duplicadas. Não há campo de área ou identificação de bairro.


## 10. Exploração da camada de inundação de 2015

### Camada `poa_inundacao_2015_294cm`

O nome faz referência ao evento de 2015 e a uma cota de 294 cm. A metodologia de delimitação e a referência dessa cota ainda precisam ser confirmadas na documentação. A cota não deve ser interpretada automaticamente como profundidade de água em cada polígono.


**Esclarecimento do glossário:** O glossário descreve esta camada como simulação de 600 cm, em divergência com o ano e a cota indicados no nome. A classificação do produto e sua referência precisam ser esclarecidas antes de usá-lo como representação do evento de 2015.


In [15]:
exploracao_inicial(camadas['name'][4])  # Camada de inundação de 2015


Exploração inicial da camada: poa_inundacao_2015_294cm

Exibindo as primeiras 100 de 450 linhas. As verificações usam a camada completa.


,dn,geometry
0,1,"MULTIPOLYGON (((473660.024 6688037.737, 473696..."
1,1,"MULTIPOLYGON (((488291.616 6685965.656, 488295..."
2,1,"MULTIPOLYGON (((488295.615 6685953.655, 488299..."
3,1,"MULTIPOLYGON (((488299.615 6685945.655, 488303..."
4,1,"MULTIPOLYGON (((488303.615 6685937.655, 488311..."
5,1,"MULTIPOLYGON (((488343.614 6685877.652, 488347..."
6,1,"MULTIPOLYGON (((477739.91 6685465.636, 477743...."
7,1,"MULTIPOLYGON (((473988.015 6687869.731, 474000..."
8,1,"MULTIPOLYGON (((475875.962 6686145.663, 475895..."
9,1,"MULTIPOLYGON (((488843.6 6685049.62, 488851.6 ..."



Tipos de dados:


dn             int32
geometry    geometry
dtype: object


Valores ausentes:


dn          0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,dn,geometry
count,450.0,450
unique,NaN,450
top,NaN,MULTIPOLYGON (((473660.02367629454 6688037.737...
freq,NaN,1
mean,1.0,NaN
std,0.0,NaN
min,1.0,NaN
25%,1.0,NaN
50%,1.0,NaN
75%,1.0,NaN


**Diagnóstico:** 450 registros, geometria `MultiPolygon`, EPSG:31982. Atributos: `dn`; o código de classe vale 1 em todos os registros.

Sem ausências ou linhas inteiramente duplicadas. A descrição documental diverge do nome; ver esclarecimento nesta seção.


## 11. Exploração da camada de inundação de novembro de 2023

### Camada `poa_inundacao_2023_nov_346cm`

O nome faz referência ao evento de novembro de 2023 e a uma cota de 346 cm. A metodologia de delimitação e a referência dessa cota ainda precisam ser confirmadas na documentação. A cota não deve ser interpretada automaticamente como profundidade de água em cada polígono.


In [16]:
exploracao_inicial(camadas['name'][5])  # Camada de inundação de novembro de 2023


Exploração inicial da camada: poa_inundacao_2023_nov_346cm

Exibindo as primeiras 100 de 424 linhas. As verificações usam a camada completa.


,dn,geometry
0,1,"POLYGON ((473660.024 6688037.737, 473660.024 6..."
1,1,"POLYGON ((477739.91 6685465.636, 477739.91 668..."
2,1,"POLYGON ((473988.015 6687869.731, 473988.015 6..."
3,1,"POLYGON ((475875.962 6686145.663, 475875.962 6..."
4,1,"POLYGON ((488847.6 6685061.62, 488847.6 668505..."
5,1,"POLYGON ((488623.606 6684825.611, 488623.606 6..."
6,1,"POLYGON ((484143.731 6684709.606, 484143.731 6..."
7,1,"POLYGON ((484107.732 6684705.606, 484107.732 6..."
8,1,"POLYGON ((483615.746 6684605.602, 483615.746 6..."
9,1,"POLYGON ((489175.591 6684653.604, 489175.591 6..."



Tipos de dados:


dn             int32
geometry    geometry
dtype: object


Valores ausentes:


dn          0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,dn,geometry
count,424.0,424
unique,NaN,424
top,NaN,POLYGON ((473660.02367629454 6688037.737447568...
freq,NaN,1
mean,1.0,NaN
std,0.0,NaN
min,1.0,NaN
25%,1.0,NaN
50%,1.0,NaN
75%,1.0,NaN


**Diagnóstico:** 424 registros, geometria `Polygon`, EPSG:31982. Atributos: `dn`; o código de classe vale 1 em todos os registros.

Sem ausências ou linhas inteiramente duplicadas. Não há campo de área ou identificação de bairro.


## 12. Exploração da camada de inundação de setembro de 2023

### Camada `poa_inundacao_2023_set_318cm`

O nome faz referência ao evento de setembro de 2023 e a uma cota de 318 cm. A metodologia de delimitação e a referência dessa cota ainda precisam ser confirmadas na documentação. A cota não deve ser interpretada automaticamente como profundidade de água em cada polígono.


In [17]:
exploracao_inicial(camadas['name'][6])  # Camada de inundação de setembro de 2023


Exploração inicial da camada: poa_inundacao_2023_set_318cm

Exibindo as primeiras 100 de 526 linhas. As verificações usam a camada completa.


,dn,area_ha,geometry
0,1,NaN,"MULTIPOLYGON (((473660.024 6688037.737, 473696..."
1,1,NaN,"MULTIPOLYGON (((477739.91 6685465.636, 477743...."
2,1,NaN,"MULTIPOLYGON (((473988.015 6687869.731, 474000..."
3,1,NaN,"MULTIPOLYGON (((475875.962 6686145.663, 475895..."
4,1,NaN,"MULTIPOLYGON (((488843.6 6685053.62, 488855.6 ..."
5,1,NaN,"MULTIPOLYGON (((488619.606 6684817.611, 488627..."
6,1,NaN,"MULTIPOLYGON (((488775.602 6684945.616, 488803..."
7,1,NaN,"MULTIPOLYGON (((484147.731 6684709.606, 484155..."
8,1,NaN,"MULTIPOLYGON (((484111.732 6684705.606, 484131..."
9,1,NaN,"MULTIPOLYGON (((489171.591 6684649.604, 489191..."



Tipos de dados:


dn             int32
area_ha      float64
geometry    geometry
dtype: object


Valores ausentes:


dn            0
area_ha     316
geometry      0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,dn,area_ha,geometry
count,526.0,210.000000,526
unique,NaN,NaN,526
top,NaN,NaN,MULTIPOLYGON (((473660.02367629454 6688037.737...
freq,NaN,NaN,1
mean,1.0,10.814757,NaN
std,0.0,74.792753,NaN
min,1.0,0.002000,NaN
25%,1.0,0.002000,NaN
50%,1.0,0.008000,NaN
75%,1.0,0.092500,NaN


**Diagnóstico:** 526 registros, geometria `MultiPolygon`, EPSG:31982. Atributos: `dn` e `area_ha`; o código de classe vale 1 em todos os registros.

`area_ha` possui 316 ausências (60,1%) e 210 valores preenchidos. Não há ausências em `dn` ou geometria nem linhas inteiramente duplicadas. Somar apenas as áreas disponíveis não representa toda a camada.


## 13. Exploração da camada de inundação de 2024

### Camada `poa_inundacao_2024_530cm`

O nome faz referência ao evento de 2024 e a uma cota de 530 cm. A metodologia de delimitação e a referência dessa cota ainda precisam ser confirmadas na documentação. A cota não deve ser interpretada automaticamente como profundidade de água em cada polígono.


**Esclarecimento do glossário:** O glossário descreve uma simulação de 500 cm, enquanto o nome indica 2024 e 530 cm. Não foi definido qual referência está correta; a camada não deve ser tratada como observação direta do evento sem esclarecimento.


In [18]:
exploracao_inicial(camadas['name'][7])  # Camada de inundação de 2024


Exploração inicial da camada: poa_inundacao_2024_530cm

Exibindo as primeiras 100 de 478 linhas. As verificações usam a camada completa.


,DN,geometry
0,1,"MULTIPOLYGON (((473660.024 6688037.737, 473660..."
1,1,"MULTIPOLYGON (((477739.91 6685465.636, 477739...."
2,1,"MULTIPOLYGON (((473988.015 6687869.731, 473988..."
3,1,"MULTIPOLYGON (((475875.962 6686145.663, 475875..."
4,1,"MULTIPOLYGON (((485559.692 6684769.609, 485559..."
5,1,"MULTIPOLYGON (((488631.606 6684845.612, 488631..."
6,1,"MULTIPOLYGON (((485495.694 6684765.608, 485495..."
7,1,"MULTIPOLYGON (((485519.693 6684765.608, 485519..."
8,1,"MULTIPOLYGON (((485323.698 6684757.608, 485323..."
9,1,"MULTIPOLYGON (((485363.697 6684757.608, 485363..."



Tipos de dados:


DN             int32
geometry    geometry
dtype: object


Valores ausentes:


DN          0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,DN,geometry
count,478.0,478
unique,NaN,478
top,NaN,MULTIPOLYGON (((473660.02367629454 6688037.737...
freq,NaN,1
mean,1.0,NaN
std,0.0,NaN
min,1.0,NaN
25%,1.0,NaN
50%,1.0,NaN
75%,1.0,NaN


**Diagnóstico:** 478 registros, geometria `MultiPolygon`, EPSG:31982. Atributos: `DN`; o código de classe vale 1 em todos os registros.

Sem ausências ou linhas inteiramente duplicadas. A descrição documental diverge da cota nominal; ver esclarecimento nesta seção.


## 14. Exploração da camada de inundação simulada a 500 cm

Camada `poa_inundacao_simulada_500cm`: cenário simulado de **5,0 m**, explorado com a mesma função `exploracao_inicial()`. As considerações comuns estão reunidas ao final deste bloco.


In [19]:
exploracao_inicial(camadas['name'][8])  # Camada de inundação simulada a 500 cm


Exploração inicial da camada: poa_inundacao_simulada_500cm

Exibindo as primeiras 100 de 254 linhas. As verificações usam a camada completa.


,DN,geometry
0,1,"MULTIPOLYGON (((473660.024 6688037.737, 473660..."
1,1,"MULTIPOLYGON (((477739.91 6685465.636, 477739...."
2,1,"MULTIPOLYGON (((473988.015 6687869.731, 473988..."
3,1,"MULTIPOLYGON (((475875.962 6686145.663, 475875..."
4,1,"MULTIPOLYGON (((488631.606 6684841.611, 488631..."
5,1,"MULTIPOLYGON (((488623.606 6684837.611, 488623..."
6,1,"MULTIPOLYGON (((485791.685 6684769.609, 485791..."
7,1,"MULTIPOLYGON (((485623.69 6684749.608, 485623...."
8,1,"MULTIPOLYGON (((485151.703 6684753.608, 485151..."
9,1,"MULTIPOLYGON (((484459.722 6684701.606, 484459..."



Tipos de dados:


DN             int32
geometry    geometry
dtype: object


Valores ausentes:


DN          0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,DN,geometry
count,254.0,254
unique,NaN,254
top,NaN,MULTIPOLYGON (((473660.02367629454 6688037.737...
freq,NaN,1
mean,1.0,NaN
std,0.0,NaN
min,1.0,NaN
25%,1.0,NaN
50%,1.0,NaN
75%,1.0,NaN


**Resultado:** 254 registros, geometrias `MultiPolygon`, sem valores ausentes ou linhas inteiramente duplicadas.


## 15. Exploração da camada de inundação simulada a 550 cm

Camada `poa_inundacao_simulada_550cm`: cenário simulado de **5,5 m**, explorado com a mesma função `exploracao_inicial()`. As considerações comuns estão reunidas ao final deste bloco.


In [20]:
exploracao_inicial(camadas['name'][9])  # Camada de inundação simulada a 550 cm


Exploração inicial da camada: poa_inundacao_simulada_550cm

Exibindo as primeiras 100 de 227 linhas. As verificações usam a camada completa.


,DN,geometry
0,1,"MULTIPOLYGON (((473660.024 6688037.737, 473660..."
1,1,"MULTIPOLYGON (((477739.91 6685465.636, 477739...."
2,1,"MULTIPOLYGON (((473988.015 6687869.731, 473988..."
3,1,"MULTIPOLYGON (((475875.962 6686145.663, 475875..."
4,1,"MULTIPOLYGON (((486667.661 6684821.611, 486667..."
5,1,"MULTIPOLYGON (((487127.648 6684821.611, 487127..."
6,1,"MULTIPOLYGON (((486171.675 6684801.61, 486171...."
7,1,"MULTIPOLYGON (((486083.677 6684797.61, 486083...."
8,1,"MULTIPOLYGON (((489835.573 6684797.61, 489835...."
9,1,"MULTIPOLYGON (((486011.679 6684793.61, 486011...."



Tipos de dados:


DN             int32
geometry    geometry
dtype: object


Valores ausentes:


DN          0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,DN,geometry
count,227.0,227
unique,NaN,227
top,NaN,MULTIPOLYGON (((473660.02367629454 6688037.737...
freq,NaN,1
mean,1.0,NaN
std,0.0,NaN
min,1.0,NaN
25%,1.0,NaN
50%,1.0,NaN
75%,1.0,NaN


**Resultado:** 227 registros, geometrias `MultiPolygon`, sem valores ausentes ou linhas inteiramente duplicadas.


## 16. Exploração da camada de inundação simulada a 600 cm

Camada `poa_inundacao_simulada_600cm`: cenário simulado de **6,0 m**, explorado com a mesma função `exploracao_inicial()`. As considerações comuns estão reunidas ao final deste bloco.


In [21]:
exploracao_inicial(camadas['name'][10])  # Camada de inundação simulada a 600 cm


Exploração inicial da camada: poa_inundacao_simulada_600cm

Exibindo as primeiras 100 de 396 linhas. As verificações usam a camada completa.


,DN,geometry
0,1,"POLYGON ((473660.024 6688037.737, 473660.024 6..."
1,1,"POLYGON ((477739.91 6685465.636, 477739.91 668..."
2,1,"POLYGON ((473988.015 6687869.731, 473988.015 6..."
3,1,"POLYGON ((475875.962 6686145.663, 475875.962 6..."
4,1,"POLYGON ((489967.569 6684857.612, 489967.569 6..."
5,1,"POLYGON ((489955.569 6684853.612, 489955.569 6..."
6,1,"POLYGON ((489947.569 6684849.612, 489947.569 6..."
7,1,"POLYGON ((489939.57 6684845.612, 489939.57 668..."
8,1,"POLYGON ((489931.57 6684841.611, 489931.57 668..."
9,1,"POLYGON ((486815.657 6684821.611, 486815.657 6..."



Tipos de dados:


DN             int32
geometry    geometry
dtype: object


Valores ausentes:


DN          0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,DN,geometry
count,396.0,396
unique,NaN,396
top,NaN,POLYGON ((473660.02367629454 6688037.737447568...
freq,NaN,1
mean,1.0,NaN
std,0.0,NaN
min,1.0,NaN
25%,1.0,NaN
50%,1.0,NaN
75%,1.0,NaN


**Resultado:** 396 registros, geometrias `Polygon`, sem valores ausentes ou linhas inteiramente duplicadas.


## 17. Exploração da camada de inundação simulada a 650 cm

Camada `poa_inundacao_simulada_650cm`: cenário simulado de **6,5 m**, explorado com a mesma função `exploracao_inicial()`. As considerações comuns estão reunidas ao final deste bloco.


In [22]:
exploracao_inicial(camadas['name'][11])  # Camada de inundação simulada a 650 cm


Exploração inicial da camada: poa_inundacao_simulada_650cm

Exibindo as primeiras 100 de 220 linhas. As verificações usam a camada completa.


,DN,geometry
0,1,"POLYGON ((473660.024 6688037.737, 473660.024 6..."
1,1,"POLYGON ((477739.91 6685465.636, 477739.91 668..."
2,1,"POLYGON ((473988.015 6687869.731, 473988.015 6..."
3,1,"POLYGON ((475875.962 6686145.663, 475875.962 6..."
4,1,"POLYGON ((490059.566 6684901.614, 490059.566 6..."
5,1,"POLYGON ((490051.566 6684897.614, 490051.566 6..."
6,1,"POLYGON ((490039.567 6684889.613, 490039.567 6..."
7,1,"POLYGON ((490031.567 6684885.613, 490031.567 6..."
8,1,"POLYGON ((490023.567 6684881.613, 490023.567 6..."
9,1,"POLYGON ((490015.568 6684877.613, 490015.568 6..."



Tipos de dados:


DN             int32
geometry    geometry
dtype: object


Valores ausentes:


DN          0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,DN,geometry
count,220.0,220
unique,NaN,220
top,NaN,POLYGON ((473660.02367629454 6688037.737447568...
freq,NaN,1
mean,1.0,NaN
std,0.0,NaN
min,1.0,NaN
25%,1.0,NaN
50%,1.0,NaN
75%,1.0,NaN


**Resultado:** 220 registros, geometrias `Polygon`, sem valores ausentes ou linhas inteiramente duplicadas.


## 18. Exploração da camada de inundação simulada a 700 cm

Camada `poa_inundacao_simulada_700cm`: cenário simulado de **7,0 m**, explorado com a mesma função `exploracao_inicial()`. As considerações comuns estão reunidas ao final deste bloco.


In [23]:
exploracao_inicial(camadas['name'][12])  # Camada de inundação simulada a 700 cm


Exploração inicial da camada: poa_inundacao_simulada_700cm

Exibindo as primeiras 100 de 312 linhas. As verificações usam a camada completa.


,DN,geometry
0,1,"POLYGON ((473660.024 6688037.737, 473660.024 6..."
1,1,"POLYGON ((477739.91 6685465.636, 477739.91 668..."
2,1,"POLYGON ((473988.015 6687869.731, 473988.015 6..."
3,1,"POLYGON ((475875.962 6686145.663, 475875.962 6..."
4,1,"POLYGON ((490135.564 6684949.616, 490135.564 6..."
5,1,"POLYGON ((490127.564 6684945.616, 490127.564 6..."
6,1,"POLYGON ((490119.565 6684941.615, 490119.565 6..."
7,1,"POLYGON ((490115.565 6684937.615, 490115.565 6..."
8,1,"POLYGON ((488387.613 6684745.608, 488387.613 6..."
9,1,"POLYGON ((478239.896 6685577.64, 478239.896 66..."



Tipos de dados:


DN             int32
geometry    geometry
dtype: object


Valores ausentes:


DN          0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Projected CRS: EPSG:31982>
Name: SIRGAS 2000 / UTM zone 22S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 54°W and 48°W, northern and southern hemispheres, onshore and offshore. In remainder of South America - between 54°W and 48°W, southern hemisphere, onshore and offshore.
- bounds: (-54.0, -54.18, -47.99, 7.04)
Coordinate Operation:
- name: UTM zone 22S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


Estatísticas descritivas:


,DN,geometry
count,312.0,312
unique,NaN,312
top,NaN,POLYGON ((473660.02367629454 6688037.737447568...
freq,NaN,1
mean,1.0,NaN
std,0.0,NaN
min,1.0,NaN
25%,1.0,NaN
50%,1.0,NaN
75%,1.0,NaN


**Resultado:** 312 registros, geometrias `Polygon`, sem valores ausentes ou linhas inteiramente duplicadas.


## Síntese das camadas de inundação simulada

As cinco camadas estão em **EPSG:31982**, possuem `DN = 1` e geometria, sem ausências ou linhas inteiramente duplicadas. As cotas abaixo são as **nominais dos nomes**, não profundidades de água:

| Camada | Cenário | Registros | Geometria |
| --- | --- | ---: | --- |
| `poa_inundacao_simulada_500cm` | 5,0 m | 254 | MultiPolygon |
| `poa_inundacao_simulada_550cm` | 5,5 m | 227 | MultiPolygon |
| `poa_inundacao_simulada_600cm` | 6,0 m | 396 | Polygon |
| `poa_inundacao_simulada_650cm` | 6,5 m | 220 | Polygon |
| `poa_inundacao_simulada_700cm` | 7,0 m | 312 | Polygon |

**Divergência documental:** o glossário atribui 550 cm à camada de 500 cm e 600 cm à de 550 cm. A referência correta permanece pendente. Quantidade de polígonos não mede extensão inundada.

As simulações foram excluídas da preparação; a justificativa e o escopo final estão na seção 51.


## 19. Exploração da camada de assistência social

### Camada `poa_saude_assistencia_social`

O nome sugere estabelecimentos relacionados à assistência social em Porto Alegre. A definição dos serviços, a abrangência e o período de referência precisam ser confirmados antes de interpretar a camada como um cadastro completo ou atualizado.


In [24]:
exploracao_inicial(camadas['name'][13])  # Camada de assistência social


Exploração inicial da camada: poa_saude_assistencia_social

Exibindo as primeiras 100 de 220 linhas. As verificações usam a camada completa.


,Name,descriptio,C__d,Sigla_Mant,Matriz_Man,EntidadeEx,Descri____,Modalidade,Modalida_1,Modalida_2,Categoria,Endere__o,N__mero,Complement,Munic__pio,UF,CEP,Telefones,Bairro,Creas_Refe,Regi__o_OP,Latidude,Longitude,Outros_Ser,CDL_GEO,NUM_GEO,ENDERE__O_,geometry
0,Centro Dia do Idoso,Cód: 1<br>Sigla Mantenedora: FASC<br>Matriz Ma...,1.0,FASC,FASC,CDI - Centro de Convivência do Idoso Nascer do...,Oferece atendimento especializado com equipe m...,NaN,NaN,NaN,Rua,IRENE CAPPONI SANTIAGO,290,NaN,PORTO ALEGRE,RS,9.104007E7,33615664_33414607_,Cristo Redentor,CREAS NORTE E NOROESTE,NOROESTE,-300040662,-5.11591865E8,None,8480253,290.0,"Rua IRENE CAPPONI SANTIAGO, 290 -PORTO ALEGRE",POINT Z (-51.1592 -30.0039 0)
1,Centro Dia do Idoso,Cód: 2<br>Sigla Mantenedora: FASC<br>Matriz Ma...,2.0,FASC,FASC,CDI - Centro de Convivência Portal da Felicidade,Oferece atendimento especializado com equipe m...,NaN,NaN,NaN,Rua,SILVIO SILVEIRA SOARES,2713,NaN,PORTO ALEGRE,RS,9.191046E7,32412803,Camaqua,CREAS SUL/CENTRO SUL,CENTRO SUL,-301047885,-5.12330359E8,None,7770316,2713.0,"Rua SILVIO SILVEIRA SOARES, 2713 -PORTO ALEGRE",POINT Z EMPTY
2,CENTRO POP,Cód: 3<br>Sigla Mantenedora: FASC<br>Matriz Ma...,3.0,FASC,FASC,CENTRO POP I,É um Centro de Referência Especializado para P...,NaN,NaN,NaN,Rua,ALMIRANTE ÁLVARO ALBERTO DA MOTA E SILVA,SN,NaN,PORTO ALEGRE,RS,9.016013E7,32255602 32218578,Menino Deus,"CREAS CENTRO,ILHAS,HUMAITÁ/NAVEGANTES",CENTRO,-30045879,-5.1220433E7,None,7876170,10.0,"Rua ALMIRANTE ÁLVARO ALBERTO DA MOTA E SILVA, ...",POINT Z EMPTY
3,CENTRO POP,Cód: 4<br>Sigla Mantenedora: FASC<br>Matriz Ma...,4.0,FASC,FASC,CENTRO POP II,É um Centro de Referência Especializado para P...,NaN,NaN,NaN,Rua,VOLUNTARIOS DA PATRIA,1613,NaN,PORTO ALEGRE,RS,9.0030001E7,32263544,Centro Historico,"CREAS CENTRO,ILHAS,HUMAITÁ/NAVEGANTES",CENTRO,-300200225,-5.12143104E8,None,7878168,1613.0,"Rua VOLUNTARIOS DA PATRIA, 1613 -PORTO ALEGRE",POINT Z EMPTY
4,CRAS,Cód: 5<br>Sigla Mantenedora: FASC<br>Matriz Ma...,5.0,FASC,FASC,CRAS 5ª Unidade Restinga,O Centro de Referência de Assistência Social (...,NaN,15 a 18 anos - PJ,60 anos ou mais,Rua,SÃO JOÃO CALABRIA,20,NaN,PORTO ALEGRE,RS,9.1790872E7,32508302,Restinga,CREAS RESTINGA E EXTREMO SUL,RESTINGA,-30147194,-5.112972E7,None,8765013,20.0,"Rua SÃO JOÃO CALABRIA, 20 -PORTO ALEGRE",POINT Z EMPTY
5,CRAS,Cód: 6<br>Sigla Mantenedora: FASC<br>Matriz Ma...,6.0,FASC,FASC,CRAS Ampliado Norte,O Centro de Referência de Assistência Social (...,6 a 14 anos,15 a 18 anos - PJ,60 anos ou mais,Avenida,PAULO GOMES DE OLIVEIRA,200,NaN,PORTO ALEGRE,RS,9.112057E7,33641194 33506089 33640157,Sarandi,CREAS NORTE E NOROESTE,NORTE,-29980270,-5.1126402E7,None,8783060,200.0,"Avenida PAULO GOMES DE OLIVEIRA, 200 -PORTO A...",POINT Z EMPTY
6,CRAS,Cód: 7<br>Sigla Mantenedora: FASC<br>Matriz Ma...,7.0,FASC,FASC,CRAS Centro,O Centro de Referência de Assistência Social (...,NaN,15 a 18 anos - PJ,60 anos ou mais,Rua,DR SEBASTIÃO LEÃO,276,NaN,PORTO ALEGRE,RS,9.005009E7,32895049,Azenha,"CREAS CENTRO,ILHAS,HUMAITÁ/NAVEGANTES",CENTRO,-300440431,-5.1215645E7,None,7876121,276.0,"Rua DR. SEBASTIÃO LEÃO, 276 -PORTO ALEGRE",POINT Z EMPTY
7,CRAS,Cód: 8<br>Sigla Mantenedora: FASC<br>Matriz Ma...,8.0,FASC,FASC,CRAS Centro Sul,O Centro de Referência de Assistência Social (...,6 a 14 anos,15 a 18 anos - PJ,60 anos ou mais,Rua,ARROIO GRANDE,50,NaN,PORTO ALEGRE,RS,9.174018E7,32494343_32494332,Cavalhada,CREAS SUL/CENTRO SUL,CENTRO SUL,-301284925,-5.10941494E8,None,7770282,50.0,"Rua ARROIO GRANDE , 50 -PORTO ALEGRE",POINT Z (-51.23017 -30.10133 0)
8,CRAS,Cód: 9<br>Sigla Mantenedora: FASC<br>Matriz Ma...,9.0,FASC,FASC,CRAS Cristal,O Centro de Referência de Assistência Social (...,6 a 14 anos,15 a 18 anos - PJ,60 anos ou mais,Rua,CURUPAITI,27,NaN,PORTO ALEGRE,RS,9.082009E7,32424930,Cristal,CREAS GLÓRIA CRUZEIRO CRISTAL,CRISTAL,-300793729,-5.12376879E8,None,7771686,27.0,"Rua CURUPAITI, 27 -PORTO ALEGRE",POINT Z EMPTY
9,CRAS,Cód: 10<br>Sigla Mantenedora: FASC<br>Matriz M...,10.0,FASC,FA


Tipos de dados:


Name               str
descriptio         str
C__d               str
Sigla_Mant         str
Matriz_Man         str
EntidadeEx         str
Descri____         str
Modalidade         str
Modalida_1         str
Modalida_2         str
Categoria          str
Endere__o          str
N__mero            str
Complement         str
Munic__pio         str
UF                 str
CEP                str
Telefones          str
Bairro             str
Creas_Refe         str
Regi__o_OP         str
Latidude           str
Longitude          str
Outros_Ser      object
CDL_GEO            str
NUM_GEO            str
ENDERE__O_         str
geometry      geometry
dtype: object


Valores ausentes:


Name            0
descriptio      0
C__d            0
Sigla_Mant      0
Matriz_Man      0
EntidadeEx      0
Descri____      0
Modalidade    105
Modalida_1    139
Modalida_2    181
Categoria       0
Endere__o       0
N__mero         0
Complement    196
Munic__pio      0
UF              0
CEP             0
Telefones     165
Bairro          0
Creas_Refe      0
Regi__o_OP      0
Latidude        0
Longitude       0
Outros_Ser    220
CDL_GEO         0
NUM_GEO         0
ENDERE__O_      0
geometry        0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Name,descriptio,C__d,Sigla_Mant,Matriz_Man,EntidadeEx,Descri____,Modalidade,Modalida_1,Modalida_2,...,Bairro,Creas_Refe,Regi__o_OP,Latidude,Longitude,Outros_Ser,CDL_GEO,NUM_GEO,ENDERE__O_,geometry
count,220,220,220,220,220,220,220,115,81,39,...,220,220,220,220,220,0,220,220,220,220
unique,7,220,220,137,133,220,9,1,2,1,...,55,10,17,210,209,0,160,153,201,50
top,SCFV,Cód: 1<br>Sigla Mantenedora: FASC<br>Matriz Ma...,1.0,FASC,FASC,CDI - Centro de Convivência do Idoso Nascer do...,O acesso a estes Serviços é através do CRAS de...,6 a 14 anos,14 a 18 anos,60 anos ou mais,...,Rubem Berta,CREAS GLÓRIA CRUZEIRO CRISTAL,LESTE,-299796427,-5.11980239E8,NaN,8978579,10.0,"ESTRADA ANTONIO SEVERINO, 1493 -PORTO ALEGRE",POINT Z EMPTY
freq,129,1,1,35,35,1,108,115,59,39,...,22,46,26,2,2,NaN,4,8,3,171


### Resultados iniciais — assistência social

A camada possui **220 registros** e **28 colunas**, incluindo a geometria. As geometrias lidas são `Point`, com dimensão Z, em **EPSG:4326**. Foram encontradas **0 linhas inteiramente duplicadas**.

| Atributo com ausências | Registros ausentes |
| --- | ---: |
| `Modalidade` | 105 |
| `Modalida_1` | 139 |
| `Modalida_2` | 181 |
| `Complement` | 196 |
| `Telefones` | 165 |
| `Outros_Ser` | 220 |

A geometria apresenta **0 valores ausentes**, mas a saída de `describe()` mostra **171 ocorrências de `POINT Z EMPTY`** (77,7% dos registros). Assim, ausência de nulos não significa disponibilidade de localização: esses registros têm geometria vazia e não permitem cruzamentos espaciais diretamente. Os outros 49 registros ainda precisam de avaliação de validade e localização. A dimensão Z também não confirma que exista uma altitude útil para a análise.

### Possíveis caminhos de análise

A localização dos estabelecimentos com geometria utilizável pode apoiar mapas de distribuição de serviços e possíveis cruzamentos com bairros e manchas de inundação, caso esses caminhos sejam considerados relevantes. Os atributos podem ajudar a identificar e caracterizar os serviços, após esclarecer seus significados e sua qualidade.

Um ponto dentro de uma mancha de inundação indica uma relação espacial com a área; isso, isoladamente, não comprova danos, interrupção do atendimento ou quantidade de pessoas afetadas. A decisão de aproveitamento permanece aberta. Nenhum registro ou atributo original foi alterado.


### Avaliação provisória — assistência social

Camada **complementar e de baixa prioridade para análises espaciais**: 171 das 220 geometrias estão vazias. Os 49 registros restantes não representam necessariamente a distribuição do conjunto.

Uma possível reavaliação depende de verificar se `Latidude` e `Longitude` permitem recuperar localizações válidas. Essa investigação ainda não foi realizada.


## 20. Exploração da camada de saúde — SUS

### Camada `poa_saude_sus`

O nome sugere estabelecimentos relacionados ao SUS em Porto Alegre. A definição dos serviços, a abrangência e o período de referência precisam ser confirmados antes de interpretar a camada como um cadastro completo ou atualizado.


**Esclarecimento do glossário:** A descrição do glossário menciona uma mancha de inundação de novembro de 2033, incompatível com os atributos de estabelecimentos de saúde encontrados. Essa descrição não foi adotada como definição válida da camada.


In [25]:
exploracao_inicial(camadas['name'][14])  # Camada de saúde — SUS


Exploração inicial da camada: poa_saude_sus

Exibindo as primeiras 100 de 353 linhas. As verificações usam a camada completa.


,Name,classe,id01,geometry
0,CLINICAS,hospital,1,POINT Z (-51.20804 -30.03912 0)
1,PRONTO SOCORRO,hospital,2,POINT Z (-51.2096 -30.03557 0)
2,COMPLEXO SANTA CASA,hospital,3,POINT Z (-51.21986 -30.0288 0)
3,HMIPV,hospital,4,POINT Z (-51.21467 -30.02863 0)
4,HOSPITAL BENEFICENCIA PORTUGUESA,hospital,5,POINT Z (-51.21885 -30.02861 0)
5,FEMINA,hospital,6,POINT Z (-51.20725 -30.02819 0)
6,MOINHOS DE VENTO,hospital,7,POINT Z (-51.20905 -30.02627 0)
7,CENTRO DE DIÁLISE E TRANSPLANTE,hospital,8,POINT Z (-51.20353 -30.01854 0)
8,PETROPOLIS,hospital,9,POINT Z (-51.19432 -30.03972 0)
9,ULBRA,hospital,10,POINT Z (-51.19794 -30.03657 0)



Tipos de dados:


Name             str
classe           str
id01           int64
geometry    geometry
dtype: object


Valores ausentes:


Name        0
classe      0
id01        0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Name,classe,id01,geometry
count,353,353,353.000000,353
unique,227,14,NaN,219
top,RESTINGA,ubs,NaN,POINT Z (-51.232379 -30.027663000000366 0)
freq,5,106,NaN,13
mean,NaN,NaN,177.000000,NaN
std,NaN,NaN,102.046558,NaN
min,NaN,NaN,1.000000,NaN
25%,NaN,NaN,89.000000,NaN
50%,NaN,NaN,177.000000,NaN
75%,NaN,NaN,265.000000,NaN


### Resultados iniciais — saúde SUS

A camada possui **353 registros** e **4 colunas**, incluindo a geometria. As geometrias lidas são `Point`, com dimensão Z, em **EPSG:4326**. Foram encontradas **0 linhas inteiramente duplicadas**.

| Atributo com ausências | Registros ausentes |
| --- | ---: |
| Nenhum atributo | 0 |

A geometria apresenta **0 valores ausentes**. Além disso, foram identificadas **0 geometrias vazias**. Essas contagens não verificam a validade das demais geometrias ou sua localização na cobertura territorial. A dimensão Z também não confirma que exista uma altitude útil para a análise.

O campo `classe` apresenta **14 categorias**, incluindo UBS (106 registros), USF (89), CAPS (34), hospitais (28) e bases SAMU (28). Esses totais descrevem registros classificados na camada, não necessariamente endereços ou estabelecimentos físicos distintos.

A saída de `describe()` mostra **219 geometrias distintas para 353 registros** e **227 nomes distintos**. Essa repetição pode refletir serviços diferentes no mesmo local, duplicidades de cadastro ou outras situações; a ausência de linhas inteiramente duplicadas não esclarece esses casos. Uma eventual contagem de unidades físicas dependeria de investigação adicional.

### Possíveis caminhos de análise

A localização dos estabelecimentos pode apoiar mapas de distribuição de serviços e possíveis cruzamentos com bairros e manchas de inundação, caso esses caminhos sejam considerados relevantes. Os atributos podem ajudar a identificar e caracterizar os serviços, após esclarecer seus significados e sua qualidade.

Um ponto dentro de uma mancha de inundação indica uma relação espacial com a área; isso, isoladamente, não comprova danos, interrupção do atendimento ou quantidade de pessoas afetadas. A decisão de aproveitamento permanece aberta. Nenhum registro ou atributo original foi alterado.


## 21. Exploração das unidades de atenção primária — 09/05/2024

### Camada `poa_saude_unidades_atencao_primaria_09052024`

A exploração segue a mesma função `exploracao_inicial()`, com inspeção das DataFrame com limite de 100 linhas na visualização inicial, tipos de dados, ausências, linhas inteiramente duplicadas, CRS e estatísticas descritivas da camada completa.

O nome sugere um levantamento de unidades de atenção primária com referência a **9 de maio de 2024**. A data exata de atualização, a abrangência e o significado dos atributos precisam ser confirmados na documentação antes de interpretar os registros como um retrato completo dos serviços naquele dia.


In [26]:
exploracao_inicial(camadas['name'][15])  # Unidades de atenção primária — 09/05/2024


Exploração inicial da camada: poa_saude_unidades_atencao_primaria_09052024

Exibindo as primeiras 100 de 136 linhas. As verificações usam a camada completa.


,Nome,End,End_cidade,Latitude,Longitude,Bairro,Cidade,Situacao,Telefone,Atendimento,geometry
0,Clínica da Família José Mauro Ceratti Lopes,"Estrada João Antônio da Silveira, 3330","Estrada João Antônio da Silveira, 3330, Restin...",-30.143014,-51.128977,Restinga,Porto Alegre,Aberta,"(51) 3289-5203 (WhatsApp), 3289-5201, 3289-520...",De segunda a sexta das 7h às 22h. Fechada par ...,POINT (-51.12898 -30.14301)
1,Clínica da Família Álvaro Difini,"Rua Álvaro Difini, 520","Rua Álvaro Difini, 520, Restinga, Porto Alegre",-30.148886,-51.146084,Restinga,Porto Alegre,Aberta,"(51) 4076-5011 (WhatsApp), 4076-5012, 4076-501...",De segunda a sexta das 7h às 22h. Fechada para...,POINT (-51.14608 -30.14889)
2,US Alto Erechim,"Rua Doutor Ney Cabral, 581","Rua Doutor Ney Cabral, 581, Nonoai, Porto Alegre",-30.089942,-51.212331,Nonoai,Porto Alegre,Aberta,"(51) 3289-5589 (WhatsApp), 3289-2347",De segunda a sexta das 08h às 17h. Fechada par...,POINT (-51.21233 -30.08994)
3,US Aparício Borges,"Rua São Miguel, 487","Rua São Miguel, 487, Coronel Aparício Borges, ...",-30.074953,-51.191408,Coronel Aparício Borges,Porto Alegre,Aberta,"(51) 3289-5774 (WhatsApp), 3289-5773, 3289-8265",De segunda a sexta das 08h às 17h. Fechada par...,POINT (-51.19141 -30.07495)
4,US Asa Branca,"Rua 25 de Outubro, 318","Rua 25 de Outubro, 318, Sarandi, Porto Alegre",-29.978815,-51.121226,Sarandi,Porto Alegre,Fechada,"(51) 3356-3479 (WhatsApp), Recepção: 3289-5657...",De segunda a sexta das 08h às 17h.,POINT (-51.12123 -29.97881)
5,US Assis Brasil,"Avenida Assis Brasil, 6615","Avenida Assis Brasil, 6615, Sarandi, Porto Alegre",-29.993569,-51.128466,Sarandi,Porto Alegre,Fechada,"(51) 3289-5657 (WhatsApp), 3289-5656",De segunda a sexta das 07h às 19h.,POINT (-51.12847 -29.99357)
6,US Bananeiras,"Avenida Coronel Aparício Borges, 2494","Avenida Coronel Aparício Borges, 2494, Coronel...",-30.065276,-51.186758,Coronel Aparício Borges,Porto Alegre,Aberta,"(51) 3289-8227 (WhatsApp), 3289-8228, 3289-8229",De segunda a sexta das 07h às 19h. Fechada par...,POINT (-51.18676 -30.06528)
7,US Barão de Bagé,"Rua Araruama, 487","Rua Araruama, 487, Vila Jardim, Porto Alegre",-30.035311,-51.157603,Vila Jardim,Porto Alegre,Aberta,"(51) 3345-1503 (WhatsApp), 3334-7196, 3357-2476",De segunda a sexta das 08h às 18h. Fechada par...,POINT (-51.1576 -30.03531)
8,US Batista Flores,"Avenida Serafin Machado, 215 - Vila Batista Fl...","Avenida Serafin Machado, 215 - Vila Batista Fl...",-30.027876,-51.105048,Mario Quintana,Porto Alegre,Aberta,"(51) 3289-5614 (WhatsApp), 3289-5615",De segunda à sexta das 08 às 12h e das 13 às 1...,POINT (-51.10505 -30.02788)
9,US Beco do Adelar,"Avenida Juca Batista, 3480","Avenida Juca Batista, 3480, Campo Novo, Porto ...",-30.154835,-51.199226,Campo Novo,Porto Alegre,Aberta,"(51) 3289-5690 (WhatsApp), 3289-5554, 3289-5689",De segunda a sexta das 07h às 19h. Fechada par...,POINT (-51.19923 -30.15483)



Tipos de dados:


Nome                str
End                 str
End_cidade          str
Latitude        float64
Longitude       float64
Bairro              str
Cidade              str
Situacao            str
Telefone            str
Atendimento         str
geometry       geometry
dtype: object


Valores ausentes:


Nome           0
End            0
End_cidade     0
Latitude       0
Longitude      0
Bairro         0
Cidade         0
Situacao       0
Telefone       0
Atendimento    0
geometry       0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Nome,End,End_cidade,Latitude,Longitude,Bairro,Cidade,Situacao,Telefone,Atendimento,geometry
count,136,136,136,136.000000,136.000000,136,136,136,136,136,136
unique,136,135,136,NaN,NaN,56,2,2,136,94,135
top,Clínica da Família José Mauro Ceratti Lopes,"Rua Paulo Maciel, 220","Estrada João Antônio da Silveira, 3330, Restin...",NaN,NaN,Rubem Berta,Porto Alegre,Aberta,"(51) 3289-5203 (WhatsApp), 3289-5201, 3289-520...",De segunda a sexta das 08h às 17h.,POINT (-51.2140794 -30.1307424)
freq,1,2,1,NaN,NaN,12,135,104,1,20,2
mean,NaN,NaN,NaN,-30.068102,-51.162455,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,0.058047,0.045688,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,-30.238689,-51.254659,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,-30.109519,-51.199790,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,-30.065375,-51.157673,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,-30.024386,-51.123117,NaN,NaN,NaN,NaN,NaN,NaN


### Resultados iniciais e limitações

A camada possui **136 registros e 11 colunas**, com geometrias `Point` em **EPSG:4326**. Foram encontradas **0 linhas inteiramente duplicadas**, **0 geometrias nulas** e **0 geometrias vazias**.

| Atributo com ausências | Registros ausentes |
| --- | ---: |
| Nenhum atributo | 0 |

A geometria apresenta **135 valores distintos**. As contagens iniciais não comprovam a validade ou a precisão das localizações, nem a unicidade dos estabelecimentos. Campos de coordenadas e endereço podem apoiar uma eventual conferência, mas sua consistência com as geometrias ainda não foi verificada.

O campo `Nome` apresenta 136 valores distintos, enquanto há 135 geometrias distintas: dois registros compartilham uma geometria. Essa coincidência não comprova duplicação de estabelecimento.

O campo `Situacao` contém **104 registros com “Aberta” e 32 com “Fechada”**. Essas categorias não demonstram, por si só, que os fechamentos foram causados pela inundação. Há também uma inconsistência candidata à investigação: `Cidade` contém **135 valores “Porto Alegre” e um valor “Fechada”**, que não corresponde a um nome de município. Não foi aplicada correção automática.

### Possíveis caminhos de análise

A camada pode apoiar mapas de distribuição das unidades e possíveis cruzamentos com bairros e manchas de inundação. Os campos `Situacao` e `Atendimento` podem apoiar uma avaliação do atendimento, após confirmar suas definições e o período de referência; localização em uma mancha não comprova interrupção de serviço.

Outro possível caminho é avaliar sua relação com `poa_saude_sus` e com a camada de atenção primária de 10/05/2024, disponível no mesmo banco. Isso exigiria verificar a equivalência dos registros e das definições antes de inferir mudanças entre datas ou somar unidades, evitando contagens repetidas.


### Hipótese sobre a US Nossa Senhora Aparecida

O registro `fid = 76`, no Sarandi, informa `Cidade = "Fechada"` e `Situacao = "Aberta"`; o endereço composto também contém “Fechada”. A hipótese considerada é de fechamento no contexto das enchentes, mas os campos contraditórios e a descrição de reuniões regulares **não confirmam essa hipótese**.

Um possível caminho é cruzar sua posição (**-29.9886944, -51.1168529**) com uma mancha temporalmente compatível com 09/05/2024. Isso avaliaria exposição; a confirmação do funcionamento dependeria de registros ou comunicados do período.


## 22. Exploração da infraestrutura de água — ETAs e EBABs

### Camada `poa_sb_agua_etas`


**Esclarecimento do glossário:** O glossário confirma que a camada reúne estações de tratamento e bombeamento de água bruta, esclarecendo a presença de ETAs e EBABs. Essa definição não preenche as situações operacionais ausentes.


In [27]:
exploracao_inicial(camadas['name'][16])  # Estações de tratamento de água


Exploração inicial da camada: poa_sb_agua_etas



,Name,Nome,Status_07_05,Status_08_05,Status_09_05,lat,long,geometry
0,ETA Falha,ETA Moinhos de Vento,Falha,Falha,Falha,-30.026290,-51.205816,POINT Z (-51.20582 -30.02629 0)
1,ETA Parcial,ETA São João,Parcial,Parcial,Parcial,-30.013243,-51.185272,POINT Z (-51.18527 -30.01324 0)
2,ETA Parcial,ETA Menino Deus,Falha,Parcial,Parcial,-30.062702,-51.223854,POINT Z (-51.22385 -30.0627 0)
3,ETA Falha,ETA Ilhas,Falha,Falha,Falha,-30.013732,-51.259930,POINT Z (-51.25993 -30.01373 0)
4,ETA Parcial,ETA Belém Novo,Parcial,Parcial,Parcial,-30.210016,-51.177764,POINT Z (-51.17776 -30.21002 0)
5,ETA Parcial,ETA Tristeza,Falha,Parcial,Parcial,-30.099200,-51.253466,POINT Z (-51.25347 -30.0992 0)
6,EBAB,EBAB Moinhos de Vento,,,,-30.014573,-51.215040,POINT Z (-51.21504 -30.01457 6.39552)
7,EBAB,EBAB Menino Deus,,,,-30.060710,-51.231097,POINT Z (-51.2311 -30.06071 17.27334)
8,EBAB,EBAB Belém Novo,,,,-30.213403,-51.180270,POINT Z (-51.18027 -30.2134 4.35261)
9,EBAB,EBAB São João,,,,-30.015097,-51.212610,POINT Z (-51.21261 -30.0151 6.14043)



Tipos de dados:


Name                 str
Nome                 str
Status_07_05         str
Status_08_05         str
Status_09_05         str
lat              float64
long             float64
geometry        geometry
dtype: object


Valores ausentes:


Name            0
Nome            0
Status_07_05    0
Status_08_05    0
Status_09_05    0
lat             0
long            0
geometry        0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Name,Nome,Status_07_05,Status_08_05,Status_09_05,lat,long,geometry
count,12,12,12,12,12,12.000000,12.000000,12
unique,3,12,3,3,3,NaN,NaN,12
top,EBAB,ETA Moinhos de Vento,,,,NaN,NaN,POINT Z (-51.2058162854978 -30.0262903578675 0)
freq,6,1,6,6,6,NaN,NaN,1
mean,NaN,NaN,NaN,NaN,NaN,-30.069996,-51.221810,NaN
std,NaN,NaN,NaN,NaN,NaN,0.073519,0.031048,NaN
min,NaN,NaN,NaN,NaN,NaN,-30.213403,-51.260270,NaN
25%,NaN,NaN,NaN,NaN,NaN,-30.097746,-51.254184,NaN
50%,NaN,NaN,NaN,NaN,NaN,-30.043500,-51.219447,NaN
75%,NaN,NaN,NaN,NaN,NaN,-30.014363,-51.200680,NaN


### Resultados e limitações

A camada reúne **seis registros de ETAs e seis de EBABs**, com oito colunas e 12 geometrias distintas em **EPSG:4326**. Não foram encontradas geometrias nulas ou vazias nem linhas inteiramente duplicadas; a precisão das localizações ainda não foi validada.

Os campos de situação apresentam:

| Campo | Falha | Parcial | Texto vazio |
| --- | ---: | ---: | ---: |
| `Status_07_05` | 4 | 2 | 6 |
| `Status_08_05` | 2 | 4 | 6 |
| `Status_09_05` | 2 | 4 | 6 |

Os seis textos vazios de cada data correspondem às EBABs e **não são detectados por `isna()`**. Representam informação não preenchida, sem indicar funcionamento normal ou falha. Uma eventual complementação exigiria fontes históricas específicas para cada dia; nenhum valor foi inferido.

O ano e o horário de referência, o significado de “Falha” e “Parcial” e a função das EBABs precisam ser confirmados na documentação. Os rótulos de `Name` também não devem ser considerados indicadores válidos para todas as datas.

### Possíveis caminhos

A situação das seis ETAs poderia apoiar comparações entre as datas, enquanto as EBABs poderiam contribuir apenas com a localização. Mapas e cruzamentos com bairros, manchas de inundação ou outras camadas de abastecimento são possibilidades, após validação. Relações espaciais não comprovam interrupção de serviço, conexão operacional ou população atendida.


## 23. Exploração das fontes públicas de água

### Camada `poa_sb_agua_fontes_publicas`


In [28]:
exploracao_inicial(camadas['name'][17])  # Fontes públicas de água


Exploração inicial da camada: poa_sb_agua_fontes_publicas



,Name,Coliformes,E. Coli,N amostras,%PresColif,%PresEcoli,IniAnalise,UltAnalise,geometry
0,Afonso Lourenço Mariante,Presença,Ausência,115.0,52.2,3.5,2011-03-30,2023-12-28,POINT Z (-51.16298 -30.11253 0)
1,Beco dos Rossato,Presença,Presença,108.0,99.1,80.6,2011-12-20,2023-12-27,POINT Z (-51.20245 -30.14522 0)
2,Carlos Trein Filho,Presença,Ausência,145.0,93.1,74.5,2005-01-26,2023-12-26,POINT Z (-51.18781 -30.02292 0)
3,Costa do Cerro,Presença,Presença,114.0,99.1,95.6,2011-08-24,2023-12-27,POINT Z (-51.12611 -30.19253 0)
4,Fábio Luiz Silveira,Presença,Presença,138.0,19.6,0.7,2005-01-19,2023-12-26,POINT Z (-51.12449 -30.01409 0)
5,Gruta da Glória,Ausência,Ausência,132.0,28.0,5.3,2005-01-11,2023-12-28,POINT Z (-51.18978 -30.08491 0)
6,Irmã Teresilda Steffen,Presença,Presença,79.0,84.8,36.7,2016-05-04,2023-12-26,POINT Z (-51.11612 -30.02608 0)
7,Jabuticabas,Presença,Ausência,74.0,98.6,95.9,2017-05-31,2023-12-28,POINT Z (-51.1077 -30.11998 0)
8,Praça Federico Garcia Lorca,Presença,Presença,126.0,99.2,74.6,2005-01-26,2023-12-27,POINT Z (-51.21299 -30.14932 0)
9,Fonte Deputado Asterio,NaN,NaN,NaN,NaN,NaN,NaT,NaT,POINT Z (-51.20277 -30.09604 0)



Tipos de dados:


Name                     str
Coliformes               str
E. Coli                  str
N amostras           float64
%PresColif           float64
%PresEcoli           float64
IniAnalise    datetime64[ms]
UltAnalise    datetime64[ms]
geometry            geometry
dtype: object


Valores ausentes:


Name          0
Coliformes    2
E. Coli       2
N amostras    2
%PresColif    2
%PresEcoli    2
IniAnalise    2
UltAnalise    2
geometry      0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Name,Coliformes,E. Coli,N amostras,%PresColif,%PresEcoli,IniAnalise,UltAnalise,geometry
count,11,9,9,9.000000,9.000000,9.000000,9,9,11
unique,11,2,2,NaN,NaN,NaN,NaN,NaN,11
top,Afonso Lourenço Mariante,Presença,Presença,NaN,NaN,NaN,NaN,NaN,POINT Z (-51.16297699642466 -30.11253091837759 0)
freq,1,8,5,NaN,NaN,NaN,NaN,NaN,1
mean,NaN,NaN,NaN,114.555556,74.855556,51.933333,2009-11-13 21:20:00,2023-12-27 00:00:00,NaN
min,NaN,NaN,NaN,74.000000,19.600000,0.700000,2005-01-11 00:00:00,2023-12-26 00:00:00,NaN
25%,NaN,NaN,NaN,108.000000,52.200000,5.300000,2005-01-26 00:00:00,2023-12-26 00:00:00,NaN
50%,NaN,NaN,NaN,115.000000,93.100000,74.500000,2011-03-30 00:00:00,2023-12-27 00:00:00,NaN
75%,NaN,NaN,NaN,132.000000,99.100000,80.600000,2011-12-20 00:00:00,2023-12-28 00:00:00,NaN
max,NaN,NaN,NaN,145.000000,99.200000,95.900000,2017-05-31 00:00:00,2023-12-28 00:00:00,NaN


### Resultados e possíveis usos

A camada contém **11 registros e 9 colunas**, em **EPSG:4326**, com **0 linhas inteiramente duplicadas**, **0 geometrias nulas** e **0 vazias**. Há **11 geometrias distintas**.

**Limitação temporal:** os nove registros com datas preenchidas têm `UltAnalise` entre **26 e 28/12/2023**, antes da cheia de maio de 2024. Portanto, esses resultados não descrevem a qualidade da água durante a enchente. Os sete campos de análises e datas apresentam duas ausências cada; a identificação e a geometria estão preenchidas nos 11 registros.

Na inspeção dos valores de `Coliformes`, há oito registros com “Presença”, um com “Ausência” e dois sem informação; em `E. Coli`, cinco com “Presença”, quatro com “Ausência” e dois sem informação. A relação dessas categorias com os percentuais e com as datas ainda precisa ser esclarecida, sem presumir que todos os campos representem a mesma coleta.


## 24. Exploração dos sistemas de abastecimento de água

### Camada `poa_sb_agua_saa`


In [29]:
exploracao_inicial(camadas['name'][18])  # Sistemas de abastecimento de água


Exploração inicial da camada: poa_sb_agua_saa



,OBJECTID,Name,NIVEL,EBAT,RESALIM,RESDISTR,NOMESISTEM,DATAATUALI,USUARIOATU,DATACRIACA,USUARIOCRI,OBSERVACOE,NOMESUBSIS,SHAPE_AREA,SHAPE_LEN,geometry
0,126,4.2.14,1,411,0,408,BELÉM NOVO,NaT,NaN,NaT,NaN,None,EBAT BOA VISTA e RES BOA VISTA (VRP29),0.0,0.0,"MULTIPOLYGON (((-51.17323 -30.21446, -51.17169..."
1,85,3.6.2,2,335,0,328,MENINO DEUS,2017-10-02,cecilia.mossner,NaT,NaN,None,EBAT CRISTIANO FISCHER e RES SAO JOSE II,0.0,0.0,"MULTIPOLYGON (((-51.25991 -30.12254, -51.26006..."
2,53,6.1.1,2,602,0,0,ILHA DA PINTADA,NaT,NaN,NaT,NaN,None,EBAT ILHAS (INLINE),0.0,0.0,"MULTIPOLYGON (((-51.24991 -30.03189, -51.24963..."
3,26,1.1.1,2,103,0,102,MOINHOS DE VENTO,2019-08-27,andre.goncalves,NaT,NaN,None,EBAT BORDINI e RES BELA VISTA,0.0,0.0,"MULTIPOLYGON (((-51.18956 -30.02252, -51.189 -..."
4,1,2.4.1.4,2,212,0,213,SÃO JOÃO,NaT,NaN,NaT,NaN,None,EBAT MANOEL ELIAS I e RES MANOEL ELIAS II (VRP53),0.0,0.0,"MULTIPOLYGON (((-51.13953 -30.0574, -51.13981 ..."
5,2561,7.3,1,0,0,704,TRISTEZA,2021-05-21,thiago.thomaz,2019-06-11,andre.goncalves,None,RESERVATORIO BALNEARIOS ESPIRITO SANTO,0.0,0.0,"MULTIPOLYGON (((-51.21656 -30.18526, -51.21657..."



Tipos de dados:


OBJECTID               int64
Name                     str
NIVEL                    str
EBAT                     str
RESALIM                  str
RESDISTR                 str
NOMESISTEM               str
DATAATUALI    datetime64[ms]
USUARIOATU               str
DATACRIACA    datetime64[ms]
USUARIOCRI               str
OBSERVACOE            object
NOMESUBSIS               str
SHAPE_AREA           float64
SHAPE_LEN            float64
geometry            geometry
dtype: object


Valores ausentes:


OBJECTID      0
Name          0
NIVEL         0
EBAT          0
RESALIM       0
RESDISTR      0
NOMESISTEM    0
DATAATUALI    3
USUARIOATU    3
DATACRIACA    5
USUARIOCRI    5
OBSERVACOE    6
NOMESUBSIS    0
SHAPE_AREA    0
SHAPE_LEN     0
geometry      0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,OBJECTID,Name,NIVEL,EBAT,RESALIM,RESDISTR,NOMESISTEM,DATAATUALI,USUARIOATU,DATACRIACA,USUARIOCRI,OBSERVACOE,NOMESUBSIS,SHAPE_AREA,SHAPE_LEN,geometry
count,6.000000,6,6,6,6,6,6,3,3,1,1,0,6,6.0,6.0,6
unique,NaN,6,2,6,1,6,6,NaN,3,NaN,1,0,6,NaN,NaN,6
top,NaN,4.2.14,2,411,0,408,BELÉM NOVO,NaN,cecilia.mossner,NaN,andre.goncalves,NaN,EBAT BOA VISTA e RES BOA VISTA (VRP29),NaN,NaN,MULTIPOLYGON (((-51.17322579190159 -30.2144567...
freq,NaN,1,4,1,6,1,1,NaN,1,NaN,1,NaN,1,NaN,NaN,1
mean,475.333333,NaN,NaN,NaN,NaN,NaN,NaN,2019-08-06 16:00:00,NaN,2019-06-11 00:00:00,NaN,NaN,NaN,0.0,0.0,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,2017-10-02 00:00:00,NaN,2019-06-11 00:00:00,NaN,NaN,NaN,0.0,0.0,NaN
25%,32.750000,NaN,NaN,NaN,NaN,NaN,NaN,2018-09-14 00:00:00,NaN,2019-06-11 00:00:00,NaN,NaN,NaN,0.0,0.0,NaN
50%,69.000000,NaN,NaN,NaN,NaN,NaN,NaN,2019-08-27 00:00:00,NaN,2019-06-11 00:00:00,NaN,NaN,NaN,0.0,0.0,NaN
75%,115.750000,NaN,NaN,NaN,NaN,NaN,NaN,2020-07-08 12:00:00,NaN,2019-06-11 00:00:00,NaN,NaN,NaN,0.0,0.0,NaN
max,2561.000000,NaN,NaN,NaN,NaN,NaN,NaN,2021-05-21 00:00:00,NaN,2019-06-11 00:00:00,NaN,NaN,NaN,0.0,0.0,NaN


### Resultados, limitações e possíveis usos

A camada contém **seis registros e 16 colunas**, com geometrias `MultiPolygon` em **EPSG:4326**. Não há geometrias nulas ou vazias nem linhas inteiramente duplicadas. Cada registro apresenta um valor distinto de `NOMESISTEM`: Belém Novo, Menino Deus, Ilha da Pintada, Moinhos de Vento, São João e Tristeza.

Os atributos incluem códigos de infraestrutura, nomes de sistemas e subsistemas e metadados de cadastro. A definição oficial dos polígonos precisa ser confirmada: os seis nomes não demonstram que as geometrias cubram integralmente os respectivos sistemas ou o município.

**Limitações principais:**

- `SHAPE_AREA` e `SHAPE_LEN` valem zero em todos os registros e não fornecem medidas utilizáveis de área ou comprimento nesta exploração. Eventuais medidas poderiam ser calculadas a partir das geometrias, após validação e reprojeção adequada.
- `DATAATUALI` está ausente em três registros; as datas preenchidas são de 2017, 2019 e 2021. Elas não comprovam atualização dos polígonos para a cheia de 2024.
- `DATACRIACA` e `USUARIOCRI` têm cinco ausências cada, `USUARIOATU` tem três e `OBSERVACOE` está completamente ausente. Códigos iguais a “0”, como os de `RESALIM`, também precisam de interpretação documental.

Como possíveis caminhos, os polígonos poderiam apoiar mapas e cruzamentos com bairros, escolas, manchas de inundação ou outras infraestruturas de água. Esses usos dependeriam da definição das áreas representadas e de sua cobertura e validade. Uma interseção espacial não comprova interrupção do abastecimento, conexão operacional ou população afetada.


## 25. Exploração das linhas de proteção e drenagem

### Camada `poa_sb_drenagem_protecao_linhas`


In [30]:
exploracao_inicial(camadas['name'][19])  # Linhas de proteção e drenagem


Exploração inicial da camada: poa_sb_drenagem_protecao_linhas



,COORD_Y_MO,Name,geometry
0,NaN,Dique,"MULTILINESTRING Z ((-51.1079 -29.97807 0, -51...."
1,NaN,Dique,"MULTILINESTRING Z ((-51.11036 -29.96876 0, -51..."
2,NaN,Dique,"MULTILINESTRING Z ((-51.1079 -29.97807 0, -51...."
3,NaN,Dique,"MULTILINESTRING Z ((-51.11493 -29.97756 0, -51..."
4,NaN,Dique,"MULTILINESTRING Z ((-51.13798 -29.9778 0, -51...."
5,NaN,Dique,"MULTILINESTRING Z ((-51.13798 -29.9778 0, -51...."
6,NaN,Dique,"MULTILINESTRING Z ((-51.13657 -29.99248 0, -51..."
7,NaN,Dique,"MULTILINESTRING Z ((-51.15454 -29.99666 0, -51..."
8,NaN,Dique,"MULTILINESTRING Z ((-51.09983 -29.96514 0, -51..."
9,NaN,Dique,"MULTILINESTRING Z ((-51.21327 -30.01356 0, -51..."



Tipos de dados:


COORD_Y_MO     float64
Name               str
geometry      geometry
dtype: object


Valores ausentes:


COORD_Y_MO    16
Name           0
geometry       0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,COORD_Y_MO,Name,geometry
count,23.0,39,39
unique,NaN,2,39
top,NaN,Muro,MULTILINESTRING Z ((-51.10790448254572 -29.978...
freq,NaN,22,1
mean,0.0,NaN,NaN
std,0.0,NaN,NaN
min,0.0,NaN,NaN
25%,0.0,NaN,NaN
50%,0.0,NaN,NaN
75%,0.0,NaN,NaN


### Resultados e possíveis usos

A camada contém **39 registros e três colunas** (`COORD_Y_MO`, `Name` e `geometry`), em **EPSG:4326**. Foram encontradas 39 geometrias distintas, sem geometrias nulas ou vazias e sem linhas inteiramente duplicadas. A leitura apresenta geometrias `MultiLineString Z`, embora o inventário inicial informe `LineString Z`.

O campo `Name` classifica **22 registros como “Muro” e 17 como “Dique”**. Esses números representam registros ou trechos, não necessariamente estruturas físicas independentes. `COORD_Y_MO` contém **16 ausências e 23 valores iguais a zero**; sua definição precisa ser confirmada e não permite inferir coordenadas ou altura útil nesta etapa.

A presença de um muro ou dique no cadastro não comprova sua integridade, funcionamento ou eficácia durante a enchente. Eventuais cálculos de comprimento exigiriam reprojeção para um CRS adequado em metros; a dimensão Z não confirma altitude real.


## 26. Exploração dos pontos de proteção e drenagem

### Camada `poa_sb_drenagem_protecao_pontos`


In [31]:
exploracao_inicial(camadas['name'][20])  # Pontos de proteção e drenagem


Exploração inicial da camada: poa_sb_drenagem_protecao_pontos



,COORD_X,COORD_Y,Name,description,timestamp,begin,end,Status_07_05,Status_06_05,Status_08_05,Status_09_05,Status_10_05,geometry
0,279866.639,1674938.959,CB SANTA TEREZINHA,Casa de bombas,NaT,NaT,NaT,,,,,,POINT (-51.20876 -30.04462)
1,286628.242,1683436.207,CB 09,Casa de bombas Falha,NaT,NaT,NaT,Falha,,Falha,Falha,Falha,POINT (-51.13854 -29.96806)
2,278457.654,1677168.676,CB 18,Casa de bombas Falha,NaT,NaT,NaT,Falha,,Falha,Falha,Falha,POINT (-51.22332 -30.02448)
3,0.000,0.000,CB TRINCHEIRA CEARÁ,Casa de bombas Falha,NaT,NaT,NaT,Falha,Falha,Falha,Falha,Falha,POINT (-51.18953 -29.99623)
4,278837.990,1674579.732,CB 15,Casa de bombas,NaT,NaT,NaT,Falha,,Falha,Falha,,POINT (-51.21943 -30.04784)
5,277592.302,1675101.616,CB 16,Casa de bombas Falha,NaT,NaT,NaT,Falha,Falha,Falha,Falha,Falha,POINT (-51.23234 -30.04311)
6,288181.955,1682322.558,CB ASA BRANCA,Casa de bombas Falha,NaT,NaT,NaT,Falha,,Falha,Falha,Falha,POINT (-51.12246 -29.97812)
7,277632.254,1673827.238,CB 13,Casa de bombas Falha,NaT,NaT,NaT,Falha,Falha,Falha,Falha,Falha,POINT (-51.23195 -30.05461)
8,280940.701,1681812.215,CB VILA FARRAPOS,Casa de bombas Falha,NaT,NaT,NaT,Falha,,Falha,Falha,Falha,POINT (-51.1975 -29.98263)
9,286525.067,1680928.213,CB VILA MINUANO,Casa de bombas Falha,NaT,NaT,NaT,Falha,,Falha,Falha,Falha,POINT (-51.13964 -29.99068)



Tipos de dados:


COORD_X                float64
COORD_Y                float64
Name                       str
description                str
timestamp       datetime64[ms]
begin           datetime64[ms]
end             datetime64[ms]
Status_07_05               str
Status_06_05               str
Status_08_05               str
Status_09_05               str
Status_10_05               str
geometry              geometry
dtype: object


Valores ausentes:


COORD_X         14
COORD_Y         14
Name             0
description      0
timestamp       37
begin           37
end             37
Status_07_05     0
Status_06_05     0
Status_08_05     0
Status_09_05     0
Status_10_05     0
geometry         0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,COORD_X,COORD_Y,Name,description,timestamp,begin,end,Status_07_05,Status_06_05,Status_08_05,Status_09_05,Status_10_05,geometry
count,23.000000,2.300000e+01,37,37,0,0,0,37,37,37,37,37,37
unique,NaN,NaN,37,5,NaN,NaN,NaN,2,2,2,2,2,37
top,NaN,NaN,CB SANTA TEREZINHA,Casa de bombas Falha,NaN,NaN,NaN,Falha,,Falha,Falha,Falha,POINT (-51.20875978731046 -30.044616859350754)
freq,NaN,NaN,1,16,NaN,NaN,NaN,24,26,21,20,19,1
mean,256353.204174,1.532000e+06,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,0.000000e+00,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN
25%,277612.278000,1.674561e+06,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN
50%,279342.669000,1.677745e+06,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN
75%,281991.916500,1.681370e+06,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN
max,288181.955000,1.683436e+06,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN


### Resultados, limitações e possíveis usos

A camada contém **37 registros e 13 colunas**, com geometrias `Point` em **EPSG:4326**. São 37 nomes e geometrias distintos, sem geometrias nulas ou vazias e sem linhas inteiramente duplicadas. Os rótulos de `description` indicam **23 registros de casas de bombas e 14 de comportas**, incluindo descrições com “Falha” e variações de espaços no final do texto.

Os campos diários contêm apenas “Falha” ou texto vazio:

| Campo | Falha | Texto vazio |
| --- | ---: | ---: |
| `Status_06_05` | 11 | 26 |
| `Status_07_05` | 24 | 13 |
| `Status_08_05` | 21 | 16 |
| `Status_09_05` | 20 | 17 |
| `Status_10_05` | 19 | 18 |

**Texto vazio não comprova funcionamento normal**, nem sua mudança para vazio comprova recuperação operacional. Esses valores não aparecem como ausências em `isna()`. O significado de “Falha”, o ano e o horário de referência precisam ser confirmados; os rótulos de `description` não devem ser considerados situações válidas para todas as datas.

`timestamp`, `begin` e `end` estão completamente ausentes. `COORD_X` e `COORD_Y` têm 14 ausências cada e dois pares iguais a zero; os valores preenchidos não correspondem diretamente a longitude e latitude em graus. A referência dessas coordenadas precisaria ser esclarecida antes de usá-las, sem substituir automaticamente a geometria disponível.

Como possíveis caminhos, os pontos poderiam apoiar mapas e cruzamentos com bairros, manchas de inundação e linhas de proteção. Os campos de situação poderiam apoiar uma análise temporal limitada aos registros informados, após confirmação documental. A presença de uma estrutura não comprova eficácia de proteção, e uma interseção com a inundação não demonstra a causa da falha.

A validade e a precisão das localizações ainda não foram verificadas. A decisão de aproveitamento permanece aberta, e nenhum dado original foi alterado.


## 27. Exploração da delimitação da bacia

### Camada `rhguaiba_bacia`


In [32]:
exploracao_inicial(camadas['name'][21])  # Delimitação da bacia


Exploração inicial da camada: rhguaiba_bacia



,nome,codigootto,nivelotto,area,d_geometri,geometry
0,Lago Guaíba,0,0,2429,None,"MULTIPOLYGON Z (((-51.09959 -30.37431 0, -51.0..."



Tipos de dados:


nome               str
codigootto       int64
nivelotto        int64
area             int64
d_geometri      object
geometry      geometry
dtype: object


Valores ausentes:


nome          0
codigootto    0
nivelotto     0
area          0
d_geometri    1
geometry      0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,nome,codigootto,nivelotto,area,d_geometri,geometry
count,1,1.0,1.0,1.0,0,1
unique,1,NaN,NaN,NaN,0,1
top,Lago Guaíba,NaN,NaN,NaN,NaN,MULTIPOLYGON Z (((-51.09959166099998 -30.37431...
freq,1,NaN,NaN,NaN,NaN,1
mean,NaN,0.0,0.0,2429.0,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,0.0,0.0,2429.0,NaN,NaN
25%,NaN,0.0,0.0,2429.0,NaN,NaN
50%,NaN,0.0,0.0,2429.0,NaN,NaN
75%,NaN,0.0,0.0,2429.0,NaN,NaN


### Resultados e possíveis usos

A camada contém **1 registro e 6 colunas**, com geometria `MultiPolygon` em **EPSG:4326**. Foram identificadas **1 ausências**, **0 linhas inteiramente duplicadas** e **0 geometrias vazias**.

Um possível caminho é usar a delimitação como referência cartográfica ou avaliar recortes espaciais de outras camadas, após confirmar a abrangência e validar a geometria. **O limite de uma bacia não equivale a uma mancha de inundação ou a uma classificação de risco.**

Eventuais cálculos de área exigiriam reprojeção para um CRS adequado em metros e poderiam ser comparados ao atributo `area` após esclarecer sua unidade. A dimensão Z não comprova, por si só, altitude real ou profundidade de inundação. A exploração inicial não verifica validade, cobertura ou precisão geométrica.


## 28. Exploração da água observada por Sentinel-2 — 06/05/2024

### Camada `rhguaiba_sentinel2_agua_obs_06052024`


In [33]:
exploracao_inicial(camadas['name'][22])  # Água observada por Sentinel-2 — 06/05/2024


Exploração inicial da camada: rhguaiba_sentinel2_agua_obs_06052024

Exibindo as primeiras 100 de 4780 linhas. As verificações usam a camada completa.


,DN,layer,path,bacia,area_km2,geometry
0,1,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,0,"POLYGON ((-53.95539 -30.12527, -53.95524 -30.1..."
1,1,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,0,"POLYGON ((-53.95596 -30.12491, -53.95596 -30.1..."
2,1,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,0,"POLYGON ((-53.95575 -30.1246, -53.95578 -30.12..."
3,1,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,0,"POLYGON ((-53.95938 -30.12444, -53.95926 -30.1..."
4,1,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,0,"POLYGON ((-53.95578 -30.12437, -53.95578 -30.1..."
5,1,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,0,"POLYGON ((-53.9565 -30.12437, -53.9565 -30.124..."
6,1,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,0,"POLYGON ((-53.95794 -30.12393, -53.95794 -30.1..."
7,1,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,0,"POLYGON ((-53.95596 -30.12419, -53.95596 -30.1..."
8,1,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,0,"POLYGON ((-53.95686 -30.12419, -53.95686 -30.1..."
9,1,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,0,"POLYGON ((-53.95704 -30.12401, -53.95704 -30.1..."



Tipos de dados:


DN             int32
layer            str
path             str
bacia            str
area_km2       int32
geometry    geometry
dtype: object


Valores ausentes:


DN          0
layer       0
path        0
bacia       3
area_km2    0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,DN,layer,path,bacia,area_km2,geometry
count,4780.0,4780,4780,4777,4780.000000,4780
unique,NaN,1,1,7,NaN,4780
top,NaN,cheias_poa_bd — sentinel_clean_06-05-2024,C:\gis\_projects_\flood_poa\cheias_poa_bd.gpkg...,Vacacai,NaN,POLYGON ((-53.95538529287936 -30.1252702396499...
freq,NaN,4780,4780,1513,NaN,1
mean,1.0,NaN,NaN,NaN,0.671339,NaN
std,0.0,NaN,NaN,NaN,7.018474,NaN
min,1.0,NaN,NaN,NaN,0.000000,NaN
25%,1.0,NaN,NaN,NaN,0.000000,NaN
50%,1.0,NaN,NaN,NaN,0.000000,NaN
75%,1.0,NaN,NaN,NaN,0.000000,NaN


### Resultados e possíveis usos

A camada contém **4780 registros e 6 colunas**, com geometrias `Polygon` em **EPSG:4326**. Foram identificados **0 registros inteiramente duplicados**, **0 geometrias nulas** e **0 vazias**.

`DN` vale 1 em todos os registros. `bacia` apresenta sete categorias preenchidas e três ausências, indicando recortes regionais que não se limitam aos bairros de Porto Alegre. `layer` e `path` são constantes; o caminho aponta para o ambiente de origem, não para um arquivo disponível neste projeto.

`area_km2` está armazenado como inteiro e contém zeros. O nome sugere quilômetros quadrados, mas a unidade, o método e a precisão precisam ser confirmados; zeros não comprovam ausência de área. Uma possibilidade é conferir esse atributo com áreas calculadas a partir das geometrias após validação e reprojeção.

O nome sugere uma camada de água observada em imagens Sentinel-2 de **06/05/2024**. A metodologia, a data de aquisição, as classes e a precisão precisam ser confirmadas na documentação. **Água identificada por satélite pode incluir corpos de água permanentes, não apenas áreas excepcionalmente inundadas.**

A interpretação de `DN` e de eventuais atributos de área depende de documentação. Cálculos de área exigiriam um CRS projetado adequado. A resolução da imagem, a data e possíveis erros de classificação limitam a precisão de eventuais conclusões sobre estabelecimentos individuais.


## 29. Exploração da inundação simulada HEC-RAS — 550 cm

### Camada `rhguaiba_hecras_inundacao_simulada_550cm`


**Esclarecimento do glossário:** O glossário especifica uma mancha simulada pelo modelo HEC-RAS para o nível de 550 cm no Cais Mauá. Isso esclarece a referência nominal do nível, sem demonstrar equivalência com as outras simulações ou com o JSON do INCT.


In [34]:
exploracao_inicial(camadas['name'][23])  # Inundação simulada HEC-RAS — 550 cm


Exploração inicial da camada: rhguaiba_hecras_inundacao_simulada_550cm



,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,descri____o,fill_opacity,stroke_opacity,stroke,Count,Length,Area,Range_Min,Range_Max,FID_2,Count_2,Length_2,Area_2,Range_Min_2,Range_Max_2,Name_2,geometry
0,"{0.5, Max}",descrição: <br>fill-opacity: 0<br>stroke-opaci...,NaT,NaT,NaT,None,-1,0,-1,NaN,None,,0,1,#ff0000,250652,7241787.32932,19761906266.9,0.5,Max,0,1516708,14482752.4946,20880409166.3,0.2,Max,"{0.2, Max}","MULTIPOLYGON Z (((-51.05384 -30.35641 0, -51.0..."



Tipos de dados:


Name                         str
description                  str
timestamp         datetime64[ms]
begin             datetime64[ms]
end               datetime64[ms]
altitudeMode              object
tessellate                 int32
extrude                    int32
visibility                 int32
drawOrder                float64
icon                      object
descri____o                  str
fill_opacity                 str
stroke_opacity               str
stroke                       str
Count                        str
Length                       str
Area                         str
Range_Min                    str
Range_Max                    str
FID_2                        str
Count_2                      str
Length_2                     str
Area_2                       str
Range_Min_2                  str
Range_Max_2                  str
Name_2                       str
geometry                geometry
dtype: object


Valores ausentes:


Name              0
description       0
timestamp         1
begin             1
end               1
altitudeMode      1
tessellate        0
extrude           0
visibility        0
drawOrder         1
icon              1
descri____o       0
fill_opacity      0
stroke_opacity    0
stroke            0
Count             0
Length            0
Area              0
Range_Min         0
Range_Max         0
FID_2             0
Count_2           0
Length_2          0
Area_2            0
Range_Min_2       0
Range_Max_2       0
Name_2            0
geometry          0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,...,Range_Min,Range_Max,FID_2,Count_2,Length_2,Area_2,Range_Min_2,Range_Max_2,Name_2,geometry
count,1,1,0,0,0,0,1.0,1.0,1.0,0.0,...,1,1,1,1,1,1,1,1,1,1
unique,1,1,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,1,1,1
top,"{0.5, Max}",descrição: <br>fill-opacity: 0<br>stroke-opaci...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.5,Max,0,1516708,14482752.4946,20880409166.3,0.2,Max,"{0.2, Max}","MULTIPOLYGON Z (((-51.053842 -30.356409 0, -51..."
freq,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,1,1,1
mean,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Resultados e possíveis usos

A camada contém **1 registro e 28 colunas**, com geometria `MultiPolygon` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias** e **0 linhas inteiramente duplicadas**.

**Limitações dos atributos:** `timestamp`, `begin`, `end`, `altitudeMode`, `drawOrder` e `icon` estão ausentes; `descri____o` contém texto vazio. Há campos de estilo e valores aparentemente numéricos armazenados como texto, incluindo `Area`, `Length` e seus equivalentes com sufixo `_2`. Esses pares não devem ser somados ou tratados como medidas equivalentes sem esclarecer sua origem e unidade. `Range_Min = "0.5"` e `Range_Min_2 = "0.2"` sugerem limites de classificação, mas não comprovam profundidades em metros.

O nome indica uma simulação associada ao HEC-RAS e à cota de **550 cm (5,5 m)**. O modelo, a referência de nível, as condições de contorno e a abrangência precisam ser confirmados na documentação. Não se trata automaticamente de uma observação da área efetivamente inundada, e a cota não representa a profundidade em cada ponto.

Um possível caminho é comparar essa geometria com a camada `poa_inundacao_simulada_550cm` já explorada. A mesma cota nominal não garante equivalência entre os cenários. Também poderiam ser avaliados mapas e cruzamentos com bairros, escolas ou outras infraestruturas, após confirmar a cobertura, validar a geometria e compatibilizar os sistemas de referência.

Um único registro não significa necessariamente uma área simples ou contínua. Eventuais cálculos de área exigiriam um CRS projetado adequado; a dimensão Z não comprova, por si só, altitude real ou profundidade. A decisão de aproveitamento permanece aberta, e nenhum dado original foi alterado.


## 30. Exploração dos abrigos — 08/05/2024 às 12h

### Camada `rs_crise_abrigos_08052024_12h`


In [35]:
exploracao_inicial(camadas['name'][24])  # Abrigos — 08/05/2024 às 12h


Exploração inicial da camada: rs_crise_abrigos_08052024_12h

Exibindo as primeiras 100 de 210 linhas. As verificações usam a camada completa.


,Id,Name,Voluntarios,Doacoes,DATA ATUALIZ.,Endereco,CIDADE,LINK MAPS,TIPO DE LOCAL,ALAGADO,conferir,fonte,Latitude,Longitude,CEP,Latitude_old,Longitude_old,GRUPO WHATS,OBS,ATUALIZAÇÃO,PONTO,geometry
0,1,101 Vira Latas (ONG),s/ infos,Precisando,NaN,"R. Sete Irmãos, 190 - Cecília, Viamão",Viamão,#NAME?,Abrigo,NaN,NaN,NaN,-30.075517,-51.095357,94475-760,-30.075411,-51.096011,(51) 99431-9350,"PRECISANDO: Ração para cachorros e gatos, casi...",NaN,Abrigo,POINT (-51.09536 -30.07552)
1,2,21 BPM,p/ escala,Precisando,NaN,"R. Gov. Peracchi Barcellos, 4600 - Restinga, P...",Porto Alegre,NaN,Abrigo,NaN,NaN,NaN,-30.152539,-51.152107,91790-370,NaN,NaN,NaN,PRECISANDO: sacolas para montar kits\nDISPONÍVEL:,NaN,Abrigo,POINT (-51.15211 -30.15254)
2,3,AABB,Precisando,Precisando,NaN,"Av. Cel. Marcos, 1000 - Pedra Redonda, Porto A...",Porto Alegre,NaN,Abrigo,NaN,NaN,NaN,-30.124069,-51.245705,91760-000,-30.122746,-51.252629,NaN,"PRECISANDO: cobertores, corda, caminhas e pape...",06/05 - 09hs,Abrigo,POINT (-51.24571 -30.12407)
3,4,AFISVEC,Precisando,Precisando,NaN,"Rua Salvador Calamucci, 56 - Cavalhada Porto A...",Porto Alegre,NaN,Abrigo,NaN,NaN,NaN,-30.097028,-51.226995,NaN,-30.097030,-51.227109,NaN,"PRECISANDO: cordas, aguas, guias, mantas e rações",NaN,Abrigo,POINT (-51.227 -30.09703)
4,5,AMRIGS,s/ infos,Sob controle,NaN,"Av. Ipiranga, 5311 - Partenon, Porto Alegre",Porto Alegre,NaN,Ponto de recebimento de doações,NaN,NaN,NaN,-30.057263,-51.186334,NaN,-30.058231,-51.185890,NaN,PRECISANDO:\nDISPONÍVEL:,05/05 - 13h45min,Ponto de recebimento de doações,POINT (-51.18633 -30.05726)
5,6,AMVEP,Sob controle,Precisando,NaN,"Av. 21 de Abril, 792 - Sarandi, Porto Alegre",Porto Alegre,NaN,Abrigo,SIM,NaN,NaN,-29.985790,-51.129872,91120-630,-29.979628,-51.130051,NaN,"PRECISANDO:Material de limpeza, escovas de den...",NaN,Abrigo,POINT (-51.12987 -29.98579)
6,7,Casa de Abrigo de Animais,Ñ recebe,URGENTE,NaN,"R. Dr. Barbosa Gonçalves 61, Chácara das Pedra...",Porto Alegre,NaN,Abrigo,NaN,NaN,NaN,-30.035095,-51.161745,NaN,-30.029549,-51.161363,NaN,PRECISANDO: Cadeiras (tipo de praia); Voluntar...,06/05 - 11h,Abrigo,POINT (-51.16175 -30.03509)
7,8,CEEE Ipiranga,Ñ recebe,Sob controle,NaN,Av. Ipiranga 8300 - Porto Alegre,Porto Alegre,NaN,Abrigo,NaN,NaN,NaN,-30.056660,-51.156250,91530-000,-30.048052,-51.215576,NaN,PRECISANDO:\nDISPONÍVEL:,05/05 - 14h57min,Abrigo,POINT (-51.15625 -30.05666)
8,9,CEEE Vila Nova,Precisando c/ incrição,Precisando,NaN,"Av. Joaquim Porto Villanova, 201 - Jardim Carv...",Porto Alegre,NaN,Abrigo,NaN,NaN,NaN,-30.053059,-51.157682,91410-400,-30.055730,-51.159347,NaN,PRECISANDO:\nDISPONÍVEL:,NaN,Abrigo,POINT (-51.15768 -30.05306)
9,10,Centro VIDA,Precisando c/ incrição,Precisando,NaN,"Av. Baltazar de Oliveira Garcia, 2132",Porto Alegre,NaN,Abrigo,NaN,NaN,NaN,-30.010625,-51.122762,NaN,-30.010912,-51.122880,https://bit.ly/3UstHLx,"PRECISANDO: Medicamentos para cães, gatos e pe...",05/05 - 15h46min,Abrigo,POINT (-51.12276 -30.01062)



Tipos de dados:


Id                  int32
Name                  str
Voluntarios           str
Doacoes               str
DATA ATUALIZ.         str
Endereco              str
CIDADE                str
LINK MAPS             str
TIPO DE LOCAL         str
ALAGADO               str
conferir              str
fonte                 str
Latitude          float64
Longitude         float64
CEP                   str
Latitude_old      float64
Longitude_old     float64
GRUPO WHATS           str
OBS                   str
ATUALIZAÇÃO           str
PONTO                 str
geometry         geometry
dtype: object


Valores ausentes:


Id                 0
Name               0
Voluntarios        0
Doacoes            0
DATA ATUALIZ.    209
Endereco           1
CIDADE             1
LINK MAPS        209
TIPO DE LOCAL     22
ALAGADO          200
conferir          87
fonte            111
Latitude           2
Longitude          2
CEP              194
Latitude_old     147
Longitude_old    147
GRUPO WHATS      144
OBS              144
ATUALIZAÇÃO      197
PONTO            121
geometry           0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Id,Name,Voluntarios,Doacoes,DATA ATUALIZ.,Endereco,CIDADE,LINK MAPS,TIPO DE LOCAL,ALAGADO,...,Latitude,Longitude,CEP,Latitude_old,Longitude_old,GRUPO WHATS,OBS,ATUALIZAÇÃO,PONTO,geometry
count,210.000000,210,210,210,1,209,209,1,188,10,...,208.000000,208.000000,16,63.000000,63.000000,66,66,13,89,210
unique,NaN,201,7,5,1,200,15,1,3,2,...,NaN,NaN,16,NaN,NaN,64,58,8,2,185
top,NaN,Colégio Kennedy,s/ infos,s/ infos,05/05,"R. Hélio Pimpão, 52 - Morro Santana, Porto Alegre",Porto Alegre,#NAME?,Abrigo,SIM,...,NaN,NaN,94475-760,NaN,NaN,https://bit.ly/3UstHLx,PRECISANDO:\nDISPONÍVEL:,05/05 - 14h57min,Abrigo,POINT (-51.1587408 -30.0199672)
freq,NaN,2,165,129,1,2,137,1,159,9,...,NaN,NaN,1,NaN,NaN,2,7,3,80,4
mean,105.766667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-30.004310,-51.169982,NaN,-30.116895,-51.297370,NaN,NaN,NaN,NaN,NaN
std,61.106134,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.108994,0.111583,NaN,0.594400,0.960161,NaN,NaN,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-30.279549,-52.430847,NaN,-34.741496,-58.791108,NaN,NaN,NaN,NaN,NaN
25%,53.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-30.059563,-51.197369,NaN,-30.065439,-51.209531,NaN,NaN,NaN,NaN,NaN
50%,105.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-30.027463,-51.163357,NaN,-30.041051,-51.174275,NaN,NaN,NaN,NaN,NaN
75%,158.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-29.929690,-51.134683,NaN,-30.011745,-51.152231,NaN,NaN,NaN,NaN,NaN


### Resultados e possíveis usos

A camada contém **210 registros e 22 colunas**, com geometrias `Point` em **EPSG:4326**. Há **0 geometrias nulas**, **2 vazias**, **185 geometrias distintas** e **0 linhas inteiramente duplicadas**.

**Limitações de conteúdo:** a camada inclui abrigos e pontos de recebimento de doações, com variações de espaços em `TIPO DE LOCAL`; portanto, os 210 registros não equivalem a 210 abrigos distintos. Há 201 nomes distintos, 15 valores de cidade preenchidos e um ausente; 137 registros informam “Porto Alegre”. A equivalência de nomes, municípios e estabelecimentos ainda precisa ser avaliada.

`ALAGADO` tem 200 ausências, nove valores “SIM” e um “NÃO”; ausência não significa local não alagado. `Voluntarios` e `Doacoes` incluem respectivamente 165 e 129 valores “s/ infos”, não detectados como ausências por `isna()`. O campo `conferir` contém marcações como “duplicado” e “checar endereço”, que sugerem revisão pendente.

`DATA ATUALIZ.` tem 209 ausências e `ATUALIZAÇÃO`, 197; o único `LINK MAPS` preenchido é “#NAME?”, indicando um valor a investigar. As datas disponíveis incluem referências anteriores ao nome da camada, portanto não se deve presumir atualização de cada registro em 08/05 às 12h.


## 31. Exploração das fontes públicas de água — RMPA

### Camada `rmpa_sb_agua_fontes_publicas`


In [36]:
exploracao_inicial(camadas['name'][25])  # Fontes públicas de água — RMPA


Exploração inicial da camada: rmpa_sb_agua_fontes_publicas



,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,geometry
0,Trianon - Esteio,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (-51.11218 -29.86386 0)
1,Rua Garibaldi - Esteio,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (-51.17503 -29.84909 0)
2,Rua Caturritas - Alvorada,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (-51.08411 -30.01796 0)
3,Paróquia São Vicente de Paulo - Cachoeirinha,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (-51.09436 -29.95275 0)
4,CTG Guapos da Amizade - Cachoeirinha,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (-51.08696 -29.93717 0)
5,Ao lado do BAITA - Viamão,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (-51.07985 -29.94705 0)
6,Praça da Estalagem - Viamão,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (-51.03153 -30.0785 0)
7,Planalto - Viamão,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (-51.01393 -30.08503 0)
8,Augusta - Viamão,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (-51.07982 -30.10819 0)
9,Praça Santa Isabel - Viamão,None,NaT,NaT,NaT,None,-1,0,-1,NaN,None,POINT Z (-51.08815 -30.06445 0)



Tipos de dados:


Name                       str
description             object
timestamp       datetime64[ms]
begin           datetime64[ms]
end             datetime64[ms]
altitudeMode            object
tessellate               int32
extrude                  int32
visibility               int32
drawOrder              float64
icon                    object
geometry              geometry
dtype: object


Valores ausentes:


Name             0
description     24
timestamp       24
begin           24
end             24
altitudeMode    24
tessellate       0
extrude          0
visibility       0
drawOrder       24
icon            24
geometry         0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,geometry
count,24,0,0,0,0,0,24.0,24.0,24.0,0.0,0,24
unique,24,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,24
top,Trianon - Esteio,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT Z (-51.11218025495796 -29.86386302483499 0)
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
mean,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN
min,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN
25%,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN
50%,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN
75%,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN
max,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN


### Resultados e possíveis usos

A camada contém **24 registros e 12 colunas**, com geometrias `Point` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **24 geometrias distintas** e **0 linhas inteiramente duplicadas**.

**Conteúdo e limitações:** `Name` contém 24 nomes distintos, em vários casos com o município incorporado ao texto, sem uma coluna específica de município. Sete campos estão completamente ausentes: `description`, `timestamp`, `begin`, `end`, `altitudeMode`, `drawOrder` e `icon`. `tessellate`, `extrude` e `visibility` são constantes e parecem relacionados à representação/exportação, sem informação operacional confirmada.

Assim, os principais dados candidatos a uso são o nome e a localização; não há indicadores de qualidade da água ou situação de funcionamento nesta camada. A dimensão Z não comprova altitude real.

O nome sugere fontes públicas de água na Região Metropolitana de Porto Alegre. A abrangência, a data de referência e o significado dos atributos precisam ser confirmados na documentação. O cadastro, isoladamente, não comprova potabilidade, disponibilidade ou funcionamento durante a enchente.


## 32. Exploração das áreas propensas a movimentos de massa — Taquari

### Camada `taquari_areas_propensas_movimentos_massa`


In [37]:
exploracao_inicial(camadas['name'][26])  # Áreas propensas a movimentos de massa — Taquari


Exploração inicial da camada: taquari_areas_propensas_movimentos_massa



,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,descri____o,DN,area,layer,path,geometry
0,,descrição: <br>DN: 1<br>area: 37500,NaT,NaT,NaT,None,-1,0,-1,NaN,None,,1,37500,Áreas propensas a movimentos de massa — Parte ...,C:/Users/Ipo/Downloads/Áreas propensas a movim...,"MULTIPOLYGON Z (((-52.15352 -29.46707 0, -52.1..."
1,Área Suscetível,NaN,NaT,NaT,NaT,None,-1,0,-1,NaN,None,NaN,NaN,NaN,Áreas propensas a movimentos de massa — Parte ...,C:/Users/Ipo/Downloads/Áreas propensas a movim...,"MULTIPOLYGON Z (((-51.9277 -29.10255 0, -51.92..."
2,,descrição: <br>DN: 1<br>area: 4219,NaT,NaT,NaT,None,-1,0,-1,NaN,None,,1,4219,Áreas propensas a movimentos de massa — Parte ...,C:/Users/Ipo/Downloads/Áreas propensas a movim...,"MULTIPOLYGON Z (((-52.06062 -29.26463 0, -52.0..."
3,Área Suscetível,NaN,NaT,NaT,NaT,None,-1,0,-1,NaN,None,NaN,NaN,NaN,Áreas propensas a movimentos de massa — Parte ...,C:/Users/Ipo/Downloads/Áreas propensas a movim...,"MULTIPOLYGON Z (((-51.98002 -29.20649 0, -51.9..."
4,,descrição: <br>DN: 1<br>area: 1719,NaT,NaT,NaT,None,-1,0,-1,NaN,None,,1,1719,Áreas propensas a movimentos de massa — Parte ...,C:/Users/Ipo/Downloads/Áreas propensas a movim...,"MULTIPOLYGON Z (((-51.85054 -29.36044 0, -51.8..."
5,Área Suscetível,NaN,NaT,NaT,NaT,None,-1,0,-1,NaN,None,NaN,NaN,NaN,Áreas propensas a movimentos de massa — Parte ...,C:/Users/Ipo/Downloads/Áreas propensas a movim...,"MULTIPOLYGON Z (((-52.15341 -29.4663 0, -52.15..."
6,,descrição: <br>DN: 1<br>area: 4375,NaT,NaT,NaT,None,-1,0,-1,NaN,None,,1,4375,Áreas propensas a movimentos de massa — Parte ...,C:/Users/Ipo/Downloads/Áreas propensas a movim...,"MULTIPOLYGON Z (((-51.97322 -29.13858 0, -51.9..."



Tipos de dados:


Name                       str
description                str
timestamp       datetime64[ms]
begin           datetime64[ms]
end             datetime64[ms]
altitudeMode            object
tessellate               int32
extrude                  int32
visibility               int32
drawOrder              float64
icon                    object
descri____o                str
DN                         str
area                       str
layer                      str
path                       str
geometry              geometry
dtype: object


Valores ausentes:


Name            0
description     3
timestamp       7
begin           7
end             7
altitudeMode    7
tessellate      0
extrude         0
visibility      0
drawOrder       7
icon            7
descri____o     3
DN              3
area            3
layer           0
path            0
geometry        0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,descri____o,DN,area,layer,path,geometry
count,7,4,0,0,0,0,7.0,7.0,7.0,0.0,0,4,4,4,7,7,7
unique,2,4,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,1,1,4,7,7,7
top,,descrição: <br>DN: 1<br>area: 37500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,1,37500,Áreas propensas a movimentos de massa — Parte ...,C:/Users/Ipo/Downloads/Áreas propensas a movim...,"MULTIPOLYGON Z (((-52.153518 -29.467066 0, -52..."
freq,4,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,4,1,1,1,1
mean,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaT,NaT,NaT,NaN,-1.0,0.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Resultados e possíveis usos

A camada contém **7 registros e 17 colunas**, com geometrias `MultiPolygon` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **7 geometrias distintas** e **0 linhas inteiramente duplicadas**.

**Limitações dos atributos:** `layer` distingue quatro partes rotuladas como “Corridas” e três como “Deslizamentos”; os sete registros não representam necessariamente sete ocorrências ou áreas independentes. `Name` tem quatro textos vazios e três valores “Área Suscetível”. `DN` e `area` estão armazenados como texto e ausentes em três registros; a unidade de `area` não foi confirmada.

`timestamp`, `begin`, `end`, `altitudeMode`, `drawOrder` e `icon` estão completamente ausentes. `descri____o` contém apenas textos vazios ou ausências. Os caminhos de `path` apontam para arquivos do ambiente de origem, não para recursos disponíveis neste projeto.

O nome indica áreas propensas a movimentos de massa no recorte do Taquari. A metodologia, os critérios, a referência temporal e a cobertura precisam ser confirmados na documentação. **Propensão a movimentos de massa não equivale a área inundada nem comprova ocorrência de deslizamento.**

Um possível caminho é considerar a camada como contexto regional complementar, caso a análise inclua o Taquari e diferentes processos associados ao evento. Para análises centradas nos bairros de Porto Alegre, a pertinência espacial precisaria ser avaliada antes de qualquer integração. Eventuais cruzamentos dependeriam de validar as geometrias e compatibilizar os sistemas de referência.


## 33. Exploração dos abrigos — 09/05/2024 às 20h

### Camada `rs_crise_abrigos_09052024_20h`


In [38]:
exploracao_inicial(camadas['name'][27])  # Abrigos — 09/05/2024 às 20h


Exploração inicial da camada: rs_crise_abrigos_09052024_20h

Exibindo as primeiras 100 de 360 linhas. As verificações usam a camada completa.


,Nome,Fonte,Geo,End,Tipo,Lat,Long,name,geometry
0,21 BPM,Ajuda RS,SIM,"R. Gov. Peracchi Barcellos, 4600 - Restinga, P...",Abrigo,-30.152539,-51.152107,21 BPM,POINT (-51.15211 -30.15254)
1,35 CTG,Ajuda RS,SIM,"Av. Ipiranga, N° 5300, Porto Alegre",Abrigo,-30.054988,-51.186973,35 CTG,POINT (-51.18697 -30.05499)
2,A Igreja de Jesus Cristo dos Santos dos Último...,FSPPA,SIM,NaN,Centro de doação,-30.048684,-51.209819,A Igreja de Jesus Cristo dos Santos dos Último...,POINT (-51.20982 -30.04868)
3,AABB Porto Alegre,Ajuda RS,SIM,"Av. Cel. Marcos, 1000 - Pedra Redonda, Porto A...",Abrigo,-30.123390,-51.245654,AABB Porto Alegre,POINT (-51.24565 -30.12339)
4,ABAMF (Ginásio atrás da Brigada Militar na Ron...,FSPPA,SIM,NaN,Abrigo,-29.704012,-51.120434,ABAMF (Ginásio atrás da Brigada Militar na Ron...,POINT (-51.12043 -29.70401)
5,ABCREI,Ajuda RS,SIM,RUA BECO DOS SOARES 765,sem info,-30.086662,-51.074403,ABCREI,POINT (-51.0744 -30.08666)
6,Abrigo da Bento (Atacadao Miranda Thales Robinho),Ajuda RS,SIM,avenida bento Gonçalves 6623,sem info,-30.066993,-51.150273,Abrigo da Bento (Atacadao Miranda Thales Robinho),POINT (-51.15027 -30.06699)
7,Abrigo de animais Chácara das Pedras,Ajuda RS,SIM,rua dr Barbosa Gonçalves 61,sem info,-30.035095,-51.161745,Abrigo de animais Chácara das Pedras,POINT (-51.16175 -30.03509)
8,Abrigo Gringo,Ajuda RS,SIM,Walir Zottis n 380,sem info,-30.014568,-51.137470,Abrigo Gringo,POINT (-51.13747 -30.01457)
9,Abrigo na Walir Zottis,Ajuda RS,SIM,R. Walir Zottis,sem info,-30.014568,-51.137470,Abrigo na Walir Zottis,POINT (-51.13747 -30.01457)



Tipos de dados:


Nome             str
Fonte            str
Geo              str
End              str
Tipo             str
Lat          float64
Long         float64
name             str
geometry    geometry
dtype: object


Valores ausentes:


Nome         0
Fonte        0
Geo          0
End         82
Tipo         0
Lat          0
Long         0
name         0
geometry     0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Nome,Fonte,Geo,End,Tipo,Lat,Long,name,geometry
count,360,360,360,278,360,360.000000,360.000000,360,360
unique,360,3,1,278,3,NaN,NaN,360,355
top,21 BPM,FSPPA,SIM,"R. Gov. Peracchi Barcellos, 4600 - Restinga, P...",Abrigo,NaN,NaN,21 BPM,POINT (-51.1502728 -30.0669926)
freq,1,199,360,1,173,NaN,NaN,1,2
mean,NaN,NaN,NaN,NaN,NaN,-29.979824,-51.142243,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,0.131112,0.077020,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,-30.220161,-51.359367,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,-30.051114,-51.189586,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,-30.015310,-51.153473,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,-29.921785,-51.112847,NaN,NaN


### Resultados e possíveis usos

A camada contém **360 registros e 9 colunas**, com geometrias `Point` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **355 geometrias distintas** e **0 linhas inteiramente duplicadas**.

**Limitações de conteúdo:** `End` tem **82 ausências (22,8%)**. `Nome` e `name` têm 360 valores distintos cada, mas sua equivalência ainda precisa ser conferida antes de escolher um deles. `Geo` vale “SIM” em todos os registros, o que não comprova precisão da geolocalização. Há 355 geometrias distintas; coordenadas compartilhadas não determinam, isoladamente, duplicidade de estabelecimento.

`Tipo` distingue **173 registros com “Abrigo”**, **88 registros com “Centro de doação”**, **99 registros com “sem info”**. Assim, o total não equivale a 360 abrigos. As fontes informadas são “Ajuda RS”, “FSPPA” e “SOS RS | Zona Sul”; a relação entre essas origens e sua cobertura ainda precisa ser esclarecida.

A estrutura difere da versão de 08/05, portanto o aumento no total de registros não comprova abertura de novos abrigos. Não há campo específico de atualização por registro nesta camada.


## 34. Exploração da água observada por Sentinel-2 — versão validada

### Camada `rhguaiba_sentinel2_agua_obs_06052024_validada`


In [39]:
exploracao_inicial(camadas['name'][28])  # Água observada por Sentinel-2 — versão validada


Exploração inicial da camada: rhguaiba_sentinel2_agua_obs_06052024_validada

Exibindo as primeiras 100 de 4719 linhas. As verificações usam a camada completa.


,CD_MUN,NM_MUN,SIGLA_UF,bacia,MUN_KM2,area_km2,geometry
0,4300109,Agudo,RS,Alto Jacui,534.624,6.492469e+00,"POLYGON ((-51.69957 -29.9393, -51.70049 -29.94..."
1,4300109,Agudo,RS,Alto Jacui,534.624,1.084690e+00,"POLYGON ((-51.70049 -29.94313, -51.69957 -29.9..."
2,4300109,Agudo,RS,Alto Jacui,534.624,1.862223e+00,"POLYGON ((-51.70259 -29.95195, -51.70373 -29.9..."
3,4300109,Agudo,RS,Alto Jacui,534.624,5.444799e+00,"POLYGON ((-51.70373 -29.95675, -51.70259 -29.9..."
4,4300109,Agudo,RS,Alto Jacui,534.624,9.379665e-06,"POLYGON ((-51.70391 -29.95746, -51.70388 -29.9..."
5,4300109,Agudo,RS,Alto Jacui,534.624,2.402290e-05,"POLYGON ((-53.97357 -30.17845, -53.97357 -30.1..."
6,4300109,Agudo,RS,Alto Jacui,534.624,3.375635e-03,"POLYGON ((-53.97494 -30.17308, -53.97489 -30.1..."
7,4300109,Agudo,RS,Alto Jacui,534.624,4.161200e+00,"POLYGON ((-53.96908 -30.17072, -53.968 -30.170..."
8,4300109,Agudo,RS,Alto Jacui,534.624,2.359533e-04,"POLYGON ((-53.97501 -30.17144, -53.97501 -30.1..."
9,4300109,Agudo,RS,Alto Jacui,534.624,1.620452e-03,"POLYGON ((-53.97502 -30.17126, -53.97537 -30.1..."



Tipos de dados:


CD_MUN           str
NM_MUN           str
SIGLA_UF         str
bacia            str
MUN_KM2      float64
area_km2     float64
geometry    geometry
dtype: object


Valores ausentes:


CD_MUN      0
NM_MUN      0
SIGLA_UF    0
bacia       0
MUN_KM2     0
area_km2    0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,CD_MUN,NM_MUN,SIGLA_UF,bacia,MUN_KM2,area_km2,geometry
count,4719,4719,4719,4719,4.719000e+03,4.719000e+03,4719
unique,1,1,1,1,NaN,NaN,4719
top,4300109,Agudo,RS,Alto Jacui,NaN,NaN,"POLYGON ((-51.699573 -29.939302038053196, -51...."
freq,4719,4719,4719,4719,NaN,NaN,1
mean,NaN,NaN,NaN,NaN,5.346240e+02,6.939622e-01,NaN
std,NaN,NaN,NaN,NaN,1.136989e-13,6.995182e+00,NaN
min,NaN,NaN,NaN,NaN,5.346240e+02,8.573971e-08,NaN
25%,NaN,NaN,NaN,NaN,5.346240e+02,8.124138e-05,NaN
50%,NaN,NaN,NaN,NaN,5.346240e+02,3.564551e-04,NaN
75%,NaN,NaN,NaN,NaN,5.346240e+02,3.663280e-03,NaN


### Resultados e possíveis usos

A camada contém **4719 registros e 7 colunas**, com geometrias `Polygon` em **EPSG:4326**. Foram identificados **0 registros inteiramente duplicados**, **0 geometrias nulas** e **0 vazias**.

**Ponto crítico para investigação:** todos os 4.719 registros apresentam os mesmos atributos territoriais: `CD_MUN = "4300109"`, `NM_MUN = "Agudo"`, `SIGLA_UF = "RS"`, `bacia = "Alto Jacui"` e `MUN_KM2 = 534.624`. Essa uniformidade não comprova que todas as geometrias pertençam a Agudo e precisa ser conferida espacialmente antes de agrupar por município ou bacia. Uma possível associação incorreta de atributos não foi confirmada nem corrigida.

Não foram identificadas ausências. Diferentemente da versão anterior, `area_km2` está em ponto flutuante e apresenta valores distintos por registro; a unidade e a consistência geométrica ainda precisam ser verificadas. `MUN_KM2` é repetido e não deve ser somado entre os polígonos para estimar área municipal.

O nome sugere uma camada de água observada em imagens Sentinel-2 de **06/05/2024**. A metodologia, a data de aquisição, as classes e a precisão precisam ser confirmadas na documentação. **Água identificada por satélite pode incluir corpos de água permanentes, não apenas áreas excepcionalmente inundadas.**

Esta versão não apresenta `DN`. A interpretação dos atributos territoriais e de área depende de documentação e conferência espacial. Cálculos de área exigiriam um CRS projetado adequado. A resolução da imagem, a data e possíveis erros de classificação limitam a precisão de eventuais conclusões sobre estabelecimentos individuais.


## 35. Exploração dos abrigos — 10/05/2024 às 13h

### Camada `rs_crise_abrigos_10052024_13h`


In [40]:
exploracao_inicial(camadas['name'][29])  # Abrigos — 10/05/2024 às 13h


Exploração inicial da camada: rs_crise_abrigos_10052024_13h

Exibindo as primeiras 100 de 357 linhas. As verificações usam a camada completa.


,name,FONTE DE DADOS,ENDEREÇO,CIDADE,TIPO LOCAL,ALAGADO?,LAT,LONG,Nome da Instituição,Last Modified,Doação sobrando,Demandas,Nome Responsável (Centro/Instituição),Telefone do Responsável,Tipo,geometry
0,21 BPM,Ajuda RS,"R. Gov. Peracchi Barcellos, 4600 - Restinga, P...",Porto Alegre,Abrigo,NaN,-30.152539,-51.152107,21 BPM,15:29:00.000,0,"Água,Leite,kit higiene,Roupas Diversas",0,0,Abrigo,POINT (-51.15211 -30.15254)
1,35 CTG,Ajuda RS,"Av. Ipiranga, 5300 - Jardim Botânico, Porto Al...",Porto Alegre,Abrigo,NaN,-30.054988,-51.186973,35 CTG,10:50:00.000,0,"Roupa Masculina Infantil,Pomada Assadura,Frald...",Wagner Gaspar/Chico/Josiane,51992178245/51992836459/51996224279 / 51992178245,Abrigo,POINT (-51.18697 -30.05499)
2,A Igreja de Jesus Cristo dos Santos dos Último...,FSPPA,"Av. Princesa Isabel, 300 - Santana, Porto Alegre",Porto Alegre,Centro de doação,NaN,-30.048684,-51.209819,NaN,NaN,NaN,NaN,NaN,NaN,Doacoes,POINT (-51.20982 -30.04868)
3,AABB Porto Alegre,Ajuda RS,"Av. Cel. Marcos, 1000 - Pedra Redonda, Porto A...",Porto Alegre,Abrigo,NaN,-30.123390,-51.245654,AABB Porto Alegre,18:38:00.000,0,"Alimentos,Marmitas",Stepan/ João/Marcelo,+55 51 98497-2885 / 51 98645-4246 / 51 3243-10...,Abrigo,POINT (-51.24565 -30.12339)
4,ABAMF (Ginásio atrás da Brigada Militar na Ron...,FSPPA,"R. Germano Fehse, 90 - Rondônia, Novo Hamburgo",Novo Hamburgo,Abrigo,NaN,-29.704012,-51.120434,NaN,NaN,NaN,NaN,NaN,NaN,Abrigo,POINT (-51.12043 -29.70401)
5,ABCREI,Ajuda RS,"Rua Beco dos Soares, 765 - São Lucas, Viamão",Viamão,Abrigo,NaN,-30.086662,-51.074403,ABCREI,11:03:00.000,0,0,Ari,+55 51 9292-7070,Abrigo,POINT (-51.0744 -30.08666)
6,Abrigo da Bento (Atacadao Miranda Thales Robinho),Ajuda RS,"Av. Bento Gonçalves 6623 - Partenon, Porto Alegre",Porto Alegre,Abrigo,NaN,-30.066993,-51.150273,NaN,NaN,NaN,NaN,NaN,NaN,Abrigo,POINT (-51.15027 -30.06699)
7,Abrigo de animais Chácara das Pedras,Ajuda RS,rua dr Barbosa Gonçalves 61 - Chácara das Pedr...,Porto Alegre,Abrigo de animais,NaN,-30.035095,-51.161745,Abrigo de animais Chácara das Pedras,23:35:00.000,0,0,Julia Zelanis,51 99363-5121,Abrigo de animais,POINT (-51.16175 -30.03509)
8,Abrigo Gringo,Ajuda RS,"R. Walir Zottis, 380 - Jardim Itú, Porto Alegre",Porto Alegre,Abrigo,NaN,-30.014568,-51.137470,Abrigo Gringo,18:40:00.000,0,"Roupas de Cama,Brinquedos",Daniela,51 997819262,Abrigo,POINT (-51.13747 -30.01457)
9,AD Bom Princípio,FSPPA,"R. das Acácias, 585 - Jardim do Vale, Bom Prin...",Bom Princípio,Abrigo,NaN,-29.484687,-51.359367,NaN,NaN,NaN,NaN,NaN,NaN,Abrigo,POINT (-51.35937 -29.48469)



Tipos de dados:


name                                          str
FONTE DE DADOS                                str
ENDEREÇO                                      str
CIDADE                                        str
TIPO LOCAL                                    str
ALAGADO?                                      str
LAT                                       float64
LONG                                      float64
Nome da Instituição                           str
Last Modified                                 str
Doação sobrando                               str
Demandas                                      str
Nome Responsável (Centro/Instituição)         str
Telefone do Responsável                       str
Tipo                                          str
geometry                                 geometry
dtype: object


Valores ausentes:


name                                       0
FONTE DE DADOS                             0
ENDEREÇO                                  63
CIDADE                                   251
TIPO LOCAL                                90
ALAGADO?                                 347
LAT                                        0
LONG                                       0
Nome da Instituição                      197
Last Modified                            197
Doação sobrando                          197
Demandas                                 197
Nome Responsável (Centro/Instituição)    197
Telefone do Responsável                  197
Tipo                                      90
geometry                                   0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,name,FONTE DE DADOS,ENDEREÇO,CIDADE,TIPO LOCAL,ALAGADO?,LAT,LONG,Nome da Instituição,Last Modified,Doação sobrando,Demandas,Nome Responsável (Centro/Instituição),Telefone do Responsável,Tipo,geometry
count,357,357,294,106,267,10,357.000000,357.000000,160,160,160,160,160,160,267,357
unique,357,3,290,10,4,2,NaN,NaN,160,120,15,84,120,123,3,355
top,21 BPM,FSPPA,"Av. Princesa Isabel, 921 - Santana, Porto Alegre",Porto Alegre,Abrigo,SIM,NaN,NaN,21 BPM,10:50:00.000,0,0,0,0,Abrigo,POINT (-51.2159708 -30.1032296)
freq,1,199,2,70,178,8,NaN,NaN,1,7,113,71,39,38,178,2
mean,NaN,NaN,NaN,NaN,NaN,NaN,-29.978335,-51.142011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,0.131396,0.077179,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,-30.220161,-51.359367,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,-30.050936,-51.189340,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,-30.015224,-51.153338,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,-29.920326,-51.112884,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Resultados e possíveis usos

A camada contém **357 registros e 16 colunas**, com geometrias `Point` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **355 geometrias distintas** e **0 linhas inteiramente duplicadas**.

**Limitações principais:** `CIDADE` tem 251 ausências, `ENDEREÇO` 63, `TIPO LOCAL` e `Tipo` 90 cada, e `ALAGADO?` 347. Entre os dez valores preenchidos de `ALAGADO?`, oito são “SIM” e dois “NÃO”; ausência não significa local não alagado. Há variações como “Porto Alegre” e “Porto Alegre ”, sem padronização aplicada.

A camada reúne abrigos, abrigos de animais e locais de doação; o total não representa apenas abrigos humanos. `TIPO LOCAL` e `Tipo` usam categorias diferentes, cuja correspondência precisaria ser examinada antes de agrupar.

Os campos de demandas, doações, responsáveis, contatos e `Last Modified` têm 197 ausências cada. Há textos “0” em alguns desses atributos, que precisam ser interpretados antes de qualquer contagem. `Last Modified` contém exemplos de horários sem data completa, insuficientes para confirmar a atualização de cada registro no momento indicado no nome da camada.

A estrutura difere das versões anteriores, portanto diferenças nos totais não comprovam aberturas ou fechamentos de abrigos.


## 36. Exploração das unidades de atenção primária — 10/05/2024

### Camada `poa_saude_unidades_atencao_primaria_10052024`


In [41]:
exploracao_inicial(camadas['name'][30])  # Atenção primária — 10/05/2024


Exploração inicial da camada: poa_saude_unidades_atencao_primaria_10052024

Exibindo as primeiras 100 de 136 linhas. As verificações usam a camada completa.


,name,End,Bairro,Cidade,End_cidade,Latitude,Longitude,Situacao,Telefone,Atendimento,geometry
0,Clínica da Família Álvaro Difini,"Rua Álvaro Difini, 520",Restinga,Porto Alegre,"Rua Álvaro Difini, 520, Restinga, Porto Alegre",-30.148886,-51.146084,Aberta,"(51) 4076-5011 (WhatsApp), 4076-5012, 4076-501...",De segunda a sexta das 7h às 22h. Fechada para...,POINT (-51.14608 -30.14889)
1,Clínica da Família Campo da Tuca,"Rua Coronel José Rodrigues Sobral, 958",Partenon,Porto Alegre,"Rua Coronel José Rodrigues Sobral, 958, Parten...",-30.067885,-51.176962,Aberta,"(51) 3289-5660 (WhatsApp), 3289-5606, 3289-5607",De segunda a sexta das 07h às 22h.,POINT (-51.17696 -30.06788)
2,Clínica da Família José Mauro Ceratti Lopes,"Estrada João Antônio da Silveira, 3330",Restinga,Porto Alegre,"Estrada João Antônio da Silveira, 3330, Restin...",-30.143014,-51.128977,Aberta,"(51) 3289-5203 (WhatsApp), 3289-5201, 3289-520...",De segunda a sexta das 7h às 22h. Fechada par ...,POINT (-51.12898 -30.14301)
3,US Alto Erechim,"Rua Doutor Ney Cabral, 581",Nonoai,Porto Alegre,"Rua Doutor Ney Cabral, 581, Nonoai, Porto Alegre",-30.089942,-51.212331,Aberta,"(51) 3289-5589 (WhatsApp), 3289-2347",De segunda a sexta das 08h às 17h. Fechada par...,POINT (-51.21233 -30.08994)
4,US Aparício Borges,"Rua São Miguel, 487",Coronel Aparício Borges,Porto Alegre,"Rua São Miguel, 487, Coronel Aparício Borges, ...",-30.074953,-51.191408,Aberta,"(51) 3289-5774 (WhatsApp), 3289-5773, 3289-8265",De segunda a sexta das 08h às 17h. Fechada par...,POINT (-51.19141 -30.07495)
5,US Bananeiras,"Avenida Coronel Aparício Borges, 2494",Coronel Aparício Borges,Porto Alegre,"Avenida Coronel Aparício Borges, 2494, Coronel...",-30.065276,-51.186758,Aberta,"(51) 3289-8227 (WhatsApp), 3289-8228, 3289-8229",De segunda a sexta das 07h às 19h. Fechada par...,POINT (-51.18676 -30.06528)
6,US Barão de Bagé,"Rua Araruama, 487",Vila Jardim,Porto Alegre,"Rua Araruama, 487, Vila Jardim, Porto Alegre",-30.035311,-51.157603,Aberta,"(51) 3345-1503 (WhatsApp), 3334-7196, 3357-2476",De segunda a sexta das 08h às 18h. Fechada par...,POINT (-51.1576 -30.03531)
7,US Batista Flores,"Avenida Serafin Machado, 215 - Vila Batista Fl...",Mario Quintana,Porto Alegre,"Avenida Serafin Machado, 215 - Vila Batista Fl...",-30.027876,-51.105048,Aberta,"(51) 3289-5614 (WhatsApp), 3289-5615",De segunda à sexta das 08 às 12h e das 13 às 1...,POINT (-51.10505 -30.02788)
8,US Beco do Adelar,"Avenida Juca Batista, 3480",Campo Novo,Porto Alegre,"Avenida Juca Batista, 3480, Campo Novo, Porto ...",-30.154835,-51.199226,Aberta,"(51) 3289-5690 (WhatsApp), 3289-5554, 3289-5689",De segunda a sexta das 07h às 19h. Fechada par...,POINT (-51.19923 -30.15483)
9,US Belém Novo,"Rua Florencio Farias,195",Belém Novo,Porto Alegre,"Rua Florencio Farias,195, Belém Novo, Porto Al...",-30.210928,-51.178919,Aberta,"(51) 3289-5723 (WhatsApp), 3289-6560, 3289-572...",De segunda a sexta: das 7h às 22h. Fechada par...,POINT (-51.17892 -30.21093)



Tipos de dados:


name                str
End                 str
Bairro              str
Cidade              str
End_cidade          str
Latitude        float64
Longitude       float64
Situacao            str
Telefone            str
Atendimento         str
geometry       geometry
dtype: object


Valores ausentes:


name           0
End            0
Bairro         0
Cidade         0
End_cidade     0
Latitude       0
Longitude      0
Situacao       0
Telefone       0
Atendimento    0
geometry       0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,name,End,Bairro,Cidade,End_cidade,Latitude,Longitude,Situacao,Telefone,Atendimento,geometry
count,136,136,136,136,136,136.000000,136.000000,136,136,136,136
unique,136,135,56,2,136,NaN,NaN,2,136,94,135
top,Clínica da Família Álvaro Difini,"Rua Paulo Maciel, 220",Rubem Berta,Porto Alegre,"Rua Álvaro Difini, 520, Restinga, Porto Alegre",NaN,NaN,Aberta,"(51) 4076-5011 (WhatsApp), 4076-5012, 4076-501...",De segunda a sexta das 08h às 17h.,POINT (-51.2140794 -30.1307424)
freq,1,2,12,135,1,NaN,NaN,94,1,20,2
mean,NaN,NaN,NaN,NaN,NaN,-30.068102,-51.162455,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,0.058047,0.045688,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,-30.238689,-51.254659,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,-30.109519,-51.199790,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,-30.065375,-51.157673,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,-30.024386,-51.123117,NaN,NaN,NaN,NaN


### Resultados, limitações e possíveis usos

A camada contém **136 registros e 11 colunas**, com geometrias `Point` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **135 geometrias distintas** e **0 linhas inteiramente duplicadas**.

Não foram encontradas ausências reconhecidas por `isna()`. `Situacao` contém **94 valores “Aberta” e 42 “Fechada”**. `Cidade` apresenta 135 valores “Porto Alegre” e um valor “Fechada”, repetindo o tipo de inconsistência observado na versão anterior; isso não determina a situação real da unidade. A comparação e os ajustes entre datas estão documentados no notebook de preparação; os valores desta seção são os originais da fonte.

O identificador textual de nome é `name`, enquanto a versão de 09/05 utiliza `Nome`. Os totais de situação diferem entre os cadastros, mas não permitem identificar mudanças individuais sem correspondência e validação temporal.


## 37. Exploração da inundação observada por Planet/Skysat — Taquari

### Camada `rhguaiba-taquari_planet-skysat_inundacao_obs_06052024`


**Metodologia comum às três camadas Planet/Skysat, segundo o glossário:** imagens de 06/05/2024 e limites visuais de água e lama/detritos foram combinados com FABDEM. As cotas extraídas foram interpoladas por krigagem ordinária e comparadas ao terreno para estimar profundidades e limites; pontos de campo apoiaram a validação. É um produto derivado de observação e modelagem.

Possíveis mapas e comparações regionais dependem da cobertura e da compatibilidade com outras fontes. A extensão em uma data não descreve toda a evolução da cheia.


In [42]:
exploracao_inicial(camadas['name'][31])  # Planet/Skysat — Taquari — 06/05/2024


Exploração inicial da camada: rhguaiba-taquari_planet-skysat_inundacao_obs_06052024

Exibindo as primeiras 100 de 688 linhas. As verificações usam a camada completa.


,id,gridcode,geometry
0,1,1,"MULTIPOLYGON (((-51.84375 -28.77569, -51.84403..."
1,2,1,"MULTIPOLYGON (((-51.84486 -28.78125, -51.84542..."
2,3,1,"MULTIPOLYGON (((-51.84431 -28.78153, -51.84486..."
3,4,1,"MULTIPOLYGON (((-51.45181 -28.78153, -51.45208..."
4,5,1,"MULTIPOLYGON (((-51.45181 -28.78153, -51.45125..."
5,6,1,"MULTIPOLYGON (((-51.84292 -28.78236, -51.84347..."
6,7,1,"MULTIPOLYGON (((-51.84319 -28.78319, -51.84347..."
7,8,1,"MULTIPOLYGON (((-51.83458 -28.78319, -51.83486..."
8,9,1,"MULTIPOLYGON (((-51.83903 -28.78486, -51.83931..."
9,10,1,"MULTIPOLYGON (((-51.83847 -28.78514, -51.83875..."



Tipos de dados:


id             int64
gridcode       int64
geometry    geometry
dtype: object


Valores ausentes:


id          0
gridcode    0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,id,gridcode,geometry
count,688.000000,688.0,688
unique,NaN,NaN,688
top,NaN,NaN,MULTIPOLYGON (((-51.84374999999999 -28.7756944...
freq,NaN,NaN,1
mean,344.500000,1.0,NaN
std,198.752778,0.0,NaN
min,1.000000,1.0,NaN
25%,172.750000,1.0,NaN
50%,344.500000,1.0,NaN
75%,516.250000,1.0,NaN


### Resultados e possíveis usos

A camada contém **688 registros e 3 colunas**, com geometrias `MultiPolygon` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **688 geometrias distintas** e **0 linhas inteiramente duplicadas**.

Não foram identificadas ausências nos atributos. `gridcode` vale **1 em todos os 688 registros**, mas o significado dessa classe precisa ser confirmado. Não há identificação de município ou bairro, nem campo de área: eventuais resultados por unidade territorial dependeriam de cruzamento espacial.


## 38. Exploração da inundação observada por Planet/Skysat — Pardo

### Camada `rhguaiba-pardo_planet-skysat_inundacao_obs_06052024`


**Divergência de abrangência:** o glossário troca os vales nas entradas de Caí e Pardo. O recorte efetivo precisa de conferência espacial; os nomes originais foram mantidos. A metodologia comum está na seção Planet/Skysat — Taquari.


In [43]:
exploracao_inicial(camadas['name'][32])  # Planet/Skysat — Pardo — 06/05/2024


Exploração inicial da camada: rhguaiba-pardo_planet-skysat_inundacao_obs_06052024

Exibindo as primeiras 100 de 210 linhas. As verificações usam a camada completa.


,objectid,id,gridcode,shape_leng,shape_area,geometry
0,352,217,1,0.006111,1.003086e-06,"MULTIPOLYGON Z (((-52.37681 -29.97764 0, -52.3..."
1,353,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-52.37958 -29.97736 0, -52.3..."
2,354,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-52.37681 -29.97681 0, -52.3..."
3,355,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-52.37653 -29.97681 0, -52.3..."
4,356,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-52.37514 -29.95014 0, -52.3..."
5,357,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-52.37542 -29.94986 0, -52.3..."
6,358,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-52.41014 -29.94792 0, -52.4..."
7,359,217,1,0.006667,1.388889e-06,"MULTIPOLYGON Z (((-52.37625 -29.94708 0, -52.3..."
8,360,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-52.40708 -29.92153 0, -52.4..."
9,361,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-52.45042 -29.91542 0, -52.4..."



Tipos de dados:


objectid         int64
id               int64
gridcode         int64
shape_leng     float64
shape_area     float64
geometry      geometry
dtype: object


Valores ausentes:


objectid      0
id            0
gridcode      0
shape_leng    0
shape_area    0
geometry      0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,objectid,id,gridcode,shape_leng,shape_area,geometry
count,210.000000,210.0,210.0,210.000000,2.100000e+02,210
unique,NaN,NaN,NaN,NaN,NaN,210
top,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON Z (((-52.37680555555556 -29.97763...
freq,NaN,NaN,NaN,NaN,NaN,1
mean,456.500000,217.0,1.0,0.044228,1.589331e-04,NaN
std,60.765944,0.0,0.0,0.571855,2.284941e-03,NaN
min,352.000000,217.0,1.0,0.001111,7.716049e-08,NaN
25%,404.250000,217.0,1.0,0.001111,7.716049e-08,NaN
50%,456.500000,217.0,1.0,0.001389,1.157407e-07,NaN
75%,508.750000,217.0,1.0,0.004444,6.172840e-07,NaN


### Resultados e possíveis usos

A camada contém **210 registros e 6 colunas**, com geometrias `MultiPolygon` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **210 geometrias distintas** e **0 linhas inteiramente duplicadas**.

Não foram identificadas ausências. `id` vale **217 em todos os registros** e não funciona como identificador único desta camada; `gridcode` vale **1**, com significado ainda a confirmar. A dimensão Z não comprova altitude real.

`shape_leng` e `shape_area` possuem valores decimais, mas sua unidade e origem ainda não foram verificadas. Como o CRS é geográfico, esses atributos não devem ser interpretados automaticamente como metros ou metros quadrados. Uma possibilidade é conferir medidas recalculadas após validação e reprojeção adequada. Não há identificação de município ou bairro.


## 39. Exploração da inundação observada por Planet/Skysat — Caí

### Camada `rhguaiba-cai_planet-skysat_inundacao_obs_06052024`


**Divergência de abrangência:** o glossário troca os vales nas entradas de Caí e Pardo. O recorte efetivo precisa de conferência espacial; os nomes originais foram mantidos. A metodologia comum está na seção Planet/Skysat — Taquari.


In [44]:
exploracao_inicial(camadas['name'][33])  # Planet/Skysat — Caí — 06/05/2024


Exploração inicial da camada: rhguaiba-cai_planet-skysat_inundacao_obs_06052024

Exibindo as primeiras 100 de 225 linhas. As verificações usam a camada completa.


,objectid,id,gridcode,shape_leng,shape_area,geometry
0,323,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-51.29153 -29.92319 0, -51.2..."
1,324,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-51.29625 -29.91431 0, -51.2..."
2,325,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-51.29347 -29.91347 0, -51.2..."
3,326,217,1,0.002778,3.086420e-07,"MULTIPOLYGON Z (((-51.29069 -29.91042 0, -51.2..."
4,327,217,1,0.000222,1.806317e-09,"MULTIPOLYGON Z (((-51.29371 -29.89681 0, -51.2..."
5,328,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-51.29792 -29.89569 0, -51.2..."
6,329,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-51.35181 -29.89542 0, -51.3..."
7,330,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-51.32153 -29.89375 0, -51.3..."
8,331,217,1,0.001667,1.543210e-07,"MULTIPOLYGON Z (((-51.36764 -29.89208 0, -51.3..."
9,332,217,1,0.001111,7.716049e-08,"MULTIPOLYGON Z (((-51.35792 -29.89181 0, -51.3..."



Tipos de dados:


objectid         int64
id               int64
gridcode         int64
shape_leng     float64
shape_area     float64
geometry      geometry
dtype: object


Valores ausentes:


objectid      0
id            0
gridcode      0
shape_leng    0
shape_area    0
geometry      0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,objectid,id,gridcode,shape_leng,shape_area,geometry
count,225.000000,225.0,225.0,225.000000,2.250000e+02,225
unique,NaN,NaN,NaN,NaN,NaN,225
top,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON Z (((-51.291527777777674 -29.9231...
freq,NaN,NaN,NaN,NaN,NaN,1
mean,435.000000,217.0,1.0,0.054117,1.254067e-04,NaN
std,65.096083,0.0,0.0,0.668936,1.807832e-03,NaN
min,323.000000,217.0,1.0,0.000222,1.806317e-09,NaN
25%,379.000000,217.0,1.0,0.001111,7.716049e-08,NaN
50%,435.000000,217.0,1.0,0.001111,7.716049e-08,NaN
75%,491.000000,217.0,1.0,0.001667,1.543210e-07,NaN


### Resultados e possíveis usos

A camada contém **225 registros e 6 colunas**, com geometrias `MultiPolygon` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **225 geometrias distintas** e **0 linhas inteiramente duplicadas**.

Não foram identificadas ausências. Como na camada do Pardo, `id` vale **217** e `gridcode` vale **1** em todos os registros; `id` não identifica unicamente os polígonos. O significado da classe ainda precisa ser confirmado.

As unidades e a origem de `shape_leng` e `shape_area` não foram verificadas. Esses valores não devem ser interpretados automaticamente em metros ou metros quadrados, pois o CRS é geográfico. Uma possível conferência seria recalcular medidas após validação e reprojeção adequada. A dimensão Z não comprova altitude real.


## 40. Exploração dos abrigos — 10/05/2024 às 20h

### Camada `rs_crise_abrigos_10052024_20h`


In [45]:
exploracao_inicial(camadas['name'][34])  # Abrigos — 10/05/2024 às 20h


Exploração inicial da camada: rs_crise_abrigos_10052024_20h

Exibindo as primeiras 100 de 493 linhas. As verificações usam a camada completa.


,name,TIPO DE LOCAL,ENDEREÇO,O QUE ESTÁ FALTANDO,PRECISA DE VOLUNTARIOS,QUANTOS VOLUNTARIOS,O QUE ESTÁ SOBRANDO,VAGAS DISPONÍVEIS,PESSOAS ABRIGADAS,SOBRE PETS,VAGAS PARA PETS,ATENDIMENTO PARA CRIANÇAS,LOCAL ALAGADO,ÚLTIMA ATUALIZAÇÃO,FONTE,CIDADE,CODIGO,LAT,LONG,geometry
0,21 BPM,Abrigo,"R. Gov. Peracchi Barcellos, 4600 - Restinga, P...","Água,Leite,Kit higiene",Não,0,"Água,Leite,Kit higiene",0.0,0.0,0,None,None,0,05/10/2024 17:04:24,SOS RS,Porto Alegre,15251521,-30.152539,-51.152107,POINT (-51.15211 -30.15254)
1,35 CTG,Abrigo,"Av. Ipiranga, 5300 - Jardim Botânico, Porto Al...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,0,10/05/2024 12:14:39,SOS RS,Porto Alegre,5491869,-30.054988,-51.186973,POINT (-51.18697 -30.05499)
2,A Igreja de Jesus Cristo dos Santos dos Último...,Coleta de doações,"Av. Princesa Isabel, 300 - Santana, Porto Alegre",NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,0,05/10/2024 16:31:14,FSPPA,Porto Alegre,4862098,-30.048684,-51.209819,POINT (-51.20982 -30.04868)
3,AABB Porto Alegre,Abrigo,"Av. Cel. Marcos, 1000 - Pedra Redonda, Porto A...",0,Não,0,0,0.0,120.0,0,None,None,0,10/05/2024 12:15:25,SOS RS,Porto Alegre,12332456,-30.123390,-51.245654,POINT (-51.24565 -30.12339)
4,ABAMF (Ginásio atrás da Brigada Militar na Ron...,Abrigo,"R. Germano Fehse, 90 - Rondônia, Novo Hamburgo",NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,0,10/05/2024 11:51:23,FSPPA,Novo Hamburgo,70401204,-29.704012,-51.120434,POINT (-51.12043 -29.70401)
5,ABCREI,Abrigo,"R. Beco dos Soares, 765 - São Lucas, Viamão",0,0,0,0,0.0,0.0,0,None,None,0,05/10/2024 14:19:17,SOS RS,Viamão,8660744,-30.086662,-51.074403,POINT (-51.0744 -30.08666)
6,Abrigo da Bento (Atacadao Miranda Thales Robinho),Abrigo,"Av. Bento Gonçalves 6623 - Partenon, Porto Alegre",NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,0,05/10/2024 15:40:18,SOS RS,Porto Alegre,6691502,-30.066993,-51.150273,POINT (-51.15027 -30.06699)
7,Abrigo de animais Chácara das Pedras,Abrigo de animais,"R. Dr Barbosa Gonçalves, 61 - Chácara das Pedr...",0,Sim,0,0,0.0,0.0,0,None,None,0,05/10/2024 15:23:43,SOS RS,Porto Alegre,3501617,-30.035095,-51.161745,POINT (-51.16175 -30.03509)
8,Abrigo Gringo,Abrigo,"R. Walir Zottis, 380 - Jardim Itú, Porto Alegre",NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,0,10/05/2024 12:15:21,SOS RS,Porto Alegre,1451374,-30.014568,-51.137470,POINT (-51.13747 -30.01457)
9,AD Bom Princípio,Abrigo,"R. das Acácias, 585 - Jardim do Vale, Bom Prin...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,0,10/05/2024 12:09:50,FSPPA,Bom Princípio,48463593,-29.484687,-51.359367,POINT (-51.35937 -29.48469)



Tipos de dados:


name                              str
TIPO DE LOCAL                     str
ENDEREÇO                          str
O QUE ESTÁ FALTANDO               str
PRECISA DE VOLUNTARIOS            str
QUANTOS VOLUNTARIOS               str
O QUE ESTÁ SOBRANDO               str
VAGAS DISPONÍVEIS             float64
PESSOAS ABRIGADAS             float64
SOBRE PETS                        str
VAGAS PARA PETS                object
ATENDIMENTO PARA CRIANÇAS      object
LOCAL ALAGADO                     str
ÚLTIMA ATUALIZAÇÃO                str
FONTE                             str
CIDADE                            str
CODIGO                            str
LAT                           float64
LONG                          float64
geometry                     geometry
dtype: object


Valores ausentes:


name                           0
TIPO DE LOCAL                  0
ENDEREÇO                       1
O QUE ESTÁ FALTANDO          310
PRECISA DE VOLUNTARIOS       310
QUANTOS VOLUNTARIOS          310
O QUE ESTÁ SOBRANDO          310
VAGAS DISPONÍVEIS            310
PESSOAS ABRIGADAS            310
SOBRE PETS                   310
VAGAS PARA PETS              493
ATENDIMENTO PARA CRIANÇAS    493
LOCAL ALAGADO                 32
ÚLTIMA ATUALIZAÇÃO             1
FONTE                          1
CIDADE                         1
CODIGO                         1
LAT                            1
LONG                           1
geometry                       1
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,name,TIPO DE LOCAL,ENDEREÇO,O QUE ESTÁ FALTANDO,PRECISA DE VOLUNTARIOS,QUANTOS VOLUNTARIOS,O QUE ESTÁ SOBRANDO,VAGAS DISPONÍVEIS,PESSOAS ABRIGADAS,SOBRE PETS,VAGAS PARA PETS,ATENDIMENTO PARA CRIANÇAS,LOCAL ALAGADO,ÚLTIMA ATUALIZAÇÃO,FONTE,CIDADE,CODIGO,LAT,LONG,geometry
count,493,493,492,183,183,183,183,183.000000,183.000000,183,0,0,461,492,492,492,492,4.920000e+02,492.000000,492
unique,488,7,462,82,6,22,82,NaN,NaN,20,0,0,3,388,12,30,460,NaN,NaN,488
top,Colégio Dom Feliciano,Abrigo,0,0,0,0,0,NaN,NaN,0,NaN,NaN,0,30/12/1899 00:00:00,FSPPA,Porto Alegre,São Leopoldo,NaN,NaN,POINT (-51.0972789040565 -29.7486230601761)
freq,2,387,8,95,87,162,95,NaN,NaN,164,NaN,NaN,450,32,198,213,20,NaN,NaN,2
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.846995,87.519126,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.698155e+06,-49.793507,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.218864,450.875581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.881919e+07,5.687886,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.176837e+01,-53.909067,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.006914e+01,-51.204899,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.001111e+01,-51.155402,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,80.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.987373e+01,-51.108078,NaN


### Resultados e possíveis usos

A camada contém **493 registros e 20 colunas**, com geometrias `Point` em **EPSG:4326**. Há **1 geometrias nulas**, **0 vazias**, **488 geometrias distintas** e **3 linhas inteiramente duplicadas**.

**Limitações principais:** foram encontradas **três linhas inteiramente duplicadas**, preservadas sem exclusão. Os 493 registros incluem diferentes tipos de local, variações de categorias e “#N/A”; não representam necessariamente 493 abrigos distintos. O campo `CODIGO` também não é único e precisaria de avaliação antes de servir como chave.

Sete campos de demandas, voluntariado, vagas, pessoas abrigadas e pets têm **310 ausências cada**. `VAGAS PARA PETS` e `ATENDIMENTO PARA CRIANÇAS` estão completamente ausentes. Há textos “0” e zeros numéricos cuja interpretação precisa ser confirmada; não se deve somar vagas ou pessoas sem considerar incompletude e duplicações.

`LOCAL ALAGADO` tem 32 ausências e valores “0”, “SIM” e “NÃO”; o significado de “0” permanece indefinido. As datas de `ÚLTIMA ATUALIZAÇÃO` apresentam exemplos como “05/10/2024” e “10/05/2024”, sugerindo possível ambiguidade de formato dia/mês. A referência temporal precisaria ser esclarecida antes de ordenar ou comparar atualizações. Os exemplos de campos de itens faltantes e sobrando também coincidem, o que poderia motivar uma verificação adicional antes de interpretar necessidades.

Diferenças em relação às versões anteriores podem refletir mudanças no cadastro, na cobertura ou nas fontes, sem comprovar abertura de novos abrigos.


## 41. Exploração dos abrigos — 11/05/2024 às 13h

### Camada `rs_crise_abrigos_11052024_13h`


In [46]:
exploracao_inicial(camadas['name'][35])  # Abrigos — 11/05/2024 às 13h


Exploração inicial da camada: rs_crise_abrigos_11052024_13h

Exibindo as primeiras 100 de 504 linhas. As verificações usam a camada completa.


,name,TIPO DE LOCAL,ENDEREÇO,O QUE ESTÁ FALTANDO,PRECISA DE VOLUNTARIOS,QUANTOS VOLUNTARIOS,O QUE ESTÁ SOBRANDO,VAGAS DISPONÍVEIS,PESSOAS ABRIGADAS,SOBRE PETS,VAGAS PARA PETS,ATENDIMENTO PARA CRIANÇAS,LOCAL ALAGADO,ÚLTIMA ATUALIZAÇÃO,FONTE,CIDADE,CÓDIGO,LATITUDE,LONGITUDE,geometry
0,21 BPM,Abrigo,"R. Gov. Peracchi Barcellos, 4600 - Restinga, P...",Centro de Coleta de Doações,"Água,Leite,Kit higiene",0,Centro de Coleta de Doações,45420.766667,0.0,0,None,None,0,05/10/2024 21:17:27,SOS RS,Porto Alegre,15251521,-30.152539,-51.152107,POINT (-51.15211 -30.15254)
1,35 CTG,Abrigo,"Av. Ipiranga, 5300 - Jardim Botânico, Porto Al...",NaN,NaN,",",NaN,NaN,NaN,NaN,None,None,0,05/11/2024 11:12:49,SOS RS,Porto Alegre,5491869,-30.054988,-51.186973,POINT (-51.18697 -30.05499)
2,A Igreja de Jesus Cristo dos Santos dos Último...,Coleta de doações,"Av. Princesa Isabel, 300 - Santana, Porto Alegre",NaN,NaN,",",NaN,NaN,NaN,NaN,None,None,0,05/11/2024 11:12:36,FSPPA,Porto Alegre,4862098,-30.048684,-51.209819,POINT (-51.20982 -30.04868)
3,AABB Porto Alegre,Abrigo,"Av. Cel. Marcos, 1000 - Pedra Redonda, Porto A...",Abrigos,0,120,Abrigos,45422.719444,0.0,0,None,None,0,05/10/2024 21:17:37,SOS RS,Porto Alegre,12332456,-30.123390,-51.245654,POINT (-51.24565 -30.12339)
4,ABAMF (Ginásio atrás da Brigada Militar na Ron...,Abrigo,"R. Germano Fehse, 90 - Rondônia, Novo Hamburgo",NaN,NaN,",",NaN,NaN,NaN,NaN,None,None,0,05/10/2024 21:17:39,FSPPA,Novo Hamburgo,70401204,-29.704012,-51.120434,POINT (-51.12043 -29.70401)
5,ABCREI,Abrigo,"R. Beco dos Soares, 765 - São Lucas, Viamão",0,0,0,0,45421.460417,0.0,0,None,None,0,05/10/2024 21:17:43,SOS RS,Viamão,8660744,-30.086662,-51.074403,POINT (-51.0744 -30.08666)
6,Abrigo da Bento (Atacadao Miranda Thales Robinho),Abrigo,"Av. Bento Gonçalves 6623 - Partenon, Porto Alegre",NaN,NaN,",",NaN,NaN,NaN,NaN,None,None,0,05/10/2024 15:40:18,SOS RS,Porto Alegre,6691502,-30.066993,-51.150273,POINT (-51.15027 -30.06699)
7,Abrigo de animais Chácara das Pedras,Abrigo de animais,"R. Dr Barbosa Gonçalves, 61 - Chácara das Pedr...",Hotel Pet,0,0,Hotel Pet,0.000000,0.0,0,None,None,0,05/10/2024 15:23:43,SOS RS,Porto Alegre,3501617,-30.035095,-51.161745,POINT (-51.16175 -30.03509)
8,Abrigo Gringo,Abrigo,"R. Walir Zottis, 380 - Jardim Itú, Porto Alegre",NaN,NaN,",",NaN,NaN,NaN,NaN,None,None,0,10/05/2024 12:15:21,SOS RS,Porto Alegre,1451374,-30.014568,-51.137470,POINT (-51.13747 -30.01457)
9,AD Bom Princípio,Abrigo,"R. das Acácias, 585 - Jardim do Vale, Bom Prin...",NaN,NaN,",",NaN,NaN,NaN,NaN,None,None,0,10/05/2024 12:09:50,FSPPA,Bom Princípio,48463593,-29.484687,-51.359367,POINT (-51.35937 -29.48469)



Tipos de dados:


name                              str
TIPO DE LOCAL                     str
ENDEREÇO                          str
O QUE ESTÁ FALTANDO               str
PRECISA DE VOLUNTARIOS            str
QUANTOS VOLUNTARIOS               str
O QUE ESTÁ SOBRANDO               str
VAGAS DISPONÍVEIS             float64
PESSOAS ABRIGADAS             float64
SOBRE PETS                        str
VAGAS PARA PETS                object
ATENDIMENTO PARA CRIANÇAS      object
LOCAL ALAGADO                     str
ÚLTIMA ATUALIZAÇÃO                str
FONTE                             str
CIDADE                            str
CÓDIGO                          int32
LATITUDE                      float64
LONGITUDE                     float64
geometry                     geometry
dtype: object


Valores ausentes:


name                           0
TIPO DE LOCAL                  0
ENDEREÇO                       0
O QUE ESTÁ FALTANDO          323
PRECISA DE VOLUNTARIOS       323
QUANTOS VOLUNTARIOS          171
O QUE ESTÁ SOBRANDO          323
VAGAS DISPONÍVEIS            323
PESSOAS ABRIGADAS            323
SOBRE PETS                   323
VAGAS PARA PETS              504
ATENDIMENTO PARA CRIANÇAS    504
LOCAL ALAGADO                  0
ÚLTIMA ATUALIZAÇÃO             0
FONTE                          0
CIDADE                         0
CÓDIGO                         0
LATITUDE                       0
LONGITUDE                      0
geometry                       0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,name,TIPO DE LOCAL,ENDEREÇO,O QUE ESTÁ FALTANDO,PRECISA DE VOLUNTARIOS,QUANTOS VOLUNTARIOS,O QUE ESTÁ SOBRANDO,VAGAS DISPONÍVEIS,PESSOAS ABRIGADAS,SOBRE PETS,VAGAS PARA PETS,ATENDIMENTO PARA CRIANÇAS,LOCAL ALAGADO,ÚLTIMA ATUALIZAÇÃO,FONTE,CIDADE,CÓDIGO,LATITUDE,LONGITUDE,geometry
count,504,504,504,181,181,333,181,181.000000,181.000000,181,0,0,504,504,504,504,5.040000e+02,504.000000,504.000000,504
unique,499,7,479,27,82,52,27,NaN,NaN,18,0,0,3,398,9,25,NaN,NaN,NaN,498
top,Colégio Dom Feliciano,Abrigo,"Av. Ipiranga, 5300 - Jardim Botânico, Porto Al...",0,0,Sem inf,0,NaN,NaN,0,NaN,NaN,0,30/12/1899 00:00:00,FSPPA,Porto Alegre,NaN,NaN,NaN,POINT (-50.993543 -29.9447303)
freq,2,271,2,68,90,145,68,NaN,NaN,164,NaN,NaN,493,53,201,221,NaN,NaN,NaN,2
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23840.237638,13.209945,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.739670e+07,-30.077170,-51.239677,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22745.790699,51.545343,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.846143e+07,0.464991,0.312956,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.315270e+05,-31.768369,-52.430857,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.559396e+06,-30.076012,-51.207302,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45417.677778,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.992103e+07,-30.014305,-51.160600,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45422.510417,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.962633e+07,-29.888949,-51.117337,NaN


### Resultados e possíveis usos

A camada contém **504 registros e 20 colunas**, com geometrias `Point` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **498 geometrias distintas** e **4 linhas inteiramente duplicadas**.

**Problema importante de interpretação:** os exemplos sugerem possível desalinhamento entre colunas e conteúdo. `O QUE ESTÁ FALTANDO` e `O QUE ESTÁ SOBRANDO` contêm categorias como “Abrigos” e “Centro de Coleta de Doações”; `PRECISA DE VOLUNTARIOS` contém listas de materiais; `SOBRE PETS` contém exemplos de quantidades e turnos de voluntários. `VAGAS DISPONÍVEIS` inclui valores como **45420.7666666666**, incompatíveis com uma contagem usual de vagas e possivelmente relacionados a datas serializadas. Essas hipóteses exigem conferência com a fonte; nenhum realinhamento ou conversão foi aplicado.

Por isso, **os campos operacionais não devem ser interpretados pelas legendas atuais nem somados sem esclarecimento**. Seis desses campos têm 323 ausências cada; `QUANTOS VOLUNTARIOS` tem 171, e os campos de vagas para pets e atendimento infantil estão totalmente ausentes.

Há **quatro linhas inteiramente duplicadas**, preservadas. `CÓDIGO` não é único. Textos “0”, “Sem inf” e variações de categorias podem representar informação faltante não reconhecida por `isna()`. `ÚLTIMA ATUALIZAÇÃO` também apresenta exemplos de datas potencialmente ambíguas entre dia e mês.


## 42. Exploração dos abrigos — 11/05/2024 às 18h

### Camada `rs_crise_abrigos_11052024_18h`


In [47]:
exploracao_inicial(camadas['name'][36])  # Abrigos — 11/05/2024 às 18h


Exploração inicial da camada: rs_crise_abrigos_11052024_18h

Exibindo as primeiras 100 de 574 linhas. As verificações usam a camada completa.


,name,TIPO DE LOCAL,ENDEREÇO,O QUE ESTÁ FALTANDO,O QUE ESTÁ SOBRANDO,PRECISA DE VOLUNTARIOS,QUANTOS VOLUNTARIOS,PESSOAS ABRIGADAS,VAGAS DISPONÍVEIS,VAGAS DISPONÍVEIS PARA PETS,INFORMAÇÕES SOBRE PETS,SITUAÇÃO,ÚLTIMA ATUALIZAÇÃO,FONTE,CIDADE,CÓDIGO,LATITUDE,LONGITUDE,geometry
0,35 CTG,Abrigo,"Av. Ipiranga, 5300 - Jardim Botânico, Porto Al...","Shampoo/condicionador,Escova dental,Gilette,Fi...",Não,Não,0,0.0,0.0,0.0,0,0,11/05/2024 16:58:59,SOS RS,Porto Alegre,5491869,-30.054988,-51.186973,POINT (-51.18697 -30.05499)
1,AABB Porto Alegre,Abrigo,"Av. Cel. Marcos, 1000 - Pedra Redonda, Porto A...",0,"Roupas Femininas,Roupas Masculino,Água,Cesta b...",Não,0,120.0,NaN,0.0,0,0,10/05/2024 17:16:00,SOS RS,Porto Alegre,12332456,-30.123390,-51.245654,POINT (-51.24565 -30.12339)
2,ABAMF (Ginásio atrás da Brigada Militar na Ron...,Abrigo,"R. Germano Fehse, 90 - Rondônia, Novo Hamburgo",NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0,NaN,FSPPA,Novo Hamburgo,70401204,-29.704012,-51.120434,POINT (-51.12043 -29.70401)
3,ABCREI,Abrigo,"R. Beco dos Soares, 765 - São Lucas, Viamão",0,0,0,0,0.0,NaN,0.0,0,0,09/05/2024 11:02:59,SOS RS,Viamão,8660744,-30.086662,-51.074403,POINT (-51.0744 -30.08666)
4,Abrigo da Bento (Atacadao Miranda Thales Robinho),Abrigo,"Av. Bento Gonçalves 6623 - Partenon, Porto Alegre",NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0,NaN,SOS RS,Porto Alegre,6691502,-30.066993,-51.150273,POINT (-51.15027 -30.06699)
5,Abrigo Gringo,Abrigo,"R. Walir Zottis, 380 - Jardim Itú, Porto Alegre",NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,DUPLICADO,NaN,SOS RS,Porto Alegre,1451374,-30.014568,-51.137470,POINT (-51.13747 -30.01457)
6,AD Bom Princípio,Abrigo,"R. das Acácias, 585 - Jardim do Vale, Bom Prin...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,FSPPA,Bom Princípio,48463593,-29.484687,-51.359367,POINT (-51.35937 -29.48469)
7,AFISVEC,Abrigo,"R. Salvador Calamucci, 56 - Cavalhada, Porto A...","Colchão,Meias",0,Não,0,140.0,NaN,0.0,0,0,09/05/2024 16:15:59,SOS RS,Porto Alegre,9702269,-30.097028,-51.226995,POINT (-51.227 -30.09703)
8,AMOVI,Abrigo,"Av. dos Industriários. 433 - Passo d'Areia, Po...","Colchão,Arroz,Feijão,Café,Macarrão,óleo de coz...",0,Não,0,0.0,NaN,0.0,0,0,10/05/2024 21:10:00,SOS RS,Porto Alegre,1521740,-30.015224,-51.174038,POINT (-51.17404 -30.01522)
9,AMVEP,Abrigo,"Av. 21 de Abril, 792 - Sarandi, Porto Alegre",0,0,Não,0,0.0,0.0,0.0,0,0,05/05/2024 10:44:59,FSPPA,Porto Alegre,98571298,-29.985790,-51.129872,POINT (-51.12987 -29.98579)



Tipos de dados:


name                                str
TIPO DE LOCAL                       str
ENDEREÇO                            str
O QUE ESTÁ FALTANDO                 str
O QUE ESTÁ SOBRANDO                 str
PRECISA DE VOLUNTARIOS              str
QUANTOS VOLUNTARIOS                 str
PESSOAS ABRIGADAS               float64
VAGAS DISPONÍVEIS               float64
VAGAS DISPONÍVEIS PARA PETS     float64
INFORMAÇÕES SOBRE PETS              str
SITUAÇÃO                            str
ÚLTIMA ATUALIZAÇÃO                  str
FONTE                               str
CIDADE                              str
CÓDIGO                            int32
LATITUDE                        float64
LONGITUDE                       float64
geometry                       geometry
dtype: object


Valores ausentes:


name                             0
TIPO DE LOCAL                    0
ENDEREÇO                         0
O QUE ESTÁ FALTANDO            379
O QUE ESTÁ SOBRANDO            380
PRECISA DE VOLUNTARIOS         380
QUANTOS VOLUNTARIOS            380
PESSOAS ABRIGADAS              380
VAGAS DISPONÍVEIS              380
VAGAS DISPONÍVEIS PARA PETS    380
INFORMAÇÕES SOBRE PETS         380
SITUAÇÃO                         0
ÚLTIMA ATUALIZAÇÃO             380
FONTE                            0
CIDADE                           0
CÓDIGO                           0
LATITUDE                         0
LONGITUDE                        0
geometry                         0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,name,TIPO DE LOCAL,ENDEREÇO,O QUE ESTÁ FALTANDO,O QUE ESTÁ SOBRANDO,PRECISA DE VOLUNTARIOS,QUANTOS VOLUNTARIOS,PESSOAS ABRIGADAS,VAGAS DISPONÍVEIS,VAGAS DISPONÍVEIS PARA PETS,INFORMAÇÕES SOBRE PETS,SITUAÇÃO,ÚLTIMA ATUALIZAÇÃO,FONTE,CIDADE,CÓDIGO,LATITUDE,LONGITUDE,geometry
count,574,574,574,195,194,194,194,194.000000,194.000000,194.0,194,574,194,574,574,5.740000e+02,574.000000,574.000000,574
unique,566,9,546,88,15,6,20,NaN,NaN,NaN,26,2,92,9,25,NaN,NaN,NaN,565
top,Assembleia de Deus,Abrigo,"Av. Ipiranga, 5300 - Jardim Botânico, Porto Al...",0,0,0,0,NaN,NaN,NaN,0,0,30/12/1899 00:00:00,FSPPA,Porto Alegre,NaN,NaN,NaN,POINT (-52.210391 -31.733696)
freq,2,322,2,95,120,89,175,NaN,NaN,NaN,167,557,89,201,234,NaN,NaN,NaN,2
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.773196,11.778351,0.0,NaN,NaN,NaN,NaN,NaN,4.889415e+07,-30.044718,-51.228647,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,438.393251,49.266089,0.0,NaN,NaN,NaN,NaN,NaN,3.726505e+07,0.455453,0.299360,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,2.915490e+05,-31.768369,-52.430857,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,6.547208e+06,-30.062838,-51.197967,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,6.907612e+07,-29.999190,-51.153317,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.500000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,7.839649e+07,-29.783238,-51.116370,NaN


### Resultados e possíveis usos

A camada contém **574 registros e 19 colunas**, com geometrias `Point` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **565 geometrias distintas** e **4 linhas inteiramente duplicadas**.

**Limitações principais:** há **quatro linhas inteiramente duplicadas**, mantidas sem exclusão. `SITUAÇÃO` contém “0” e “DUPLICADO”, que são marcações de cadastro e não informam funcionamento ou alagamento. `CÓDIGO` não é único, e os tipos de local incluem abrigos, espaços para animais, coleta de doações, cozinhas e combinações dessas atividades.

A maioria dos campos operacionais e `ÚLTIMA ATUALIZAÇÃO` têm **380 ausências cada**; `O QUE ESTÁ FALTANDO` tem 379. Todos os valores preenchidos de `VAGAS DISPONÍVEIS PARA PETS` são zero, mas sua interpretação depende da fonte. Os textos “0” em outros campos não devem ser considerados informação válida de ausência de demanda sem confirmação.

Os exemplos dos campos operacionais são mais compatíveis com seus títulos do que na versão de 11/05 às 13h, mas isso não confirma correção integral do cadastro. `QUANTOS VOLUNTARIOS` mistura quantidades, turnos, descrições e links, impedindo uma soma direta. As datas preenchidas variam entre dias, sem comprovar atualização simultânea de todos os registros.

Possíveis comparações com versões anteriores exigiriam correspondência de estabelecimentos, revisão de duplicações e compatibilidade dos campos; o aumento de registros não comprova abertura de novos abrigos.


## 43. Exploração da severidade simulada HEC-RAS — 06/05/2024

### Camada `rhguaiba_hecras_severidade_simulada_06052024`


**Esclarecimento do glossário:** Segundo o glossário, as classes de severidade física do escoamento são **1 = baixa, 2 = média e 3 = alta**, para 06/05/2024. A nota técnica define um mapa de perigo baseado em **IP = velocidade × profundidade** e alerta para incertezas nas profundidades dos modelos regionais, pela falta de batimetria dos rios e pela resolução espacial. Não se trata de uma medida direta de danos.


In [48]:
exploracao_inicial(camadas['name'][37])  # Severidade simulada HEC-RAS — 06/05/2024


Exploração inicial da camada: rhguaiba_hecras_severidade_simulada_06052024

Exibindo as primeiras 100 de 42273 linhas. As verificações usam a camada completa.


,Severidade,geometry
0,1,"POLYGON ((-51.93682 -29.09194, -51.93682 -29.0..."
1,1,"POLYGON ((-51.91364 -29.12805, -51.91364 -29.1..."
2,1,"POLYGON ((-51.84052 -29.13147, -51.84052 -29.1..."
3,1,"POLYGON ((-51.85058 -29.13236, -51.85058 -29.1..."
4,1,"POLYGON ((-51.84699 -29.13308, -51.84699 -29.1..."
5,1,"POLYGON ((-51.85112 -29.13308, -51.85112 -29.1..."
6,2,"POLYGON ((-51.84896 -29.13344, -51.84896 -29.1..."
7,1,"POLYGON ((-51.8486 -29.13344, -51.8486 -29.133..."
8,2,"POLYGON ((-51.831 -29.13362, -51.831 -29.13398..."
9,1,"POLYGON ((-51.8513 -29.13398, -51.8513 -29.134..."



Tipos de dados:


Severidade       int32
geometry      geometry
dtype: object


Valores ausentes:


Severidade    0
geometry      0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,Severidade,geometry
count,42273.000000,42273
unique,NaN,42273
top,NaN,POLYGON ((-51.93681680359743 -29.0919404762107...
freq,NaN,1
mean,1.572115,NaN
std,0.608622,NaN
min,1.000000,NaN
25%,1.000000,NaN
50%,2.000000,NaN
75%,2.000000,NaN


### Resultados e possíveis usos

A camada contém **42273 registros e 2 colunas**, com geometrias `Polygon` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **42273 geometrias distintas** e **0 linhas inteiramente duplicadas**.

Não foram identificadas ausências. `Severidade` contém três códigos: **1 (20.743 registros), 2 (18.875) e 3 (2.655)**. Esses totais contam polígonos e não representam proporções de área. O glossário identifica as classes como baixa, média e alta severidade física e apresenta ressalvas sobre o modelo; médias e quartis produzidos automaticamente por `describe()` não têm interpretação física confirmada para esses códigos. Não há município, bairro ou campo de área.


## 44. Exploração da análise ADA — 06/05/2024

### Camada `analise_rhguaiba_ada_06052024`


**Esclarecimento do glossário:** O glossário define **ADA como área diretamente afetada**, derivada das manchas de água e lama observadas por sensoriamento remoto. Entretanto, a descrição menciona **06/06/2024**, enquanto o nome indica **06/05/2024**; a data permanece divergente. A definição não equivale a comprovação de danos materiais.


In [49]:
exploracao_inicial(camadas['name'][38])  # Análise ADA — 06/05/2024


Exploração inicial da camada: analise_rhguaiba_ada_06052024

Exibindo as primeiras 100 de 153 linhas. As verificações usam a camada completa.


,CD_MUN,NM_MUN,SIGLA_UF,MUN_KM2,ADA_M2,geometry
0,4300002,Lagoa dos Patos,RS,10201.524,1.748324e+06,"MULTIPOLYGON (((-51.31244 -30.24022, -51.31257..."
1,4300109,Agudo,RS,534.624,1.117410e+08,"MULTIPOLYGON (((-53.35709 -29.66146, -53.35707..."
2,4300554,Alto Alegre,RS,115.335,9.056600e+06,"MULTIPOLYGON (((-53.05691 -28.83227, -53.05687..."
3,4300604,Alvorada,RS,71.700,7.402266e+04,"MULTIPOLYGON (((-51.08824 -29.99693, -51.08824..."
4,4300703,Anta Gorda,RS,242.100,9.564608e+05,"MULTIPOLYGON (((-52.02378 -28.92168, -52.02377..."
5,4301008,Arroio do Meio,RS,157.088,4.539230e+07,"MULTIPOLYGON (((-52.0012 -29.40738, -52.00112 ..."
6,4301206,Arroio do Tigre,RS,315.132,5.652525e+06,"MULTIPOLYGON (((-53.1432 -29.3535, -53.14319 -..."
7,4301404,Arvorezinha,RS,269.310,5.136730e+05,"MULTIPOLYGON (((-52.05037 -28.83412, -52.05035..."
8,4301909,Barra do Ribeiro,RS,729.316,3.768837e+07,"MULTIPOLYGON (((-51.2824 -30.38955, -51.2824 -..."
9,4302006,Barros Cassal,RS,647.994,4.072586e+04,"MULTIPOLYGON (((-52.43764 -29.14971, -52.43764..."



Tipos de dados:


CD_MUN           str
NM_MUN           str
SIGLA_UF         str
MUN_KM2      float64
ADA_M2       float64
geometry    geometry
dtype: object


Valores ausentes:


CD_MUN      0
NM_MUN      0
SIGLA_UF    0
MUN_KM2     0
ADA_M2      0
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,CD_MUN,NM_MUN,SIGLA_UF,MUN_KM2,ADA_M2,geometry
count,153,153,153,153.000000,1.530000e+02,153
unique,153,153,1,NaN,NaN,153
top,4300002,Lagoa dos Patos,RS,NaN,NaN,MULTIPOLYGON (((-51.3124422955911 -30.24021578...
freq,1,1,153,NaN,NaN,1
mean,NaN,NaN,NaN,467.150222,2.789672e+07,NaN
std,NaN,NaN,NaN,1026.089302,6.010923e+07,NaN
min,NaN,NaN,NaN,27.676000,1.886210e+03,NaN
25%,NaN,NaN,NaN,103.009000,8.981062e+05,NaN
50%,NaN,NaN,NaN,182.756000,4.822946e+06,NaN
75%,NaN,NaN,NaN,424.339000,2.168180e+07,NaN


### Resultados e possíveis usos

A camada contém **153 registros e 6 colunas**, com geometrias `MultiPolygon` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **153 geometrias distintas** e **0 linhas inteiramente duplicadas**.

Não foram identificadas ausências. `CD_MUN` e `NM_MUN` têm 153 valores distintos cada, mas há um registro denominado **“Lagoa dos Patos”**. Portanto, o total não deve ser descrito automaticamente como 153 municípios, e a validade dos códigos territoriais precisa ser conferida.

Os nomes `MUN_KM2` e `ADA_M2` sugerem áreas em quilômetros quadrados e metros quadrados, respectivamente, mas as unidades e os detalhes metodológicos ainda precisam de confirmação; o glossário define ADA como área diretamente afetada. Caso sejam confirmados, uma possibilidade é avaliar proporções entre áreas, convertendo primeiro para a mesma unidade (**1 km² = 1.000.000 m²**), sem interpretar antecipadamente o resultado como proporção inundada ou danificada.


## 45. Exploração da lama observada por Sentinel-2 — 06/05/2024

### Camada `rhguaiba_sentinel2_lama_obs_06052024_2`


**Esclarecimento do glossário:** O glossário repete o nome sem o sufixo `_2` em duas entradas: uma descreve mancha de lama e a outra setores censitários recortados pela mancha. Não é possível atribuir automaticamente uma dessas descrições à camada local apenas pelo nome; a correspondência permanece pendente.


In [50]:
exploracao_inicial(camadas['name'][39])  # Lama observada por Sentinel-2 — 06/05/2024


Exploração inicial da camada: rhguaiba_sentinel2_lama_obs_06052024_2

Exibindo as primeiras 100 de 108 linhas. As verificações usam a camada completa.


,DN,geometry
0,0.0,"MULTIPOLYGON (((-51.90883 -28.98644, -51.90887..."
1,0.0,"MULTIPOLYGON (((-53.00954 -29.1464, -53.00956 ..."
2,0.0,"MULTIPOLYGON (((-52.0208 -28.84123, -52.02079 ..."
3,0.0,"MULTIPOLYGON (((-51.62057 -29.07962, -51.62056..."
4,0.0,"MULTIPOLYGON (((-51.31922 -29.48152, -51.31916..."
5,0.0,"MULTIPOLYGON (((-52.13384 -28.61269, -52.13384..."
6,0.0,"MULTIPOLYGON (((-53.06248 -28.83324, -53.06248..."
7,0.0,"MULTIPOLYGON (((-51.38253 -29.74045, -51.38253..."
8,0.0,"MULTIPOLYGON (((-51.92009 -29.29404, -51.92009..."
9,0.0,"MULTIPOLYGON (((-51.84102 -28.65048, -51.84102..."



Tipos de dados:


DN           float64
geometry    geometry
dtype: object


Valores ausentes:


DN          5
geometry    0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,DN,geometry
count,103.0,108
unique,NaN,108
top,NaN,MULTIPOLYGON (((-51.908830897148505 -28.986436...
freq,NaN,1
mean,0.0,NaN
std,0.0,NaN
min,0.0,NaN
25%,0.0,NaN
50%,0.0,NaN
75%,0.0,NaN


### Resultados e possíveis usos

A camada contém **108 registros e 2 colunas**, com geometrias `MultiPolygon` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **108 geometrias distintas** e **0 linhas inteiramente duplicadas**.

`DN` possui **103 valores iguais a zero e cinco ausências**. Não é possível interpretar zero como ausência ou presença de lama sem confirmar a legenda da classificação. Os registros sem `DN` não foram excluídos. Não há campo de área ou identificação de município e bairro.


## 46. Exploração das comportas de drenagem — RMPA

### Camada `rmpa_sb_drenagem_comporta`


In [51]:
exploracao_inicial(camadas['name'][40])  # Comportas de drenagem — RMPA


Exploração inicial da camada: rmpa_sb_drenagem_comporta



,id,id_cbs,nm_cbs,c1_m3/s,Número,Nome,COORD_X,COORD_Y,Name,Status_07_05,Status_06_05,Status_08_05,Status_09_05,Status_10_05,geometry
0,NaN,None,None,None,NaN,None,NaN,NaN,1,,,,,,POINT (-51.24063 -30.03366)
1,NaN,None,None,None,NaN,None,NaN,NaN,2,,,,,,POINT (-51.24004 -30.03213)
2,NaN,None,None,None,NaN,None,NaN,NaN,3,,,,,,POINT (-51.23685 -30.03)
3,NaN,None,None,None,NaN,None,NaN,NaN,4,,,,,,POINT (-51.2319 -30.02762)
4,NaN,None,None,None,NaN,None,NaN,NaN,5,,,,,,POINT (-51.23107 -30.02721)
5,NaN,None,None,None,NaN,None,NaN,NaN,6,,,,,,POINT (-51.22826 -30.02602)
6,NaN,None,None,None,NaN,None,NaN,NaN,7,,,,,,POINT (-51.22237 -30.02348)
7,NaN,None,None,None,NaN,None,NaN,NaN,8,,,,,,POINT (-51.2177 -30.0206)
8,NaN,None,None,None,NaN,None,NaN,NaN,9,,,,,,POINT (-51.21417 -30.01494)
9,NaN,None,None,None,NaN,None,NaN,NaN,10,,,,,,POINT (-51.21178 -30.0105)



Tipos de dados:


id               float64
id_cbs            object
nm_cbs            object
c1_m3/s           object
Número           float64
Nome              object
COORD_X          float64
COORD_Y          float64
Name                 str
Status_07_05         str
Status_06_05         str
Status_08_05         str
Status_09_05         str
Status_10_05         str
geometry        geometry
dtype: object


Valores ausentes:


id              14
id_cbs          14
nm_cbs          14
c1_m3/s         14
Número          14
Nome            14
COORD_X         14
COORD_Y         14
Name             0
Status_07_05     0
Status_06_05     0
Status_08_05     0
Status_09_05     0
Status_10_05     0
geometry         0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,id,id_cbs,nm_cbs,c1_m3/s,Número,Nome,COORD_X,COORD_Y,Name,Status_07_05,Status_06_05,Status_08_05,Status_09_05,Status_10_05,geometry
count,0.0,0,0,0,0.0,0,0.0,0.0,14,14,14,14,14,14,14
unique,NaN,0,0,0,NaN,0,NaN,NaN,14,2,2,2,2,2,14
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,,,,,,POINT (-51.2406300279669 -30.0336567939445)
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,11,13,11,11,11,1
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Resultados e possíveis usos

A camada contém **14 registros e 15 colunas**, com geometrias `Point` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **14 geometrias distintas** e **0 linhas inteiramente duplicadas**.

**Limitações dos atributos:** oito campos estão completamente ausentes: `id`, `id_cbs`, `nm_cbs`, `c1_m3/s`, `Número`, `Nome`, `COORD_X` e `COORD_Y`. `Name` contém os rótulos de “1” a “14”, sem confirmação de identificador oficial.

Os campos de situação contêm apenas “Falha” ou texto vazio: em `Status_06_05`, há **uma “Falha” e 13 vazios**; em cada campo de 07, 08, 09 e 10/05, há **três “Falha” e 11 vazios**. Os vazios não são contados por `isna()` e não comprovam funcionamento normal. As datas e o significado de falha ainda dependem de documentação.


## 47. Exploração das EBAPs — RMPA

### Camada `rmpa_sb_drenagem_ebap`


**Esclarecimento do glossário:** O glossário define **EBAP como estação de bombeamento de água pluvial**. Essa infraestrutura deve ser distinguida das EBABs de água bruta presentes na camada de ETAs.


In [52]:
exploracao_inicial(camadas['name'][41])  # EBAPs — RMPA


Exploração inicial da camada: rmpa_sb_drenagem_ebap



,id,id_cbs,c1_m3/s,Número,Name,Status_06_05,Status_08_05,Status_09_05,Status_10_05,lat,long,geometry
0,0.0,15,NaN,NaN,casa_de_bombas_ginasio,NaN,NaN,NaN,NaN,-29.758328,-51.143852,POINT (-51.14385 -29.75833)
1,0.0,12,NaN,NaN,casa_de_bombas_rodoviaria,NaN,NaN,NaN,NaN,-29.760166,-51.149260,POINT (-51.14926 -29.76017)
2,0.0,105,NaN,NaN,casa_de_bombas_joao_correa,NaN,NaN,NaN,NaN,-29.764953,-51.169418,POINT (-51.16942 -29.76495)
3,0.0,73,NaN,NaN,casa_de_bombas_campinas,NaN,NaN,NaN,NaN,-29.755089,-51.162024,POINT (-51.16202 -29.75509)
4,0.0,8,NaN,NaN,casa_de_bombas_cerquinha,NaN,NaN,NaN,NaN,-29.746604,-51.169473,POINT (-51.16947 -29.7466)
5,0.0,120,NaN,NaN,casa_de_bombas_santo_afonso,NaN,NaN,NaN,NaN,-29.742020,-51.129810,POINT (-51.12981 -29.74202)
6,0.0,NaN,NaN,NaN,casa_de_bombas_pampa,NaN,NaN,NaN,NaN,-29.705452,-51.088575,POINT (-51.08857 -29.70545)
7,NaN,NaN,6,NaN,Polder Mathias Velho 1,NaN,NaN,NaN,NaN,-29.895508,-51.236154,POINT (-51.23615 -29.89551)
8,NaN,NaN,4,NaN,Polder Fátima,NaN,NaN,NaN,NaN,-29.937817,-51.196449,POINT (-51.19645 -29.93782)
9,NaN,NaN,3,NaN,Polder Rio Branco,NaN,NaN,NaN,NaN,-29.966707,-51.190587,POINT (-51.19059 -29.96671)



Tipos de dados:


id               float64
id_cbs               str
c1_m3/s              str
Número           float64
Name                 str
Status_06_05         str
Status_08_05         str
Status_09_05         str
Status_10_05         str
lat              float64
long             float64
geometry        geometry
dtype: object


Valores ausentes:


id              29
id_cbs          30
c1_m3/s         30
Número          13
Name             0
Status_06_05    13
Status_08_05    13
Status_09_05    13
Status_10_05    13
lat              0
long             0
geometry         0
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,id,id_cbs,c1_m3/s,Número,Name,Status_06_05,Status_08_05,Status_09_05,Status_10_05,lat,long,geometry
count,7.0,6,6,23.000000,36,23,23,23,23,36.000000,36.000000,36
unique,NaN,6,6,NaN,36,2,2,2,2,NaN,NaN,36
top,NaN,15,6,NaN,casa_de_bombas_ginasio,,Falha,Falha,Falha,NaN,NaN,POINT (-51.143851664809375 -29.758327759821345)
freq,NaN,1,1,NaN,1,13,18,17,16,NaN,NaN,1
mean,0.0,NaN,NaN,16.869565,NaN,NaN,NaN,NaN,NaN,-29.952601,-51.188433,NaN
std,0.0,NaN,NaN,10.118859,NaN,NaN,NaN,NaN,NaN,0.111455,0.039321,NaN
min,0.0,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,-30.092813,-51.243344,NaN
25%,0.0,NaN,NaN,7.500000,NaN,NaN,NaN,NaN,NaN,-30.025946,-51.216867,NaN
50%,0.0,NaN,NaN,18.000000,NaN,NaN,NaN,NaN,NaN,-29.982268,-51.196975,NaN
75%,0.0,NaN,NaN,24.500000,NaN,NaN,NaN,NaN,NaN,-29.927737,-51.165291,NaN


### Resultados e possíveis usos

A camada contém **36 registros e 12 colunas**, com geometrias `Point` em **EPSG:4326**. Há **0 geometrias nulas**, **0 vazias**, **36 geometrias distintas** e **0 linhas inteiramente duplicadas**.

**Limitações dos atributos:** `id` tem 29 ausências e sete zeros; `id_cbs` e `c1_m3/s` têm 30 ausências cada, e `Número` tem 13. Esses campos não fornecem, por si só, uma chave uniforme para toda a camada. O nome `c1_m3/s` sugere uma grandeza de vazão, mas seu significado, unidade e valores precisam ser confirmados antes de interpretar capacidade de bombeamento.

| Campo | Falha | Texto vazio | Ausente |
| --- | ---: | ---: | ---: |
| `Status_06_05` | 10 | 13 | 13 |
| `Status_08_05` | 18 | 5 | 13 |
| `Status_09_05` | 17 | 6 | 13 |
| `Status_10_05` | 16 | 7 | 13 |

Textos vazios e ausências não comprovam funcionamento normal; a redução de registros com “Falha” não comprova recuperação operacional. Não há `Status_07_05`. As definições, as datas e o horário de referência dependem da documentação.


## 48. Exploração dos abrigos — 13/05/2024 às 09h

### Camada `rs_crise_abrigos_13052024_09h`


In [53]:
exploracao_inicial(camadas['name'][42])  # Abrigos — 13/05/2024 às 09h


Exploração inicial da camada: rs_crise_abrigos_13052024_09h

Exibindo as primeiras 100 de 595 linhas. As verificações usam a camada completa.


,name,TIPO DE LOCAL,ENDEREÇO,O QUE ESTÁ FALTANDO,O QUE ESTÁ SOBRANDO,PRECISA DE VOLUNTARIOS,QUANTOS VOLUNTARIOS,PESSOAS ABRIGADAS,VAGAS DISPONÍVEIS,VAGAS DISPONÍVEIS PARA PETS,INFORMAÇÕES SOBRE PETS,ÚLTIMA ATUALIZAÇÃO,FONTE,CIDADE,CÓDIGO,LATITUDE,LONGITUDE,geometry
0,21 BPM,Abrigo,"R. Gov. Peracchi Barcellos, 4600 - Restinga, P...","Água,Leite,Kit higiene",0,Não,0,0.0,NaN,0.0,0,05/08/2024 18:23:59,SOS RS,Porto Alegre,CD15251521,-30.152539,-51.152107,POINT (-51.15211 -30.15254)
1,CTG 35,Abrigo,"Av. Ipiranga, 5300 - Jardim Botânico, Porto Al...","Cesta Básica,Rodo,Bombril,Ração de gatos,Casin...",Não,Não,0,0.0,0.0,0.0,0,05/11/2024 18:59:59,SOS RS,Porto Alegre,CD05491869,-30.054988,-51.186973,POINT (-51.18697 -30.05499)
2,A Igreja de Jesus Cristo dos Santos dos Último...,Coleta de doações,"Av. Princesa Isabel, 300 - Santana, Porto Alegre",NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,Fórum Social das Periferias de Porto Alegre,Porto Alegre,CD04862098,-30.048684,-51.209819,POINT (-51.20982 -30.04868)
3,AABB Porto Alegre,Abrigo,"Av. Cel. Marcos, 1000 - Pedra Redonda, Porto A...",Cesta Básica,"Roupas Femininas,Roupas Masculino,Água",Não,0,120.0,NaN,0.0,0,05/12/2024 10:24:59,SOS RS,Porto Alegre,CD12332456,-30.123390,-51.245654,POINT (-51.24565 -30.12339)
4,ABAMF (Ginásio atrás da Brigada Militar na Ron...,Abrigo,"R. Germano Fehse, 90 - Rondônia, Novo Hamburgo",NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,Fórum Social das Periferias de Porto Alegre,Novo Hamburgo,CD70401204,-29.704012,-51.120434,POINT (-51.12043 -29.70401)
5,ABCREI,Abrigo,"R. Beco dos Soares, 765 - São Lucas, Viamão",0,0,0,0,0.0,NaN,0.0,0,05/09/2024 11:02:59,SOS RS,Viamão,CD08660744,-30.086662,-51.074403,POINT (-51.0744 -30.08666)
6,Abrigo da Bento Atacadão Miranda,Abrigo,"Av. Bento Gonçalves 6623 - Partenon, Porto Alegre","toalha de banho,Meias",0,Não,0,33.0,0.0,0.0,0,30/12/1899 00:00:00,SOS RS,Porto Alegre,CD06691502,-30.066993,-51.150273,POINT (-51.15027 -30.06699)
7,Abrigo de animais Chácara das Pedras,Abrigo de animais,"R. Dr Barbosa Gonçalves, 61 - Chácara das Pedr...",0,0,Sim,0,0.0,77.0,0.0,0,30/12/1899 00:00:00,SOS RS,Porto Alegre,CD03501617,-30.035095,-51.161745,POINT (-51.16175 -30.03509)
8,Abrigo Gringo,Abrigo,"R. Walir Zottis, 380 - Jardim Itú, Porto Alegre","Leite,Molho de tomate,Alimentos Enlatados,Maca...",Não,Sim,"10 voluntários, todos os turnos (são 22 volun...",50.0,0.0,0.0,0,05/12/2024 21:00:00,SOS RS,Porto Alegre,CD01451374,-30.014568,-51.137470,POINT (-51.13747 -30.01457)
9,AD Bom Princípio,Abrigo,"R. das Acácias, 585 - Jardim do Vale, Bom Prin...",NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN,Fórum Social das Periferias de Porto Alegre,Bom Princípio,CD48463593,-29.484687,-51.359367,POINT (-51.35937 -29.48469)



Tipos de dados:


name                                str
TIPO DE LOCAL                       str
ENDEREÇO                            str
O QUE ESTÁ FALTANDO                 str
O QUE ESTÁ SOBRANDO                 str
PRECISA DE VOLUNTARIOS              str
QUANTOS VOLUNTARIOS                 str
PESSOAS ABRIGADAS               float64
VAGAS DISPONÍVEIS               float64
VAGAS DISPONÍVEIS PARA PETS     float64
INFORMAÇÕES SOBRE PETS              str
ÚLTIMA ATUALIZAÇÃO                  str
FONTE                               str
CIDADE                              str
CÓDIGO                              str
LATITUDE                        float64
LONGITUDE                       float64
geometry                       geometry
dtype: object


Valores ausentes:


name                             1
TIPO DE LOCAL                    1
ENDEREÇO                         1
O QUE ESTÁ FALTANDO            325
O QUE ESTÁ SOBRANDO            325
PRECISA DE VOLUNTARIOS         325
QUANTOS VOLUNTARIOS            325
PESSOAS ABRIGADAS              325
VAGAS DISPONÍVEIS              325
VAGAS DISPONÍVEIS PARA PETS    325
INFORMAÇÕES SOBRE PETS         325
ÚLTIMA ATUALIZAÇÃO             325
FONTE                            1
CIDADE                           1
CÓDIGO                           1
LATITUDE                         1
LONGITUDE                        1
geometry                         1
dtype: int64

... (2 outputs left out)


Sistema de cordenadas (CRS):


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich


Estatísticas descritivas:


,name,TIPO DE LOCAL,ENDEREÇO,O QUE ESTÁ FALTANDO,O QUE ESTÁ SOBRANDO,PRECISA DE VOLUNTARIOS,QUANTOS VOLUNTARIOS,PESSOAS ABRIGADAS,VAGAS DISPONÍVEIS,VAGAS DISPONÍVEIS PARA PETS,INFORMAÇÕES SOBRE PETS,ÚLTIMA ATUALIZAÇÃO,FONTE,CIDADE,CÓDIGO,LATITUDE,LONGITUDE,geometry
count,594,594,594,270,270,270,270,270.000000,270.000000,270.000000,270,270,594,594,594,594.000000,594.000000,594
unique,589,17,581,106,22,6,30,NaN,NaN,NaN,32,122,10,25,586,NaN,NaN,587
top,Banco de Alimentos,Abrigo,"R. Germano Fehse, 90 - Rondônia, Novo Hamburgo",0,0,0,0,NaN,NaN,NaN,0,30/12/1899 00:00:00,SOS RS,Porto Alegre,CD95050016,NaN,NaN,POINT (-51.001622799999 -29.9505364999999)
freq,2,297,2,145,150,125,241,NaN,NaN,NaN,236,130,209,263,2,NaN,NaN,2
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85.562963,9.059259,0.018519,NaN,NaN,NaN,NaN,NaN,-29.920769,-51.166689,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,434.010966,41.995267,0.304290,NaN,NaN,NaN,NaN,NaN,0.260784,0.143317,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,-31.759000,-52.430857,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,-30.050557,-51.191265,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,-29.989521,-51.151855,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,58.250000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,-29.772546,-51.114388,NaN


### Resultados e possíveis usos

A camada contém **595 registros e 18 colunas**, com geometrias `Point` em **EPSG:4326**. Há **1 geometrias nulas**, **0 vazias**, **587 geometrias distintas** e **2 linhas inteiramente duplicadas**.

**Limitações principais:** há **duas linhas inteiramente duplicadas**, preservadas. A camada mistura abrigos humanos, locais para animais, cozinhas e pontos de doação, com variações de categorias e registros “a confirmar”; o total não equivale a abrigos humanos distintos. `CÓDIGO` também não é único e utiliza prefixo “CD” nos exemplos, diferente de versões anteriores.

Os nove campos operacionais e `ÚLTIMA ATUALIZAÇÃO` têm **325 ausências cada**. Textos “0” e zeros numéricos dependem de interpretação antes de considerar ausência de demanda ou somar vagas e pessoas abrigadas. `QUANTOS VOLUNTARIOS` mistura quantidades e descrições.

`ÚLTIMA ATUALIZAÇÃO` contém exemplos com possível ambiguidade entre dia e mês e o valor **“30/12/1899 00:00:00”**, que não deve ser interpretado como atualização real do evento sem conferir a origem. Assim, a referência de 13/05 às 09h no nome da camada não confirma atualização simultânea de cada registro.


### Síntese dos cadastros de abrigos e pontos de apoio

| Referência nominal | Registros | Geometrias nulas | Vazias | Linhas duplicadas |
| --- | ---: | ---: | ---: | ---: |
| 08/05/2024 às 12h | 210 | 0 | 2 | 0 |
| 09/05/2024 às 20h | 360 | 0 | 0 | 0 |
| 10/05/2024 às 13h | 357 | 0 | 0 | 0 |
| 10/05/2024 às 20h | 493 | 1 | 0 | 3 |
| 11/05/2024 às 13h | 504 | 0 | 0 | 4 |
| 11/05/2024 às 18h | 574 | 0 | 0 | 4 |
| 13/05/2024 às 09h | 595 | 1 | 0 | 2 |

As versões misturam abrigos humanos, espaços para animais, cozinhas e pontos de doação. Não devem ser somadas como estabelecimentos independentes. Mudanças de total podem refletir fontes e cobertura, sem comprovar abertura ou fechamento.

Datas dos nomes não garantem atualização simultânea dos registros. Vazios, “0” e outros marcadores exigem interpretação; campos operacionais incompletos não sustentam totais de capacidade ou ocupação sem preparação. Duplicatas foram preservadas.

Um possível caminho é escolher um recorte temporal ou relacionar estabelecimentos entre versões, após conferir identificadores, categorias, coordenadas e datas. As limitações particulares estão nos diagnósticos individuais.


## 49. Agrupamento temático das 43 camadas

Os grupos abaixo organizam relações entre os dados; não indicam equivalência metodológica ou integração. Os diagnósticos individuais e o glossário registram as limitações.

### 1. Referências territoriais (2 camadas)

- `poa_admin_bairros`
- `rhguaiba_bacia`

### 2. Camadas associadas a eventos de inundação em Porto Alegre (6 camadas)

- `poa_inundacao_1941_475cm`
- `poa_inundacao_1967_313cm`
- `poa_inundacao_2015_294cm`
- `poa_inundacao_2023_set_318cm`
- `poa_inundacao_2023_nov_346cm`
- `poa_inundacao_2024_530cm`

### 3. Simulações de inundação e severidade (7 camadas)

- `poa_inundacao_simulada_500cm`
- `poa_inundacao_simulada_550cm`
- `poa_inundacao_simulada_600cm`
- `poa_inundacao_simulada_650cm`
- `poa_inundacao_simulada_700cm`
- `rhguaiba_hecras_inundacao_simulada_550cm`
- `rhguaiba_hecras_severidade_simulada_06052024`

### 4. Água, inundação e lama identificadas por imagens (6 camadas)

- `rhguaiba_sentinel2_agua_obs_06052024`
- `rhguaiba_sentinel2_agua_obs_06052024_validada`
- `rhguaiba-taquari_planet-skysat_inundacao_obs_06052024`
- `rhguaiba-pardo_planet-skysat_inundacao_obs_06052024`
- `rhguaiba-cai_planet-skysat_inundacao_obs_06052024`
- `rhguaiba_sentinel2_lama_obs_06052024_2`

### 5. Produto de análise ADA por unidade territorial (1 camada)

- `analise_rhguaiba_ada_06052024`

### 6. Escolas, saúde e assistência social (5 camadas)

- `poa_edu_escolas`
- `poa_saude_sus`
- `poa_saude_unidades_atencao_primaria_09052024`
- `poa_saude_unidades_atencao_primaria_10052024`
- `poa_saude_assistencia_social`

### 7. Abastecimento e fontes públicas de água (4 camadas)

- `poa_sb_agua_etas`
- `poa_sb_agua_saa`
- `poa_sb_agua_fontes_publicas`
- `rmpa_sb_agua_fontes_publicas`

### 8. Drenagem e proteção contra inundação (4 camadas)

- `poa_sb_drenagem_protecao_linhas`
- `poa_sb_drenagem_protecao_pontos`
- `rmpa_sb_drenagem_comporta`
- `rmpa_sb_drenagem_ebap`

### 9. Abrigos, doações e outros pontos de apoio — versões temporais (7 camadas)

- `rs_crise_abrigos_08052024_12h`
- `rs_crise_abrigos_09052024_20h`
- `rs_crise_abrigos_10052024_13h`
- `rs_crise_abrigos_10052024_20h`
- `rs_crise_abrigos_11052024_13h`
- `rs_crise_abrigos_11052024_18h`
- `rs_crise_abrigos_13052024_09h`

### 10. Suscetibilidade a movimentos de massa — contexto do Taquari (1 camada)

- `taquari_areas_propensas_movimentos_massa`

### Direção discutida para a próxima etapa

Foram definidos cinco grupos para aprofundamento: **referência territorial de bairros, escolas, SUS, atenção primária e inundação em Porto Alegre**. Isso não resolve a escolha entre versões ou autoriza presumir equivalência das manchas com o JSON. Os procedimentos posteriores e os produtos gerados estão documentados no [notebook de preparação](preparacao_cheias_guaiba_2024.ipynb).


A seleção final e as exclusões estão consolidadas na seção 51.


## 50. Divergências documentais pendentes

Fonte: [glossário do GeoPackage](../../dados/cheias_rhguaiba_2024_db_v5_glossario.csv). As definições de ADA, severidade, Planet/Skysat e EBAP/EBAB foram incorporadas às seções correspondentes. O arquivo descreve camadas e não substitui um dicionário de atributos.

| Camada ou tema | Divergência documental |
| --- | --- |
| `analise_rhguaiba_ada_06052024` | Nome indica 06/05/2024; descrição informa 06/06/2024. |
| `poa_inundacao_2015_294cm` | Nome indica 2015 e 294 cm; descrição informa simulação de 600 cm. |
| `poa_inundacao_2024_530cm` | Nome indica 530 cm; descrição informa simulação de 500 cm. |
| `poa_inundacao_simulada_500cm` | Nome indica 500 cm; descrição informa 550 cm. |
| `poa_inundacao_simulada_550cm` | Nome indica 550 cm; descrição informa 600 cm. |
| `poa_saude_sus` | Descrição de inundação em novembro de 2033 é incompatível com o conteúdo de saúde inspecionado. |
| Planet/Skysat Caí e Pardo | A abrangência “Vale do Pardo” aparece na entrada Caí e “Vale do Cai” na entrada Pardo. |
| Lama Sentinel-2 | Nome repetido sem `_2`, com descrições de mancha de lama e de setores censitários recortados. |
| Atenção primária, Taquari e abrigos | Algumas entradas utilizam nome genérico, prefixo diferente ou marcador `{data}`; a correspondência com as camadas locais precisa ser considerada explicitamente. |

A Prefeitura de Porto Alegre é indicada como fonte de diversas camadas territoriais e de infraestrutura; CORSAN, das fontes públicas metropolitanas; SEMA-RS, da delimitação hidrográfica. A atribuição de fonte não garante atualização ou completude.

Permanecem pendentes `cota_mean`, vários códigos `DN`/`gridcode`, unidades de medidas, referências temporais e precisão das localizações. Uma possibilidade é manter comparações quantitativas afetadas por essas divergências pendentes de esclarecimento.


## 51. Bases escolhidas para aprofundamento

Após a exploração das 43 camadas, foram escolhidos **cinco grupos para a próxima etapa de avaliação e preparação**. Essa seleção delimita o foco; não significa que todos os atributos ou versões estejam prontos para uso. Os procedimentos abaixo são possibilidades, e eventuais tratamentos dependem de identificar e justificar os problemas.

| Grupo | Camadas escolhidas | Justificativa |
| --- | --- | --- |
| Referência territorial | `poa_admin_bairros` | Oferece nomes, códigos e polígonos para mapas e possíveis resultados por bairro. Pode servir como referência territorial para relacionar estabelecimentos e manchas. |
| Educação | `poa_edu_escolas` | Possui identificação e localização dos estabelecimentos, permitindo avaliar sua distribuição e possível exposição espacial aos cenários. O foco escolhido é localização e identificação, sem priorizar os indicadores educacionais. |
| Saúde | `poa_saude_sus` | Reúne localização e categorias de serviços, possibilitando avaliar a distribuição espacial de diferentes tipos de atendimento e possíveis relações com as manchas. |
| Atenção primária | `poa_saude_unidades_atencao_primaria_09052024` e `poa_saude_unidades_atencao_primaria_10052024` | Oferecem unidades identificadas, localização e situação de atendimento em dois recortes nominais de data. São candidatas a uma análise mais específica dos serviços, sem somar as versões como estabelecimentos independentes. |
| Inundação em Porto Alegre | Quatro manchas históricas estimadas, listadas abaixo | Podem complementar os cenários já disponíveis no INCT com referências históricas estimadas, para possíveis mapas e cruzamentos. A metodologia e a referência das cotas ainda precisam ser consideradas. |

### Camadas de inundação mantidas na seleção

- `poa_inundacao_1941_475cm`
- `poa_inundacao_1967_313cm`
- `poa_inundacao_2023_set_318cm`
- `poa_inundacao_2023_nov_346cm`

O glossário descreve essas quatro camadas como **manchas estimadas de eventos históricos**, sem apresentá-las como as simulações de cenários excluídas. Isso não as torna observações diretas: a origem das estimativas e a referência das cotas permanecem pendentes de esclarecimento.

### Simulações fora da seleção

Foi decidido **não preparar nem selecionar simulações deste GeoPackage**, pois o INCT já dispõe de cenários para Porto Alegre. A exploração é mantida como registro do inventário, sem indicar aproveitamento das simulações nesta etapa.

Isso inclui os cinco cenários nominais de 500 a 700 cm e os produtos simulados HEC-RAS de inundação e severidade. **`poa_inundacao_2015_294cm` e `poa_inundacao_2024_530cm` também ficam fora da preparação nesta etapa:** apesar de seus nomes remeterem a eventos, o glossário as descreve como simulações e apresenta cotas divergentes. Não serão tratadas como manchas observadas sem confirmação documental.

As camadas de água e inundação derivadas de imagens permanecem fora da seleção inicial já definida; sua avaliação poderia ser retomada se houver necessidade de dados observacionais de 2024, sem pressupor equivalência com as manchas históricas.

### Encaminhamentos para a preparação

- **Bairros:** conferir geometrias, identificadores e correspondência dos 83 registros com os 94 bairros presentes em cada cenário do JSON do INCT. O JSON é uma referência externa para essa conferência, não uma camada do GeoPackage.
- **Escolas:** avaliar os identificadores e as coordenadas compartilhadas. Permanece o critério adotado de considerar os 1.062 registros pertencentes a Porto Alegre, preservando as posições originais e a sinalização dos cinco pontos fora da cobertura territorial.
- **SUS:** investigar localizações compartilhadas e distinguir registros de serviços, estabelecimentos e endereços antes de produzir contagens.
- **Atenção primária:** avaliar a correspondência das unidades entre datas, a inconsistência de `Cidade` e a referência das situações. A escolha de uma versão ou de uma comparação temporal depende da pergunta de análise; as duas versões não devem ser somadas diretamente nem automaticamente combinadas com o cadastro SUS.
- **Inundação:** esclarecer método, cotas e códigos de classe, avaliar geometrias e sobreposições e conferir medidas de área. A compatibilidade com os cenários do JSON também permanece pendente.

As demais camadas ficam **fora do foco inicial de preparação**, sem descarte definitivo. Produtos regionais de satélite, HEC-RAS, infraestrutura e apoio à crise podem ser reconsiderados conforme a necessidade das análises.

Esta seção registra a seleção ao término da exploração; a preparação dos dados selecionados está documentada no [notebook de preparação](preparacao_cheias_guaiba_2024.ipynb).


**Resultados posteriores:** a preparação relacionou os 83 bairros aos nomes do JSON por quatro equivalências, preservou as 1.062 escolas, reduziu o cadastro SUS para 286 registros pelo critério documentado, ajustou as duas versões de atenção primária e calculou áreas nas quatro manchas históricas. Os detalhes e limitações constam no notebook de preparação. Os diagnósticos deste notebook continuam descrevendo as fontes originais.
